# 🌑 Singularity — Enterprise RAG on Google Colab (T4 GPU)

## Cell 1 — Verifying GPU


In [20]:
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ No GPU — go to Runtime → Change runtime type → T4 GPU')

import torch
print(f'PyTorch CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('❌ GPU not available — change runtime type first!')


Wed Mar 18 19:50:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cell 2 — Installing dependencies

In [21]:
%%capture install_out
# Core RAG stack
!pip install -q fastapi 'uvicorn[standard]' streamlit pyngrok
!pip install -q sentence-transformers faiss-gpu
!pip install -q ollama
# Document parsers
!pip install -q pymupdf pdfplumber python-docx
!pip install -q pandas openpyxl python-pptx 'beautifulsoup4' lxml
!pip install -q requests urllib3
# Vision — moondream2 image captioning
!pip install -q 'transformers>=4.36.0' Pillow einops timm
# BM25 + cross-encoder
!pip install -q rank_bm25 'sentence-transformers>=2.2.0'
# OCR support
!apt-get install -q -y tesseract-ocr
!pip install -q pytesseract
print('✅ All packages installed')


In [22]:
print('✅ Dependencies installed successfully')


✅ Dependencies installed successfully


## Cell 3 — Installing Ollama + pulling Llama 3.2:3b



In [23]:
import subprocess, time, os
import requests as _req

# Install zstd first (required by newer Ollama installer)
print('📦 Installing zstd...')
subprocess.run(['apt-get', 'install', '-y', 'zstd'], capture_output=True)
print('✅ zstd installed')

# Install Ollama
print('📦 Installing Ollama...')
r = subprocess.run(
    'curl -fsSL https://ollama.com/install.sh | sh',
    shell=True, capture_output=True, text=True
)
print(r.stdout[-500:] if r.stdout else '')
print(r.stderr[-200:] if r.stderr else '')

# Force PATH
os.environ['PATH'] = f'/usr/local/bin:{os.environ.get("PATH", "")}'

# Find binary
OLLAMA_BIN = '/usr/local/bin/ollama'
if not os.path.exists(OLLAMA_BIN):
    for candidate in ['/usr/bin/ollama', '/usr/local/lib/ollama/ollama']:
        if os.path.exists(candidate):
            OLLAMA_BIN = candidate
            break
    else:
        raise FileNotFoundError('Ollama binary not found — check install output above')

print(f'✅ Ollama binary: {OLLAMA_BIN}')

# Start server
print('🔧 Starting Ollama server...')
ollama_proc = subprocess.Popen(
    [OLLAMA_BIN, 'serve'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env={**os.environ, 'HOME': '/root', 'OLLAMA_HOST': '0.0.0.0:11434'}
)

# Wait for ready
print('⏳ Waiting for Ollama server...')
for i in range(30):
    time.sleep(1)
    try:
        if _req.get('http://localhost:11434', timeout=1).status_code == 200:
            print(f'✅ Ollama server ready ({i+1}s)')
            break
    except Exception:
        pass
else:
    print('⚠️  Server slow — continuing anyway')

# Pull model
print('\n📥 Pulling llama3.2:3b (~2GB)...')
result = subprocess.run(
    [OLLAMA_BIN, 'pull', 'llama3.2:3b'],
    env={**os.environ, 'HOME': '/root'}, text=True
)
print('✅ Model ready!' if result.returncode == 0 else f'❌ Failed (code {result.returncode})')

models = subprocess.run([OLLAMA_BIN, 'list'], capture_output=True, text=True,
                        env={**os.environ, 'HOME': '/root'})
print(f'\nAvailable models:\n{models.stdout}')

import builtins
builtins.OLLAMA_BIN = OLLAMA_BIN

📦 Installing zstd...
✅ zstd installed
📦 Installing Ollama...

...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

✅ Ollama binary: /usr/local/bin/ollama
🔧 Starting Ollama server...
⏳ Waiting for Ollama server...
✅ Ollama server ready (2s)

📥 Pulling llama3.2:3b (~2GB)...
✅ Model ready!

Available models:
NAME           ID              SIZE      MODIFIED               
llama3.2:3b    a80c4f17acd5    2.0 GB    Less than a second ago    



## Cell 4 — Creating project directory

In [24]:
from pathlib import Path

APP_DIR = Path('/content/singularity')
for d in ['uploads','sessions','feedback','vector_store','doc_meta','knowledge_graph']:
    (APP_DIR / d).mkdir(parents=True, exist_ok=True)

print(f'✅ Project directory ready at {APP_DIR}')
print('Subdirs:', [p.name for p in APP_DIR.iterdir()])


✅ Project directory ready at /content/singularity
Subdirs: ['doc_meta', 'traces', 'memory.py', 'knowledge_graph', 'ingestion.py', 'main.py', 'feedback', 'retrieval.py', '__pycache__', 'app.py', 'sessions', 'uploads', 'vector_store', 'llm.py', 'knowledge_graph.py', 'tracer.py']


## Cell 5 — Writing all Python source files



### 5a — ingestion.py
*Ingestion pipeline — GPU-accelerated embeddings (batch=128 on T4)*

In [25]:
%%writefile /content/singularity/ingestion.py
"""
Enterprise Ingestion Pipeline  v4.0
=====================================
Supported formats: PDF, DOCX, TXT, CSV, XLSX, PPTX, MD, HTML

Key improvements over v3:
  - Multi-format support (CSV, XLSX, PPTX, Markdown, HTML)
  - Superior table extraction: PDF tables via pdfplumber, XLSX/CSV as structured data
  - Batch embedding with larger batch sizes for speed
  - Embedding model loaded ONCE at import time (no cold-start on first request)
  - Dedup threshold loosened slightly to avoid over-filtering
"""

import re
import base64
import hashlib
import json
import time
import numpy as np
from pathlib import Path
from typing import Optional
from datetime import datetime, timezone

import fitz           # PyMuPDF — fast PDF text extraction + image extraction
import docx           # python-docx
import faiss
from sentence_transformers import SentenceTransformer

# Optional format libraries — fail gracefully
try:
    import pdfplumber          # superior table extraction from PDFs
    _pdfplumber = True
except ImportError:
    _pdfplumber = False

try:
    import pandas as pd
    _pandas = True
except ImportError:
    _pandas = False

try:
    from pptx import Presentation as _PPTXPresentation
    _pptx = True
except ImportError:
    _pptx = False

try:
    from bs4 import BeautifulSoup
    _bs4 = True
except ImportError:
    _bs4 = False

# ── Paths ──────────────────────────────────────────────────────
BASE_DIR   = Path(__file__).parent
VECTOR_DIR = BASE_DIR / "vector_store"
VECTOR_DIR.mkdir(exist_ok=True)
META_DIR   = BASE_DIR / "doc_meta"
META_DIR.mkdir(exist_ok=True)

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
EMBED_DIM        = 384
CHUNK_SIZE       = 200   # words — slightly larger for better context
CHUNK_OVERLAP    = 50    # words
MIN_CHUNK_LEN    = 15    # words

FAISS_INDEX_FILE = VECTOR_DIR / "index.faiss"
METADATA_FILE    = VECTOR_DIR / "metadata.json"

# ── Detect GPU availability ──
import torch as _torch
_DEVICE = "cuda" if _torch.cuda.is_available() else "cpu"
if _DEVICE == "cuda":
    _gpu_name = _torch.cuda.get_device_name(0)
    _vram_gb  = _torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[Ingestion] GPU detected: {_gpu_name} ({_vram_gb:.1f} GB VRAM)")
    # GTX 1650 / any card ≤ 6GB: keep FAISS on CPU to avoid OOM.
    # Sentence-transformer embeddings on GPU still give a big speedup.
    _USE_FAISS_GPU = False
else:
    print("[Ingestion] No GPU detected, using CPU.")
    _USE_FAISS_GPU = False

# ── Load sentence-transformer ONCE at import ──
print(f"[Ingestion] Pre-loading sentence-transformer model on {_DEVICE.upper()}...")
_embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=_DEVICE)
print(f"[Ingestion] Model ready on {_DEVICE.upper()}.")

_faiss_index = None
_metadata_store: list[dict] = []

# ── Moondream2 — lazy-loaded only when images exist ──
_moondream_model = None
_moondream_proc  = None
_moondream_failed = False          # sentinel — don't retry after a load failure
MAX_IMAGES_PER_DOC = 15

def _get_vision_model():
    """Load BLIP image captioning model. Returns (model, processor) or (None, None).

    Uses Salesforce/blip-image-captioning-base — a standard HuggingFace model
    with no trust_remote_code, no custom config classes, works on every
    transformers version, ~450MB, runs well on T4 in fp16.
    """
    global _moondream_model, _moondream_proc, _moondream_failed
    if _moondream_model is not None:
        return _moondream_model, _moondream_proc
    if _moondream_failed:
        return None, None
    try:
        import torch
        from transformers import BlipProcessor, BlipForConditionalGeneration

        print("[Vision] Loading BLIP image captioning model...")
        t0 = time.time()

        _moondream_proc = BlipProcessor.from_pretrained(
            "Salesforce/blip-image-captioning-base"
        )
        _moondream_model = BlipForConditionalGeneration.from_pretrained(
            "Salesforce/blip-image-captioning-base",
            torch_dtype=torch.float16 if _DEVICE == "cuda" else torch.float32,
        )
        if _DEVICE == "cuda":
            _moondream_model = _moondream_model.cuda()
        _moondream_model.eval()
        print(f"[Vision] BLIP ready in {time.time()-t0:.1f}s")
        return _moondream_model, _moondream_proc

    except Exception as e:
        import traceback as _tb
        print(f"[Vision] BLIP unavailable ({e}) — images will use placeholder captions")
        _tb.print_exc()
        _moondream_failed = True
        return None, None


def _describe_image_b64(b64_str: str, ext: str = "png") -> str:
    """Return a natural-language caption for a base64 image using BLIP."""
    import io, base64 as _b64, torch
    try:
        from PIL import Image
        model, processor = _get_vision_model()
        if model is None:
            return "Visual content (caption unavailable)"
        img_bytes = _b64.b64decode(b64_str)
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        inputs = processor(img, return_tensors="pt")
        if _DEVICE == "cuda":
            inputs = {k: v.to("cuda").half() if v.is_floating_point() else v.to("cuda")
                      for k, v in inputs.items()}
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=80)
        caption = processor.decode(out[0], skip_special_tokens=True)
        return caption.strip() if caption else "Visual content (empty caption)"
    except Exception as e:
        return f"Visual content (caption error: {type(e).__name__}: {e})"


# ══════════════════════════════════════════════════════════════
#  FAISS STORE  (CPU — GPU FAISS skipped: 4GB VRAM too small)
# ══════════════════════════════════════════════════════════════

def _load_store():
    global _faiss_index, _metadata_store
    if FAISS_INDEX_FILE.exists() and METADATA_FILE.exists():
        _faiss_index = faiss.read_index(str(FAISS_INDEX_FILE))
        with open(METADATA_FILE) as f:
            _metadata_store = json.load(f)
    else:
        _faiss_index = faiss.IndexFlatIP(EMBED_DIM)
        _metadata_store = []


def _save_store():
    faiss.write_index(_faiss_index, str(FAISS_INDEX_FILE))
    with open(METADATA_FILE, "w") as f:
        json.dump(_metadata_store, f)


def _get_store():
    global _faiss_index, _metadata_store
    if _faiss_index is None:
        _load_store()
    return _faiss_index, _metadata_store


# ══════════════════════════════════════════════════════════════
#  STEP 1: PARSERS  (one per format)
# ══════════════════════════════════════════════════════════════

def _parse_pdf(path: str) -> tuple[list[dict], list[dict]]:
    """Extract text (with heading detection) + tables (via pdfplumber) + images."""
    pages = []
    images = []

    # --- Text + headings via PyMuPDF (fast) ---
    doc = fitz.open(path)
    fitz_pages: dict[int, dict] = {}

    for page_num, page in enumerate(doc):
        blocks = page.get_text("dict")["blocks"]
        page_text_parts = []
        headings = []

        for block in blocks:
            if block.get("type") != 0:
                continue
            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    text = span["text"].strip()
                    if not text:
                        continue
                    size  = span.get("size", 12)
                    flags = span.get("flags", 0)
                    is_bold = bool(flags & 2**4)
                    if size >= 14 or (is_bold and size >= 11):
                        headings.append(text)
                        page_text_parts.append(f"\n## {text}\n")
                    else:
                        page_text_parts.append(text + " ")
                page_text_parts.append("\n")

        fitz_pages[page_num] = {
            "page_num": page_num + 1,
            "text": "".join(page_text_parts).strip(),
            "headings": headings,
            "tables": [],
        }

        # Extract images
        for img_index, img_ref in enumerate(page.get_images(full=True)):
            xref = img_ref[0]
            try:
                base_image = doc.extract_image(xref)
                img_bytes = base_image["image"]
                images.append({
                    "page": page_num + 1,
                    "index": img_index,
                    "ext": base_image["ext"],
                    "b64": base64.b64encode(img_bytes).decode("utf-8"),
                    "width": base_image.get("width", 0),
                    "height": base_image.get("height", 0),
                })
            except Exception:
                pass

    doc.close()

    # --- Table extraction via pdfplumber (much more accurate) ---
    if _pdfplumber:
        try:
            with pdfplumber.open(path) as plumb:
                for page_num, page in enumerate(plumb.pages):
                    tables = page.extract_tables()
                    for table in tables:
                        if not table:
                            continue
                        rows = []
                        for row in table:
                            clean_row = [str(cell or "").strip() for cell in row]
                            if any(c for c in clean_row):
                                rows.append(clean_row)
                        if rows:
                            # Build markdown-style table text
                            table_lines = []
                            if len(rows) > 1:
                                header = rows[0]
                                table_lines.append("| " + " | ".join(header) + " |")
                                table_lines.append("| " + " | ".join(["---"] * len(header)) + " |")
                                for row in rows[1:]:
                                    table_lines.append("| " + " | ".join(row) + " |")
                            else:
                                for row in rows:
                                    table_lines.append("| " + " | ".join(row) + " |")

                            table_str = "\n[TABLE]\n" + "\n".join(table_lines) + "\n[/TABLE]\n"
                            if page_num in fitz_pages:
                                fitz_pages[page_num]["tables"].append(table_str)
        except Exception as e:
            print(f"[Ingestion] pdfplumber table extraction warning: {e}")

    for page_num in sorted(fitz_pages):
        p = fitz_pages[page_num]
        combined = p["text"]
        for tbl in p["tables"]:
            combined += "\n" + tbl
        pages.append({
            "page_num": p["page_num"],
            "text": combined,
            "headings": p["headings"],
        })

    return pages, images


def _table_to_markdown(table, caption: str = "") -> str:
    """Convert a docx table object to a [TABLE]...[/TABLE] markdown block with optional caption."""
    rows = []
    for row in table.rows:
        cells = [c.text.strip() for c in row.cells]
        # Deduplicate merged cells (python-docx repeats merged cell text)
        deduped = []
        for j, cell in enumerate(cells):
            if j == 0 or cell != cells[j-1]:
                deduped.append(cell)
        if any(deduped):
            rows.append(deduped)
    if not rows:
        return ""

    # Normalize row widths
    max_cols = max(len(r) for r in rows)
    rows = [r + [""] * (max_cols - len(r)) for r in rows]

    table_lines = []
    header = rows[0]
    table_lines.append("| " + " | ".join(header) + " |")
    table_lines.append("| " + " | ".join(["---"] * len(header)) + " |")
    for row in rows[1:]:
        table_lines.append("| " + " | ".join(row) + " |")

    caption_line = f"\nTable caption: {caption}\n" if caption else ""
    return caption_line + "\n[TABLE]\n" + "\n".join(table_lines) + "\n[/TABLE]\n"


def _parse_docx(path: str) -> tuple[list[dict], list[dict]]:
    """
    Parse a DOCX file with INLINE table placement.

    Key fix over previous version: tables are inserted at the position they
    appear in the document body (using the XML element order), not dumped at
    the end. The preceding paragraph is used as the table caption so the LLM
    can match "Table IV" queries to the correct data chunk.
    """
    document = docx.Document(path)
    pages = []
    images = []
    current_section = []
    headings = []
    page_num = 1

    # Build a lookup from table XML element to table object
    # so we can emit tables in document order
    table_map = {tbl._element: tbl for tbl in document.tables}

    # Caption detection: a paragraph immediately before a table whose text
    # matches TABLE_CAPTION_RE is treated as the table's caption
    TABLE_CAPTION_RE = re.compile(
        r'\b(table|tbl|tab)\.?\s*([\d]+(?:\.[\d]+)*|[IVXLC]+)',
        re.IGNORECASE
    )

    last_para_text = ""  # track preceding paragraph for caption detection

    # Iterate the document body's direct children in XML order
    body = document.element.body
    for child in body:
        tag = child.tag.split("}")[-1] if "}" in child.tag else child.tag

        if tag == "p":
            # Paragraph
            # Find the matching paragraph object
            para = None
            for p in document.paragraphs:
                if p._element is child:
                    para = p
                    break
            if para is None:
                continue

            text = para.text.strip()
            if not text:
                continue

            style = para.style.name if para.style else ""
            if "Heading" in style:
                if current_section:
                    pages.append({
                        "page_num": page_num,
                        "text": "\n".join(current_section),
                        "headings": list(headings),
                    })
                    page_num += 1
                    current_section = []
                    headings = []
                headings.append(text)
                current_section.append(f"\n## {text}\n")
            else:
                current_section.append(text)

            last_para_text = text

        elif tag == "tbl":
            # Table — inline at current position with caption
            tbl_obj = table_map.get(child)
            if tbl_obj is None:
                continue

            # Use preceding paragraph as caption if it looks like a table label
            caption = ""
            if TABLE_CAPTION_RE.search(last_para_text):
                caption = last_para_text

            md = _table_to_markdown(tbl_obj, caption=caption)
            if md:
                current_section.append(md)

    if current_section:
        pages.append({
            "page_num": page_num,
            "text": "\n".join(current_section),
            "headings": headings,
        })

    # ── Inline image extraction with figure-label detection ──────────
    # Build rel_id → raw image data map
    _rel_imgs: dict[str, dict] = {}
    for _rel_id, _rel in document.part.rels.items():
        if "image" in _rel.reltype:
            try:
                _part = _rel.target_part
                _ext  = _part.content_type.split("/")[-1].replace("jpeg", "jpg")
                _b64  = base64.b64encode(_part.blob).decode("utf-8")
                _rel_imgs[_rel_id] = {"rel_id": _rel_id, "ext": _ext, "b64": _b64}
            except Exception:
                pass

    # Walk body XML, collect paragraph texts + positions for ±3-para context window
    _FIGURE_RE = re.compile(r"\b(fig(?:ure)?\.?\s*\d+[\.(\d|[a-z]]*)\.?\b", re.IGNORECASE)
    _all_paras: list[dict] = []
    _h_count = 0
    for _child in document.element.body:
        _tag = _child.tag.split("}")[-1] if "}" in _child.tag else _child.tag
        if _tag == "p":
            _full = "".join(_child.itertext()).strip()
            _pobj = next((p for p in document.paragraphs if p._element is _child), None)
            _is_h = bool(_pobj and _pobj.style and "Heading" in _pobj.style.name)
            if _is_h:
                _h_count += 1
            _all_paras.append({"text": _full, "page": _h_count + 1, "elem": _child})

    # Extract images in document order, deduplicate same rel on same page
    _seen: set = set()
    _BLIP_NS = "{http://schemas.openxmlformats.org/drawingml/2006/main}blip"
    _R_NS    = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"

    for _pi, _pinfo in enumerate(_all_paras):
        for _blip in _pinfo["elem"].iter(_BLIP_NS):
            _embed = _blip.get(f"{{{_R_NS}}}embed")
            if not _embed or _embed not in _rel_imgs:
                continue
            _key = (_embed, _pinfo["page"])
            if _key in _seen:
                continue
            _seen.add(_key)

            # ±3 paragraph context window for figure label detection
            _ctx_before = [_all_paras[j]["text"] for j in range(max(0, _pi - 3), _pi) if _all_paras[j]["text"]]
            _ctx_after  = [_all_paras[j]["text"] for j in range(_pi + 1, min(len(_all_paras), _pi + 4)) if _all_paras[j]["text"]]
            _ctx = " ".join(_ctx_before + [_pinfo["text"]] + _ctx_after)
            # Deduplicate repeated DOCX text artefacts
            _ctx_dedup = re.sub(r"(\S.{10,80})\1+", r"\1", _ctx)

            _fig_m = _FIGURE_RE.search(_ctx_dedup)
            _fig_label = re.sub(r"\s+", " ", _fig_m.group(1)).strip() if _fig_m else None

            _img = dict(_rel_imgs[_embed])
            _img["page"]      = _pinfo["page"]
            _img["fig_label"] = _fig_label
            _img["context"]   = _ctx_dedup[:300]
            images.append(_img)

    return pages, images


def _parse_txt(path: str) -> tuple[list[dict], list[dict]]:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    return [{"page_num": 1, "text": text, "headings": []}], []


def _parse_markdown(path: str) -> tuple[list[dict], list[dict]]:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    # Extract headings
    headings = re.findall(r'^#{1,3}\s+(.+)$', text, re.MULTILINE)
    return [{"page_num": 1, "text": text, "headings": headings}], []


def _parse_html(path: str) -> tuple[list[dict], list[dict]]:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()
    if _bs4:
        soup = BeautifulSoup(raw, "html.parser")
        # Remove script/style
        for tag in soup(["script", "style", "nav", "footer", "head"]):
            tag.decompose()
        # Extract headings
        headings = [h.get_text(strip=True) for h in soup.find_all(["h1", "h2", "h3"])]
        # Tables as markdown
        table_texts = []
        for table in soup.find_all("table"):
            rows = []
            for row in table.find_all("tr"):
                cells = [td.get_text(strip=True) for td in row.find_all(["td", "th"])]
                if any(cells):
                    rows.append(cells)
            if rows:
                lines = []
                if len(rows) > 1:
                    lines.append("| " + " | ".join(rows[0]) + " |")
                    lines.append("| " + " | ".join(["---"] * len(rows[0])) + " |")
                    for r in rows[1:]:
                        lines.append("| " + " | ".join(r) + " |")
                table_texts.append("\n[TABLE]\n" + "\n".join(lines) + "\n[/TABLE]\n")
                table.replace_with("\n" + "\n".join(lines) + "\n")
        text = soup.get_text(separator="\n", strip=True)
        return [{"page_num": 1, "text": text, "headings": headings}], []
    else:
        # Fallback: strip tags
        text = re.sub(r'<[^>]+>', ' ', raw)
        text = re.sub(r'\s+', ' ', text).strip()
        return [{"page_num": 1, "text": text, "headings": []}], []


def _parse_csv(path: str) -> tuple[list[dict], list[dict]]:
    if not _pandas:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return [{"page_num": 1, "text": f.read(), "headings": []}], []

    df = pd.read_csv(path)
    rows, cols = df.shape
    text_parts = [f"CSV Data — {rows} rows × {cols} columns\n"]
    text_parts.append("Columns: " + ", ".join(str(c) for c in df.columns) + "\n")

    # Full markdown table
    text_parts.append("\n[TABLE]\n")
    text_parts.append("| " + " | ".join(str(c) for c in df.columns) + " |")
    text_parts.append("| " + " | ".join(["---"] * len(df.columns)) + " |")
    for _, row in df.head(500).iterrows():
        text_parts.append("| " + " | ".join(str(v) for v in row) + " |")
    text_parts.append("[/TABLE]\n")

    # Numeric summary
    try:
        desc = df.describe(include='all')
        text_parts.append("\nStatistical Summary:\n")
        for col in desc.columns:
            stats = desc[col].dropna()
            stat_str = ", ".join(f"{k}: {v}" for k, v in stats.items())
            text_parts.append(f"  {col}: {stat_str}")
    except Exception:
        pass

    return [{"page_num": 1, "text": "\n".join(text_parts), "headings": [f"CSV: {Path(path).name}"]}], []


def _parse_xlsx(path: str) -> tuple[list[dict], list[dict]]:
    if not _pandas:
        return [{"page_num": 1, "text": "XLSX parsing requires pandas.", "headings": []}], []

    pages = []
    xl = pd.ExcelFile(path)
    for sheet_num, sheet_name in enumerate(xl.sheet_names):
        df = xl.parse(sheet_name)
        rows, cols = df.shape
        text_parts = [f"Sheet: {sheet_name} — {rows} rows × {cols} columns\n"]
        text_parts.append("Columns: " + ", ".join(str(c) for c in df.columns) + "\n")

        text_parts.append("\n[TABLE]\n")
        text_parts.append("| " + " | ".join(str(c) for c in df.columns) + " |")
        text_parts.append("| " + " | ".join(["---"] * len(df.columns)) + " |")
        for _, row in df.head(500).iterrows():
            text_parts.append("| " + " | ".join(str(v) for v in row) + " |")
        text_parts.append("[/TABLE]\n")

        try:
            desc = df.describe(include='all')
            text_parts.append("\nStatistical Summary:\n")
            for col in desc.columns:
                stats = desc[col].dropna()
                stat_str = ", ".join(f"{k}: {v}" for k, v in stats.items())
                text_parts.append(f"  {col}: {stat_str}")
        except Exception:
            pass

        pages.append({
            "page_num": sheet_num + 1,
            "text": "\n".join(text_parts),
            "headings": [f"Sheet: {sheet_name}"],
        })

    return pages, []


def _parse_pptx(path: str) -> tuple[list[dict], list[dict]]:
    if not _pptx:
        return [{"page_num": 1, "text": "PPTX parsing requires python-pptx.", "headings": []}], []

    prs = _PPTXPresentation(path)
    pages = []
    for slide_num, slide in enumerate(prs.slides):
        parts = []
        headings = []
        for shape in slide.shapes:
            if not shape.has_text_frame:
                continue
            for para in shape.text_frame.paragraphs:
                text = para.text.strip()
                if not text:
                    continue
                # Heuristic: large/bold text on first shape = title/heading
                if shape.shape_type == 13 or (hasattr(shape, 'placeholder_format') and
                   shape.placeholder_format and shape.placeholder_format.idx == 0):
                    headings.append(text)
                    parts.append(f"\n## {text}\n")
                else:
                    parts.append(text)

        # Tables in slides
        for shape in slide.shapes:
            if shape.has_table:
                table = shape.table
                rows = []
                for row in table.rows:
                    cells = [cell.text.strip() for cell in row.cells]
                    if any(cells):
                        rows.append(cells)
                if rows:
                    lines = []
                    if len(rows) > 1:
                        lines.append("| " + " | ".join(rows[0]) + " |")
                        lines.append("| " + " | ".join(["---"] * len(rows[0])) + " |")
                        for r in rows[1:]:
                            lines.append("| " + " | ".join(r) + " |")
                    parts.append("\n[TABLE]\n" + "\n".join(lines) + "\n[/TABLE]\n")

        if parts:
            pages.append({
                "page_num": slide_num + 1,
                "text": "\n".join(parts),
                "headings": headings,
            })

    return pages if pages else [{"page_num": 1, "text": "No text found.", "headings": []}], []


def parse_document(file_path: str) -> tuple[list[dict], list[dict]]:
    ext = Path(file_path).suffix.lower()
    parsers = {
        ".pdf":  _parse_pdf,
        ".docx": _parse_docx,
        ".txt":  _parse_txt,
        ".md":   _parse_markdown,
        ".html": _parse_html,
        ".htm":  _parse_html,
        ".csv":  _parse_csv,
        ".xlsx": _parse_xlsx,
        ".xls":  _parse_xlsx,
        ".pptx": _parse_pptx,
    }
    parser = parsers.get(ext)
    if not parser:
        raise ValueError(f"Unsupported file type: {ext}. Supported: {', '.join(parsers)}")
    return parser(file_path)


# ══════════════════════════════════════════════════════════════
#  STEP 2: CHUNKING
# ══════════════════════════════════════════════════════════════

def _detect_numerical_context(text: str) -> list[str]:
    patterns = [
        r'\b\d{4}\b',
        r'\d+\.?\d*\s*%',
        r'\$[\d,]+\.?\d*[BMK]?\b',
        r'\b\d+\.?\d*\s*(billion|million|trillion|thousand)\b',
        r'(?:Table|Figure|Section|Equation|Eq\.?)\s*[\d]+(?:\.[\d]+)*',
    ]
    found = []
    for p in patterns:
        found.extend(re.findall(p, text, re.IGNORECASE))
    return list(set(found))


def _sentence_split(text: str) -> list[str]:
    text = re.sub(r'\n{2,}', ' <PARA> ', text)
    text = re.sub(r'\n', ' ', text)
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
    result = []
    for s in sentences:
        parts = s.split('<PARA>')
        for p in parts:
            p = p.strip()
            if p:
                result.append(p)
    return result


def _sliding_window_chunks(sentences: list[str], chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP) -> list[str]:
    chunks = []
    current: list[str] = []
    current_words = 0

    for sent in sentences:
        words = sent.split()
        if current_words + len(words) > chunk_size and current:
            chunks.append(" ".join(current))
            overlap_sents = []
            overlap_count = 0
            for s in reversed(current):
                overlap_sents.insert(0, s)
                overlap_count += len(s.split())
                if overlap_count >= overlap:
                    break
            current = overlap_sents + [sent]
            current_words = sum(len(s.split()) for s in current)
        else:
            current.append(sent)
            current_words += len(words)

    if current:
        chunks.append(" ".join(current))

    return [c for c in chunks if len(c.split()) >= MIN_CHUNK_LEN]


_AUTHOR_PATTERNS = [
    re.compile(r'\b(?:by|authors?|written\s+by|prepared\s+by|submitted\s+by)\b', re.I),
    re.compile(r'[A-Z][a-z]+\s+[A-Z][a-z]+(?:\s*,\s*[A-Z][a-z]+\s+[A-Z][a-z]+)+'),
    re.compile(r'[A-Z][a-z]+\s+[A-Z]\.\s*[A-Z][a-z]+'),
    re.compile(r'\b(?:department|university|institute|college|faculty|school)\b', re.I),
    re.compile(r'\b[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}\b', re.I),
    re.compile(r'\b(?:abstract|introduction|keywords?)\b', re.I),
]

def _is_frontmatter(text: str, page_num: int) -> bool:
    if page_num > 2:
        return False
    score = sum(1 for p in _AUTHOR_PATTERNS if p.search(text))
    return score >= 2


def chunk_pages(pages: list[dict], filename: str) -> list[dict]:
    all_chunks = []

    for page in pages:
        text = page["text"]
        page_num = page["page_num"]
        page_headings = page["headings"]
        frontmatter = _is_frontmatter(text, page_num)

        # Keep table blocks as dedicated chunks (never split them)
        table_pattern = re.compile(r'\[TABLE\](.*?)\[/TABLE\]', re.DOTALL)
        table_chunks_raw = table_pattern.findall(text)
        text_without_tables = table_pattern.sub("[TABLE_PLACEHOLDER]", text)

        sections = re.split(r'\n##\s+', text_without_tables)
        current_section_title = page_headings[0] if page_headings else filename

        for section in sections:
            if not section.strip():
                continue
            lines = section.split("\n", 1)
            if len(lines) == 2 and len(lines[0]) < 120:
                current_section_title = lines[0].strip() or current_section_title
                section_body = lines[1]
            else:
                section_body = section

            # Replace table placeholders with actual table content
            for tbl_text in table_chunks_raw:
                if "[TABLE_PLACEHOLDER]" in section_body:
                    section_body = section_body.replace("[TABLE_PLACEHOLDER]", tbl_text, 1)
                    # Emit the table as its own dedicated chunk
                    tbl_chunk_text = tbl_text.strip()
                    if len(tbl_chunk_text.split()) >= MIN_CHUNK_LEN:
                        # Extract caption if present (line before [TABLE] tag)
                        caption_match = re.search(
                            r'Table caption:\s*([^\n]+)', tbl_chunk_text
                        )
                        caption = caption_match.group(1).strip() if caption_match else ""
                        # Label the chunk with its caption so retrieval can match "Table IV" etc.
                        # Self-caption: if the table text itself starts with "Table X.Y: ..."
                        # (common in DOCX where the caption is inside the table cell),
                        # extract it and use it as caption if we don't have one yet.
                        _TABLE_SELF_ID_RE = re.compile(
                            r'^\[?(?:table|tbl|tab)\.?\s*([\d]+(?:\.[\d]+)*|[IVXLC]+)[\]:\s]',
                            re.IGNORECASE
                        )
                        if not caption:
                            _sm = _TABLE_SELF_ID_RE.search(tbl_chunk_text.strip())
                            if _sm:
                                caption = tbl_chunk_text.strip()[:80].split("\n")[0]
                        # Also check section title for table ID
                        if not caption and current_section_title:
                            _TABLE_CAPTION_RE2 = re.compile(
                                r'\b(?:table|tbl|tab)\.?\s*([\d]+(?:\.[\d]+)*|[IVXLC]+)',
                                re.IGNORECASE
                            )
                            if _TABLE_CAPTION_RE2.search(current_section_title):
                                caption = current_section_title
                        table_label = caption if caption else f"Table from {filename}, Page {page_num}"
                        all_chunks.append({
                            "text": f"[TABLE: {table_label}]\n{tbl_chunk_text}",
                            "page_num": page_num,
                            "section": current_section_title,
                            "chunk_local_idx": len(all_chunks),
                            "numerics": _detect_numerical_context(tbl_chunk_text),
                            "content_hash": hashlib.sha256(tbl_chunk_text.encode()).hexdigest()[:16],
                            "is_table": True,
                            "is_frontmatter": frontmatter,
                            "table_caption": caption,
                        })

            sentences = _sentence_split(section_body)
            chunks = _sliding_window_chunks(sentences)

            for i, chunk_text in enumerate(chunks):
                numerics = _detect_numerical_context(chunk_text)
                content_hash = hashlib.sha256(chunk_text.encode()).hexdigest()[:16]
                all_chunks.append({
                    "text": chunk_text,
                    "page_num": page_num,
                    "section": current_section_title,
                    "chunk_local_idx": i,
                    "numerics": numerics,
                    "content_hash": content_hash,
                    "is_table": False,
                    "is_frontmatter": frontmatter,
                })

    return all_chunks


# ══════════════════════════════════════════════════════════════
#  STEP 3: DEDUPLICATION
# ══════════════════════════════════════════════════════════════

def _jaccard_similarity(a: str, b: str) -> float:
    wa = set(a.lower().split())
    wb = set(b.lower().split())
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)


def deduplicate_chunks(chunks: list[dict], threshold: float = 0.88) -> list[dict]:
    seen_hashes = set()
    unique_chunks = []

    for chunk in chunks:
        h = chunk["content_hash"]
        if h in seen_hashes:
            continue
        # Always keep table chunks
        if chunk.get("is_table"):
            seen_hashes.add(h)
            unique_chunks.append(chunk)
            continue

        is_near_dup = False
        for prev in unique_chunks[-15:]:
            if prev.get("is_table"):
                continue
            if _jaccard_similarity(chunk["text"], prev["text"]) >= threshold:
                is_near_dup = True
                break

        if not is_near_dup:
            seen_hashes.add(h)
            unique_chunks.append(chunk)

    removed = len(chunks) - len(unique_chunks)
    if removed:
        print(f"[Dedup] Removed {removed} duplicate chunks")

    return unique_chunks


# ══════════════════════════════════════════════════════════════
#  STEP 4: EMBED  (batch, single model instance)
# ══════════════════════════════════════════════════════════════

def embed(text: str) -> list[float]:
    vec = _embed_model.encode(text, normalize_embeddings=True)
    return vec.tolist()


def embed_batch(texts: list[str]) -> list[list[float]]:
    # GTX 1650 has 4GB VRAM — batch 128 is safe; CPU falls back to 64
    _batch_size = 128 if _DEVICE == "cuda" else 64
    vecs = _embed_model.encode(
        texts,
        normalize_embeddings=True,
        batch_size=_batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    return vecs.tolist()


# ══════════════════════════════════════════════════════════════
#  STEP 5: STORE IN FAISS + JSON METADATA
# ══════════════════════════════════════════════════════════════

def store_in_vector_store(
    chunks: list[dict],
    file_id: str,
    filename: str,
    images: list[dict],
    version: int,
    ingested_at: str,
) -> None:
    index, metadata = _get_store()

    texts = [chunk["text"] for chunk in chunks]
    print(f"[VectorStore] Embedding {len(texts)} chunks (batch_size=64)...")
    t0 = time.time()
    embeddings = embed_batch(texts)
    print(f"[VectorStore] Embedding done in {time.time()-t0:.1f}s")

    vectors_to_add = []
    new_meta = []

    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        chunk_id = f"{file_id}_v{version}_chunk_{i}"
        vectors_to_add.append(emb)
        new_meta.append({
            "id":            chunk_id,
            "text":          chunk["text"],
            "file_id":       file_id,
            "filename":      filename,
            "version":       version,
            "ingested_at":   ingested_at,
            "chunk_index":   i,
            "total_chunks":  len(chunks),
            "page_num":      chunk["page_num"],
            "section":       chunk["section"][:200],
            "content_hash":  chunk["content_hash"],
            "numerics":      json.dumps(chunk["numerics"][:10]),
            "is_table":      chunk.get("is_table", False),
            "is_frontmatter": chunk.get("is_frontmatter", False),
            "table_caption": chunk.get("table_caption", ""),
            "type":          "text",
        })

    # Image captioning via moondream2 — real descriptions, not placeholders
    if images:
        capped = images[:MAX_IMAGES_PER_DOC]
        print(f"[VectorStore] Captioning {len(capped)} images with BLIP...")
        img_texts = []
        t_vis = time.time()
        for i, img in enumerate(capped):
            try:
                caption = _describe_image_b64(img["b64"], img.get("ext", "png"))
            except Exception as e:
                caption = "Visual content"
                print(f"[Vision] Image {i} caption failed: {e}")
            fig_tag = f" ({img['fig_label']})" if img.get("fig_label") else ""
            full_text = f"[IMAGE p.{img.get('page', i+1)}{fig_tag}] {caption}"
            img_texts.append(full_text)
            print(f"[Vision] {i+1}/{len(capped)}: {caption[:80]}")
        print(f"[Vision] Captioned {len(capped)} images in {time.time()-t_vis:.1f}s")

        img_embeddings = embed_batch(img_texts)
        for i, (img, img_emb) in enumerate(zip(capped, img_embeddings)):
            img_id = f"{file_id}_v{version}_img_{i}"
            vectors_to_add.append(img_emb)
            new_meta.append({
                "id":          img_id,
                "text":        img_texts[i],
                "file_id":     file_id,
                "filename":    filename,
                "version":     version,
                "ingested_at": ingested_at,
                "type":        "image",
                "page":        str(img.get("page", i + 1)),
                "ext":         img.get("ext", "png"),
                "b64_preview": img["b64"][:2000],
                "caption":     img_texts[i],
            })

    if vectors_to_add:
        mat = np.array(vectors_to_add, dtype="float32")
        index.add(mat)
        metadata.extend(new_meta)
        _save_store()
        print(f"[VectorStore] ✓ Stored {len(vectors_to_add)} entries (file_id={file_id}, v{version})")


# ══════════════════════════════════════════════════════════════
#  VERSIONING & METADATA
# ══════════════════════════════════════════════════════════════

def _get_doc_meta(file_id: str) -> dict:
    meta_path = META_DIR / f"{file_id}.json"
    if meta_path.exists():
        with open(meta_path) as f:
            return json.load(f)
    return {}


def _save_doc_meta(file_id: str, meta: dict):
    with open(META_DIR / f"{file_id}.json", "w") as f:
        json.dump(meta, f, indent=2)


def _compute_file_hash(file_path: str) -> str:
    h = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


# ══════════════════════════════════════════════════════════════
#  MAIN ENTRYPOINT
# ══════════════════════════════════════════════════════════════

SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".txt", ".md", ".html", ".htm", ".csv", ".xlsx", ".xls", ".pptx"}


def ingest_document(file_path: str, file_id: str, original_filename: str) -> dict:
    print(f"\n[Ingestion] ═══ Starting: {original_filename} ═══")

    ingested_at = datetime.now(timezone.utc).isoformat()
    file_hash   = _compute_file_hash(file_path)

    existing_meta = _get_doc_meta(file_id)
    version = existing_meta.get("version", 0) + 1

    # ── Step 1: Parse ────────────────────────────────────────
    t_parse = time.time()
    print("[Ingestion] 1/4 Parsing document...")
    pages, images = parse_document(file_path)
    total_text = " ".join(p["text"] for p in pages)
    parse_time = time.time() - t_parse
    print(f"[Ingestion]   → {len(pages)} pages/sheets, {len(total_text):,} chars, {len(images)} images  [{parse_time:.2f}s]")
    for i, page in enumerate(pages):
        fm = _is_frontmatter(page["text"], page["page_num"])
        hdgs = page.get("headings", [])
        hdg_str = " | ".join(hdgs[:3]) if hdgs else "(no headings)"
        flag = " ★FRONTMATTER" if fm else ""
        print(f"[Ingestion]   Page {page['page_num']:>2}: {len(page['text']):>6} chars  headings=[{hdg_str}]{flag}")

    if not total_text.strip():
        raise ValueError("Document appears empty or unreadable.")

    # ── Step 2: Chunk ────────────────────────────────────────
    t_chunk = time.time()
    print("\n[Ingestion] 2/4 Chunking (structure-aware + table-preserving)...")
    chunks = chunk_pages(pages, original_filename)
    chunk_time = time.time() - t_chunk
    tables  = sum(1 for c in chunks if c.get("is_table"))
    fm_cnt  = sum(1 for c in chunks if c.get("is_frontmatter"))
    print(f"[Ingestion]   → {len(chunks)} chunks total  ({tables} tables, {fm_cnt} frontmatter)  [{chunk_time:.2f}s]")
    print(f"[Ingestion]   Chunk breakdown:")
    for idx, c in enumerate(chunks[:8]):   # show first 8
        words = len(c["text"].split())
        tag   = " [TABLE]" if c.get("is_table") else ""
        tag  += " [FM]"    if c.get("is_frontmatter") else ""
        preview = c["text"][:80].replace("\n", " ")
        print(f"[Ingestion]     chunk {idx+1:>3} | p{c['page_num']} | {words:>4}w | {c['section'][:30]:<30}{tag}")
        print(f"[Ingestion]             └─ \"{preview}...\"")
    if len(chunks) > 8:
        print(f"[Ingestion]     ... +{len(chunks)-8} more chunks")

    # ── Step 3: Dedup ────────────────────────────────────────
    t_dedup = time.time()
    print("\n[Ingestion] 3/4 Deduplicating...")
    chunks = deduplicate_chunks(chunks)
    dedup_time = time.time() - t_dedup
    print(f"[Ingestion]   → {len(chunks)} chunks after dedup  [{dedup_time:.2f}s]")

    # ── Step 4: Embed + Store ────────────────────────────────
    t_embed = time.time()
    print(f"\n[Ingestion] 4/4 Embedding + storing...")
    print(f"[Ingestion]   Model : {EMBED_MODEL_NAME}  |  Device: {_DEVICE.upper()}")
    print(f"[Ingestion]   Batch size: {'128 (GPU)' if _DEVICE=='cuda' else '64 (CPU)'}")
    store_in_vector_store(chunks, file_id, original_filename, images, version, ingested_at)
    embed_time = time.time() - t_embed
    print(f"[Ingestion]   Embed+store done  [{embed_time:.2f}s]")

    doc_meta = {
        "file_id":      file_id,
        "filename":     original_filename,
        "file_path":    file_path,
        "file_hash":    file_hash,
        "version":      version,
        "ingested_at":  ingested_at,
        "total_chunks": len(chunks),
        "total_images": len(images),
        "total_pages":  len(pages),
        "sections":     list(set(c["section"] for c in chunks)),
    }
    _save_doc_meta(file_id, doc_meta)

    total_time = parse_time + chunk_time + dedup_time + embed_time
    print(f"\n[Ingestion] ✓ Done: {len(chunks)} chunks | {len(images)} images | v{version}")
    print(f"[Ingestion]   Parse:{parse_time:.1f}s  Chunk:{chunk_time:.1f}s  Dedup:{dedup_time:.1f}s  Embed:{embed_time:.1f}s  TOTAL:{total_time:.1f}s\n")
    return {
        "chunks":      len(chunks),
        "images":      len(images),
        "version":     version,
        "sections":    doc_meta["sections"],
        "_chunks_ref": chunks,
    }

Overwriting /content/singularity/ingestion.py


### 5b — retrieval.py
*Retrieval — BM25 + FAISS hybrid search with GPU embeddings*

In [26]:
%%writefile /content/singularity/retrieval.py
"""
Enterprise Retrieval Module  v4.0
====================================
Key speed improvements over v3:
  - BM25 index CACHED per file_id set — not rebuilt on every query
  - FAISS sub-index CACHED — not rebuilt by reconstruct() on every query
  - Embed model shared from ingestion (single global instance)
  - Table chunks boosted in re-ranking
  - Reduced query expansion calls (was running multiple LLM query rewrites per request)
"""

import re
import math
import json
import numpy as np
from pathlib import Path
from typing import Optional
from collections import defaultdict, Counter
from datetime import datetime, timezone

import faiss
from sentence_transformers import SentenceTransformer

BASE_DIR = Path(__file__).parent

import torch as _torch
_DEVICE = "cuda" if _torch.cuda.is_available() else "cpu"

_memory_available = False
try:
    from memory import (
        parse_temporal_query, apply_temporal_filter,
        build_temporal_context_note, log_chunk_access,
    )
    from knowledge_graph import traverse_graph
    _memory_available = True
except ImportError:
    pass

VECTOR_DIR       = BASE_DIR / "vector_store"
META_DIR         = BASE_DIR / "doc_meta"
VECTOR_DIR.mkdir(exist_ok=True)
META_DIR.mkdir(exist_ok=True)

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
EMBED_DIM        = 384
FAISS_INDEX_FILE = VECTOR_DIR / "index.faiss"
METADATA_FILE    = VECTOR_DIR / "metadata.json"

# ── Single global model instance (reused from ingestion if possible) ──
_embed_model = None

def _get_embed_model() -> SentenceTransformer:
    global _embed_model
    if _embed_model is None:
        # Try to reuse ingestion module's already-loaded model (same process = same GPU)
        try:
            import ingestion as _ing
            _embed_model = _ing._embed_model
            print("[Retrieval] Reused ingestion embed model (no reload).")
        except Exception:
            print(f"[Retrieval] Loading sentence-transformer model on {_DEVICE.upper()}...")
            _embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=_DEVICE)
    return _embed_model


# ── FAISS store (with mtime-based reload) ──
_faiss_index = None
_metadata_store: list[dict] = []
_store_mtime: float = 0.0

def _load_store():
    global _faiss_index, _metadata_store, _store_mtime, _bm25_cache, _sub_index_cache
    if FAISS_INDEX_FILE.exists() and METADATA_FILE.exists():
        _faiss_index = faiss.read_index(str(FAISS_INDEX_FILE))
        with open(METADATA_FILE) as f:
            _metadata_store = json.load(f)
        _store_mtime = METADATA_FILE.stat().st_mtime
    else:
        _faiss_index = faiss.IndexFlatIP(EMBED_DIM)
        _metadata_store = []
        _store_mtime = 0.0
    # Invalidate caches on reload
    _bm25_cache.clear()
    _sub_index_cache.clear()


def _get_store():
    global _faiss_index, _metadata_store, _store_mtime
    if _faiss_index is None:
        _load_store()
        return _faiss_index, _metadata_store
    if METADATA_FILE.exists() and METADATA_FILE.stat().st_mtime > _store_mtime:
        _load_store()
    return _faiss_index, _metadata_store


def _embed_query(text: str) -> np.ndarray:
    model = _get_embed_model()
    vec = model.encode(text, normalize_embeddings=True)
    return vec.astype("float32")


# ══════════════════════════════════════════════════════════════
#  CACHE LAYER  — BM25 and sub-FAISS index per (file_id, corpus_hash)
# ══════════════════════════════════════════════════════════════

_bm25_cache: dict[str, tuple] = {}       # key → (BM25, ids, metas, texts)
_sub_index_cache: dict[str, tuple] = {}  # key → (sub_index, ids, metas, texts)


def _cache_key(file_id: Optional[str], metadata: list[dict]) -> str:
    """A cache key that changes when the store grows (new doc ingested)."""
    n = sum(1 for m in metadata if m.get("type") in ("text", "image") and
            (file_id is None or m.get("file_id") == file_id))
    return f"{file_id or 'ALL'}_{n}"


# ══════════════════════════════════════════════════════════════
#  BM25
# ══════════════════════════════════════════════════════════════

class BM25:
    def __init__(self, corpus: list[str], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b  = b
        self.corpus_size = len(corpus)
        self.tokenized = [self._tokenize(doc) for doc in corpus]
        self.avgdl = sum(len(d) for d in self.tokenized) / max(self.corpus_size, 1)
        self._build_idf()

    def _tokenize(self, text: str) -> list[str]:
        return re.sub(r'[^a-z0-9\s]', ' ', text.lower()).split()

    def _build_idf(self):
        self.idf = {}
        df = Counter()
        for doc in self.tokenized:
            for term in set(doc):
                df[term] += 1
        for term, freq in df.items():
            self.idf[term] = math.log(
                (self.corpus_size - freq + 0.5) / (freq + 0.5) + 1.0
            )

    def get_scores(self, query: str) -> list[float]:
        query_terms = self._tokenize(query)
        scores = []
        for doc_terms in self.tokenized:
            score = 0.0
            doc_len = len(doc_terms)
            term_freq = Counter(doc_terms)
            for term in query_terms:
                if term not in self.idf:
                    continue
                tf = term_freq.get(term, 0)
                num = tf * (self.k1 + 1)
                den = tf + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
                score += self.idf[term] * (num / den)
            scores.append(score)
        return scores

    def get_top_n(self, query: str, n: int) -> list[tuple[int, float]]:
        scores = self.get_scores(query)
        return sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:n]


def _get_bm25(file_id: Optional[str], index, metadata) -> tuple:
    """Return cached (BM25, ids, metas, texts) or build + cache."""
    key = _cache_key(file_id, metadata)
    if key in _bm25_cache:
        return _bm25_cache[key]

    entries = [
        m for m in metadata
        if m.get("type") in ("text", "image") and (file_id is None or m.get("file_id") == file_id)
    ]
    texts = [m["text"] for m in entries]
    ids   = [m["id"] for m in entries]
    bm25  = BM25(texts)
    result = (bm25, ids, entries, texts)
    _bm25_cache[key] = result
    return result


def _get_sub_index(file_id: Optional[str], index, metadata, entry_indices) -> tuple:
    """Return cached sub-FAISS index or build + cache using direct index reconstruction."""
    key = _cache_key(file_id, metadata)
    if key in _sub_index_cache:
        return _sub_index_cache[key]

    entries = [metadata[i] for i in entry_indices]
    ids   = [m["id"] for m in entries]
    texts = [m["text"] for m in entries]

    # Build sub-index by embedding texts directly (avoids reconstruct() failures)
    # For large stores this is done once and cached
    print(f"[Retrieval] Building sub-index for {len(entry_indices)} entries...")
    model = _get_embed_model()
    _batch = 128 if _DEVICE == "cuda" else 64
    vecs = model.encode(texts, normalize_embeddings=True, batch_size=_batch, show_progress_bar=False, convert_to_numpy=True)
    vecs = vecs.astype("float32")

    sub_index = faiss.IndexFlatIP(EMBED_DIM)
    sub_index.add(vecs)

    result = (sub_index, ids, entries, texts)
    _sub_index_cache[key] = result
    return result


# ══════════════════════════════════════════════════════════════
#  QUERY EXPANSION (no LLM — purely rule-based, zero latency)
# ══════════════════════════════════════════════════════════════

FRONTMATTER_TRIGGER_WORDS = {
    "author", "authors", "wrote", "written", "researcher", "researchers",
    "contributor", "contributors", "affiliation", "university", "institution",
    "department", "abstract", "title", "paper", "manuscript", "publication",
    "journal", "submitted", "email", "who wrote", "who are", "name", "names",
}

def _is_author_query(query: str) -> bool:
    q = query.lower()
    return any(w in q for w in FRONTMATTER_TRIGGER_WORDS)


def _expand_query(query: str) -> list[str]:
    queries = [query]
    words = re.findall(r'\b[A-Za-z][a-z]{2,}\b', query)
    if len(words) >= 2:
        queries.append(" ".join(words))
    # For author/metadata queries always add frontmatter probes
    if _is_author_query(query):
        queries.append("authors names affiliations")
        queries.append("author affiliation email department university")
    return list(dict.fromkeys(queries))


def _decompose_query(query: str) -> list[str]:
    sub_queries = [query]
    if re.search(r'\b(acquired|founded|created|built|owns|led|compare|versus|vs)\b', query, re.IGNORECASE):
        nouns = re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)?\b', query)
        for noun in nouns[:2]:
            sub_queries.append(f"information about {noun}")
    return sub_queries


# ══════════════════════════════════════════════════════════════
#  RECIPROCAL RANK FUSION
# ══════════════════════════════════════════════════════════════

def _reciprocal_rank_fusion(rankings: list[list[tuple[str, float]]], k: int = 60) -> list[tuple[str, float]]:
    fusion: dict[str, float] = defaultdict(float)
    for ranking in rankings:
        for rank, (doc_id, _) in enumerate(ranking):
            fusion[doc_id] += 1.0 / (k + rank + 1)
    return sorted(fusion.items(), key=lambda x: x[1], reverse=True)


# ══════════════════════════════════════════════════════════════
#  RE-RANKING  (table chunks get a boost)
# ══════════════════════════════════════════════════════════════

def _rerank_chunks(query: str, chunks: list[dict]) -> list[dict]:
    query_lower  = query.lower()
    query_terms  = set(re.findall(r'\b\w{3,}\b', query_lower))
    query_numbers = set(re.findall(r'\b\d+\.?\d*\b', query))

    # Detect frontmatter-seeking queries (authors, title, affiliation, abstract)
    FRONTMATTER_KEYWORDS = {
        "author", "authors", "written", "wrote", "researcher", "researchers",
        "contributor", "contributors", "name", "names", "who", "affiliation",
        "university", "institution", "department", "abstract", "title", "paper",
        "study", "manuscript", "publication", "journal", "submitted", "email"
    }
    is_frontmatter_query = bool(query_terms & FRONTMATTER_KEYWORDS)

    for chunk in chunks:
        text_lower = chunk["text"].lower()
        meta = chunk.get("metadata", {})

        chunk_terms   = set(re.findall(r'\b\w{3,}\b', text_lower))
        overlap       = len(query_terms & chunk_terms) / max(len(query_terms), 1)
        chunk_numbers = set(re.findall(r'\b\d+\.?\d*\b', chunk["text"]))
        num_overlap   = len(query_numbers & chunk_numbers) / max(len(query_numbers), 1) if query_numbers else 0

        section = meta.get("section", "").lower()
        section_match = any(t in section for t in query_terms) * 0.2

        # Table boost — fire for ANY table-related query
        table_boost = 0.0
        table_name_match = bool(re.search(r'\btable\s*([\d]+(?:\.[\d]+)*|[IVXLCivxlc]+)', query_lower))
        is_table_chunk = meta.get("is_table") or "[TABLE" in chunk["text"]
        if is_table_chunk:
            table_boost = 0.10
            table_data_keywords = {"table", "data", "value", "statistic", "figure", "number",
                                    "column", "row", "percentage", "total", "average", "mean",
                                    "count", "result", "results", "metric", "metrics", "score",
                                    "accuracy", "performance", "comparison", "compare", "finding"}
            table_name_match = bool(re.search(r'\btable\s*([\d]+(?:\.[\d]+)*|[IVXLCivxlc]+)', query_lower))
            if query_terms & table_data_keywords or query_numbers or table_name_match:
                table_boost = 0.35
            # STRONG boost: query names this specific table (caption OR body match)
            if table_name_match:
                qmatch = re.search(r'\btable\s*([\d]+(?:\.[\d]+)*|[IVXLCivxlc]+)', query_lower)
                if qmatch:
                    q_table_id = qmatch.group(1).lower()
                    caption = meta.get("table_caption", "").lower()
                    chunk_text_lower = chunk["text"].lower()
                    # Match against caption text
                    caption_hit = caption and (
                        q_table_id in caption or
                        re.search(rf'\btable\s*{re.escape(q_table_id)}\b', caption)
                    )
                    # Also match against chunk body (table header row often has the name)
                    body_hit = (
                        q_table_id in chunk_text_lower or
                        re.search(rf'\btable\s*{re.escape(q_table_id)}\b', chunk_text_lower)
                    )
                    if caption_hit or body_hit:
                        table_boost = 0.80  # very strong: exact table ID match
                    elif is_table_chunk:
                        table_boost = 0.40  # moderate: it's a table but ID not matched

        # Frontmatter boost — strongly surface page-1 chunks for author/title queries
        frontmatter_boost = 0.0
        if is_frontmatter_query and meta.get("is_frontmatter"):
            frontmatter_boost = 0.5

        # TOC penalty — demote table-of-contents index chunks for specific table queries.
        # TOC rows look like: | Table 4.1 | Description | 38 |  (page number as last col)
        toc_penalty = 0.0
        if table_name_match and is_table_chunk:
            txt = chunk["text"]
            toc_rows = re.findall(r'\|([^|\n]+)\|([^|\n]+)\|([^|\n]{1,8})\|', txt)
            page_num_cells = sum(1 for r in toc_rows if re.match(r'^\s*\d{1,3}\s*$', r[2]))
            if toc_rows and page_num_cells / max(len(toc_rows), 1) > 0.5:
                toc_penalty = -0.60

        rerank_score = (
            chunk.get("vector_score", 0) * 0.5
            + chunk.get("bm25_score", 0) * 0.3
            + overlap * 0.1
            + num_overlap * 0.05
            + section_match
            + table_boost
            + frontmatter_boost
            + toc_penalty
            + chunk.get("feedback_boost", 0)
        )
        chunk["rerank_score"] = rerank_score

    return sorted(chunks, key=lambda c: c["rerank_score"], reverse=True)


# ══════════════════════════════════════════════════════════════
#  CONTEXT ASSEMBLY
# ══════════════════════════════════════════════════════════════

def _assemble_context(chunks: list[dict], max_words: int = 2800) -> str:
    """
    Assembles retrieved chunks into a context string.

    When multiple documents are present, chunks are grouped by document with
    clear document-level headers so the LLM can unambiguously attribute each
    piece of information to the correct source and perform cross-doc comparisons.
    """
    seen_hashes: set = set()
    total_words = 0

    # Group chunks by filename while preserving relevance order within each doc
    doc_order: list[str] = []         # first-seen order (by relevance)
    doc_chunks: dict[str, list] = {}  # filename → list of (chunk, meta)

    for chunk in chunks:
        content_hash = chunk.get("metadata", {}).get("content_hash", "")
        text  = chunk["text"]
        words = len(text.split())

        if total_words + words > max_words:
            break
        if content_hash and content_hash in seen_hashes:
            continue
        if content_hash:
            seen_hashes.add(content_hash)

        meta   = chunk.get("metadata", {})
        source = meta.get("filename", "Unknown")

        if source not in doc_chunks:
            doc_order.append(source)
            doc_chunks[source] = []
        doc_chunks[source].append((chunk, meta))
        total_words += words

    # Build output — one clearly-labelled block per document
    doc_blocks = []
    multi_doc = len(doc_chunks) > 1

    for source in doc_order:
        entries = doc_chunks[source]
        chunk_parts = []

        for chunk, meta in entries:
            page    = meta.get("page_num", "?")
            section = meta.get("section", "")
            is_tbl  = meta.get("is_table", False)
            score   = round(chunk.get("rerank_score", 0), 3)

            loc = f"Page {page}"
            if section and section != source and len(section) < 80:
                loc += f" | §{section}"
            chunk_tag = f"  [{'TABLE' if is_tbl else 'excerpt'} — {loc} | relevance {score}]"
            chunk_parts.append(f"{chunk_tag}\n  {chunk['text']}")

        if multi_doc:
            # Strong document-level header makes source attribution unambiguous
            border = "═" * 60
            doc_header = (
                f"\n{border}\n"
                f"  DOCUMENT: {source}\n"
                f"{border}"
            )
            doc_blocks.append(doc_header + "\n\n" + "\n\n".join(chunk_parts))
        else:
            # Single doc: no need for the heavy header
            doc_blocks.append("\n\n".join(chunk_parts))

    return "\n\n".join(doc_blocks)


# ══════════════════════════════════════════════════════════════
#  FEEDBACK STORE
# ══════════════════════════════════════════════════════════════

FEEDBACK_FILE = BASE_DIR / "feedback" / "retrieval_feedback.json"
FEEDBACK_FILE.parent.mkdir(exist_ok=True)


def record_feedback(query: str, chunk_ids: list[str], helpful: bool):
    try:
        existing = []
        if FEEDBACK_FILE.exists():
            with open(FEEDBACK_FILE) as f:
                existing = json.load(f)
        existing.append({
            "query":     query,
            "chunk_ids": chunk_ids,
            "helpful":   helpful,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })
        existing = existing[-500:]
        with open(FEEDBACK_FILE, "w") as f:
            json.dump(existing, f, indent=2)
    except Exception as e:
        print(f"[Feedback] Warning: {e}")


def _get_boosted_ids() -> set[str]:
    if not FEEDBACK_FILE.exists():
        return set()
    try:
        with open(FEEDBACK_FILE) as f:
            data = json.load(f)
        boosted = set()
        for entry in data:
            if entry.get("helpful"):
                boosted.update(entry.get("chunk_ids", []))
        return boosted
    except Exception:
        return set()


# ══════════════════════════════════════════════════════════════
#  MAIN RETRIEVAL FUNCTION
# ══════════════════════════════════════════════════════════════

def _get_doc_count(file_id: Optional[str], metadata: list[dict]) -> int:
    """Count distinct documents in the current scope."""
    ids = set(
        m.get("file_id") for m in metadata
        if m.get("type") in ("text", "image") and (file_id is None or m.get("file_id") == file_id)
    )
    return len(ids)


def _ensure_per_doc_coverage(
    chunks: list[dict],
    file_id: Optional[str],
    metadata: list[dict],
    min_per_doc: int = 2,
    query: str = "",
) -> list[dict]:
    """
    When querying across all docs (file_id=None), guarantees every document
    gets at least `min_per_doc` chunks in the final context — prevents any
    single large document from crowding out smaller ones.

    Works by:
    1. Taking the ranked result list as-is
    2. Checking which docs are under-represented
    3. Fetching the best-scoring chunks from those docs and inserting them
    """
    if file_id is not None:
        return chunks  # single-doc mode: no need

    # Count how many chunks each doc already has in the result
    covered: dict[str, list] = {}
    for c in chunks:
        fid = c.get("metadata", {}).get("file_id", "")
        covered.setdefault(fid, []).append(c)

    # Find all doc IDs in scope
    all_fids = set(
        m.get("file_id") for m in metadata
        if m.get("type") in ("text", "image")
    )

    # For any doc that has fewer than min_per_doc chunks, fetch its best chunks
    additions = []
    existing_ids = {c["id"] for c in chunks}

    for fid in all_fids:
        current_count = len(covered.get(fid, []))
        if current_count >= min_per_doc:
            continue

        # Get all text chunks for this doc, score them with BM25 against the query
        doc_metas = [m for m in metadata if m.get("file_id") == fid and m.get("type") in ("text", "image")]
        if not doc_metas:
            continue

        # Simple term-overlap score as lightweight relevance estimate
        q_terms = set(re.findall(r'\b\w{3,}\b', query.lower()))
        scored = []
        for m in doc_metas:
            if m["id"] in existing_ids:
                continue
            text_terms = set(re.findall(r'\b\w{3,}\b', m["text"].lower()))
            overlap = len(q_terms & text_terms) / max(len(q_terms), 1) if q_terms else 0
            scored.append((overlap, m))

        scored.sort(key=lambda x: x[0], reverse=True)
        needed = min_per_doc - current_count
        for score_val, m in scored[:needed]:
            additions.append({
                "id":           m["id"],
                "text":         m["text"],
                "metadata":     m,
                "fusion_score": 0.0,
                "bm25_score":   0.0,
                "vector_score": 0.0,
                "feedback_boost": 0.0,
                "rerank_score": score_val * 0.3,  # lower than retrieved chunks
            })
            existing_ids.add(m["id"])

    return chunks + additions


def get_relevant_context(
    query: str,
    file_id: Optional[str] = None,
    n_results: int = 7,
    use_multihop: bool = True,
) -> tuple[str, dict]:
    """
    Hybrid BM25 + FAISS retrieval with caching for speed.
    BM25 and sub-index are rebuilt only when store changes (new doc ingested).

    When file_id=None (All docs mode):
      - n_results scales with document count (min 3 per doc)
      - Per-doc coverage is guaranteed so no doc gets ignored
      - Context word budget is raised proportionally
    """
    index, metadata = _get_store()

    text_entry_indices = [
        i for i, m in enumerate(metadata)
        if m.get("type") in ("text", "image") and (file_id is None or m.get("file_id") == file_id)
    ]

    if not text_entry_indices:
        return "", {}

    # ── Scale n_results for multi-doc mode ───────────────────
    doc_count = _get_doc_count(file_id, metadata)
    if file_id is None and doc_count > 1:
        # At least 3 chunks per document, cap at 30 total to stay within LLM context
        n_results = min(max(n_results, doc_count * 3), 30)

    # ── Temporal query parsing ───────────────────────────────
    temporal = {}
    if _memory_available:
        try:
            temporal = parse_temporal_query(query)
        except Exception:
            pass

    # ── Knowledge graph traversal ────────────────────────────
    graph_context   = ""
    graph_chunk_ids = []
    if _memory_available and use_multihop:
        try:
            graph_result    = traverse_graph(query, max_hops=2, max_nodes=8)
            graph_context   = graph_result.get("graph_context", "")
            graph_chunk_ids = graph_result.get("chunk_ids", [])
        except Exception:
            pass

    # ── Expand queries (rule-based, zero latency) ───────────
    expanded_queries = _expand_query(query)
    if use_multihop:
        sub_queries  = _decompose_query(query)
        all_queries  = list(dict.fromkeys(expanded_queries + sub_queries))
    else:
        all_queries  = expanded_queries

    # ── BM25 (cached) ───────────────────────────────────────
    bm25, ids, metas, texts = _get_bm25(file_id, index, metadata)
    id_to_meta = {m["id"]: m for m in metas}

    bm25_rankings = []
    n_bm25 = min(n_results * 3, len(texts))
    for q in all_queries:
        top = bm25.get_top_n(q, n_bm25)
        max_score = max(s for _, s in top) if top else 1.0
        bm25_rankings.append([
            (ids[idx], score / max(max_score, 1e-9))
            for idx, score in top
        ])

    # ── FAISS vector search (cached sub-index) ───────────────
    sub_index, sub_ids, sub_metas, sub_texts = _get_sub_index(file_id, index, metadata, text_entry_indices)

    vector_rankings = []
    n_vec = min(n_results * 3, len(sub_ids))
    for q in all_queries:
        q_emb = _embed_query(q).reshape(1, -1)
        scores, idx_arr = sub_index.search(q_emb, n_vec)
        vector_rankings.append([
            (sub_ids[idx_arr[0][j]], float(scores[0][j]))
            for j in range(len(idx_arr[0]))
            if idx_arr[0][j] >= 0
        ])

    # ── RRF Fusion ──────────────────────────────────────────
    fused = _reciprocal_rank_fusion(bm25_rankings + vector_rankings)

    # ── Score maps ──────────────────────────────────────────
    bm25_score_map: dict[str, float] = {}
    for ranking in bm25_rankings:
        for doc_id, score in ranking:
            bm25_score_map[doc_id] = max(bm25_score_map.get(doc_id, 0), score)

    vec_score_map: dict[str, float] = {}
    for ranking in vector_rankings:
        for doc_id, score in ranking:
            vec_score_map[doc_id] = max(vec_score_map.get(doc_id, 0), score)

    boosted_ids = _get_boosted_ids()

    top_n = min(n_results * 3, len(fused))
    candidate_chunks = []
    for doc_id, fusion_score in fused[:top_n]:
        m = id_to_meta.get(doc_id)
        if m is None:
            # might be in sub_metas
            m = next((sm for sm in sub_metas if sm["id"] == doc_id), None)
        if m is None:
            continue
        candidate_chunks.append({
            "id":             doc_id,
            "text":           m["text"],
            "metadata":       m,
            "fusion_score":   fusion_score,
            "bm25_score":     bm25_score_map.get(doc_id, 0),
            "vector_score":   vec_score_map.get(doc_id, 0),
            "feedback_boost": 0.05 if doc_id in boosted_ids else 0.0,
        })

    # ── Author query: guarantee frontmatter chunks enter candidate pool ──
    if _is_author_query(query):
        existing_ids = {c["id"] for c in candidate_chunks}
        for m in metadata:
            if (m.get("is_frontmatter") and m.get("type") == "text"
                    and (file_id is None or m.get("file_id") == file_id)
                    and m["id"] not in existing_ids):
                candidate_chunks.append({
                    "id":             m["id"],
                    "text":           m["text"],
                    "metadata":       m,
                    "fusion_score":   0.4,
                    "bm25_score":     0.4,
                    "vector_score":   0.4,
                    "feedback_boost": 0.0,
                })

    # ── Re-rank ─────────────────────────────────────────────
    reranked = _rerank_chunks(query, candidate_chunks)

    # ── VERBOSE RETRIEVAL LOG ────────────────────────────────
    print(f"\n[Retrieval] ══════════════════════════════════════════")
    print(f"[Retrieval] Query     : \"{query[:100]}\"")
    print(f"[Retrieval] Expanded  : {all_queries}")
    print(f"[Retrieval] Corpus    : {len(text_entry_indices)} entries (text+images)  |  BM25 top-{n_bm25}  |  FAISS top-{n_vec}")
    print(f"[Retrieval] Candidates after RRF fusion: {len(candidate_chunks)}")
    print(f"[Retrieval] ── Top candidates after rerank ──")
    for rank, c in enumerate(reranked[:min(10, len(reranked))]):
        m    = c.get("metadata", {})
        src  = m.get("filename", m.get("file_id","?"))[:25]
        pg   = m.get("page_num", "?")
        sec  = str(m.get("section",""))[:25]
        fm   = " [FM]" if m.get("is_frontmatter") else ""
        tbl  = " [TABLE]" if m.get("is_table") else ""
        sel  = " ◄ SELECTED" if rank < n_results else ""
        preview = c["text"][:90].replace("\n"," ")
        print(f"[Retrieval]  #{rank+1:>2} | score={c.get('rerank_score',0):.3f}"
              f" (bm25={c.get('bm25_score',0):.2f} vec={c.get('vector_score',0):.2f})"
              f" | {src} p{pg} | {sec}{fm}{tbl}{sel}")
        print(f"[Retrieval]       └─ \"{preview}...\"")
    print(f"[Retrieval] ══════════════════════════════════════════\n")

    # ── Temporal filter ──────────────────────────────────────
    if _memory_available and temporal.get("has_temporal"):
        try:
            reranked = apply_temporal_filter(reranked, temporal)
        except Exception:
            pass

    final_chunks = [c for c in reranked[:n_results] if c.get("rerank_score", 0) > 0.01]

    if not final_chunks:
        final_chunks = reranked[:n_results]

    # ── Guarantee per-doc coverage in All-docs mode ──────────
    final_chunks = _ensure_per_doc_coverage(
        chunks=final_chunks,
        file_id=file_id,
        metadata=metadata,
        min_per_doc=2,
        query=query,
    )

    # ── Log chunk access ─────────────────────────────────────
    if _memory_available and final_chunks:
        try:
            log_chunk_access([c["id"] for c in final_chunks], query=query)
        except Exception:
            pass

    # ── Assemble context (word budget scales with doc count) ─
    max_words = 2800 if (file_id is not None or doc_count <= 1) else min(2800 * doc_count, 8000)
    context = _assemble_context(final_chunks, max_words=max_words)

    if graph_context:
        context = graph_context + "\n\n---\n\n" + context

    if _memory_available and temporal.get("has_temporal"):
        try:
            temporal_note = build_temporal_context_note(temporal)
            context = temporal_note + context
        except Exception:
            pass

    return context, {
        "temporal":      temporal,
        "graph_context": graph_context,
        "chunk_ids":     [c["id"] for c in final_chunks],
    }


def get_all_documents_context(max_chunks_per_doc: int = 5) -> str:
    """
    Fallback when retrieval returns nothing but docs exist.
    Returns a balanced sample from EVERY uploaded document, grouped by doc
    with clear document-level headers for attribution.
    """
    _, metadata = _get_store()

    by_doc: dict[str, list] = {}
    for m in metadata:
        if m.get("type") not in ("text", "image"):
            continue
        fid = m.get("file_id", "")
        by_doc.setdefault(fid, []).append(m)

    doc_blocks = []
    multi_doc = len(by_doc) > 1

    for fid, entries in by_doc.items():
        sorted_entries = sorted(entries, key=lambda x: (x.get("page_num", 0), x.get("chunk_index", 0)))
        chunk_parts = []
        source_name = sorted_entries[0].get("filename", "Unknown") if sorted_entries else "Unknown"

        for m in sorted_entries[:max_chunks_per_doc]:
            page    = m.get("page_num", "?")
            section = m.get("section", "")
            is_tbl  = m.get("is_table", False)
            loc = f"Page {page}"
            if section and section != source_name and len(section) < 80:
                loc += f" | §{section}"
            chunk_tag = f"  [{'TABLE' if is_tbl else 'excerpt'} — {loc}]"
            chunk_parts.append(f"{chunk_tag}\n  {m['text']}")

        if not chunk_parts:
            continue

        if multi_doc:
            border = "═" * 60
            doc_header = f"\n{border}\n  DOCUMENT: {source_name}\n{border}"
            doc_blocks.append(doc_header + "\n\n" + "\n\n".join(chunk_parts))
        else:
            doc_blocks.append("\n\n".join(chunk_parts))

    return "\n\n".join(doc_blocks)


def get_full_document_context(file_id: str, max_chunks: int = 30) -> str:
    """Retrieve all chunks for a single document (for summarization or single-doc fallback)."""
    _, metadata = _get_store()
    entries = sorted(
        [m for m in metadata if m.get("file_id") == file_id and m.get("type") in ("text", "image")],
        key=lambda x: (x.get("page_num", 0), x.get("chunk_index", 0))
    )
    if not entries:
        return ""
    source_name = entries[0].get("filename", "Unknown")
    parts = []
    for m in entries[:max_chunks]:
        page    = m.get("page_num", "?")
        section = m.get("section", "")
        is_tbl  = m.get("is_table", False)
        loc = f"Page {page}"
        if section and section != source_name and len(section) < 80:
            loc += f" | §{section}"
        chunk_tag = f"  [{'TABLE' if is_tbl else 'excerpt'} — {loc}]"
        parts.append(f"{chunk_tag}\n  {m['text']}")
    return "\n\n".join(parts)


# ══════════════════════════════════════════════════════════════
#  DOCUMENT MANAGEMENT
# ══════════════════════════════════════════════════════════════

def list_all_documents() -> list[dict]:
    _, metadata = _get_store()
    seen: dict[str, dict] = {}
    for m in metadata:
        fid = m.get("file_id")
        if fid and fid not in seen and m.get("type") in ("text", "image"):
            seen[fid] = {
                "file_id":      fid,
                "filename":     m.get("filename", "Unknown"),
                "total_chunks": m.get("total_chunks", 0),
                "version":      m.get("version", 1),
                "ingested_at":  m.get("ingested_at", ""),
            }
    return sorted(seen.values(), key=lambda x: x.get("ingested_at", ""), reverse=True)


def delete_document(file_id: str) -> None:
    global _faiss_index, _metadata_store, _store_mtime
    index, metadata = _get_store()

    keep_indices = [i for i, m in enumerate(metadata) if m.get("file_id") != file_id]
    removed = len(metadata) - len(keep_indices)
    if removed == 0:
        return

    new_index = faiss.IndexFlatIP(EMBED_DIM)
    new_metadata = [metadata[i] for i in keep_indices]

    # Re-embed kept entries to rebuild index
    if new_metadata:
        model = _get_embed_model()
        texts = [m["text"] for m in new_metadata]
        vecs  = model.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
        new_index.add(vecs.astype("float32"))

    _faiss_index    = new_index
    _metadata_store = new_metadata

    faiss.write_index(new_index, str(FAISS_INDEX_FILE))
    with open(METADATA_FILE, "w") as f:
        json.dump(new_metadata, f)
    _store_mtime = METADATA_FILE.stat().st_mtime

    # Invalidate caches
    _bm25_cache.clear()
    _sub_index_cache.clear()

    meta_path = META_DIR / f"{file_id}.json"
    if meta_path.exists():
        meta_path.unlink()

    print(f"[VectorStore] Deleted {removed} entries for {file_id}")

Overwriting /content/singularity/retrieval.py


### 5c — llm.py
*LLM module — AI-powered query rewriting + streaming chat*

In [27]:
%%writefile /content/singularity/llm.py
"""
Enterprise LLM Module  v4.0
============================
Speed improvements over v3:
  - Query rewriting is now OPTIONAL and skipped by default for speed
    (the retrieval module's rule-based expansion is sufficient for most queries)
  - History compression threshold lowered (compress after >20 turns, not 14)
  - Reduced num_ctx for rewrite calls (512 instead of 1024)
  - rewrite_query() returns quickly on timeout (0.8s max) with fallback
  - stream_chat() context window kept at 8192 for quality
"""

import re
import json
from typing import Generator, Optional
from pathlib import Path
from datetime import datetime, timezone

import ollama

BASE_DIR   = Path(__file__).parent
MEMORY_DIR = BASE_DIR / "feedback"
MEMORY_DIR.mkdir(exist_ok=True)

LLM_MODEL = "llama3.2:3b"

# ── System Prompts ─────────────────────────────────────────────

CHAT_SYSTEM_PROMPT = """You are an expert AI research assistant. You answer questions ONLY from the document context provided below.

══════════════════════════════════════════
TABLE READING — MANDATORY RULES
══════════════════════════════════════════
There are TWO types of table content in the context:

  TYPE A — TOC/Index rows (DO NOT USE FOR DATA):
    Look like: | Table 4.1 | Preprocessing Results Summary | 38 |
    Only 2-3 columns: table name, brief description, page number.
    These are a table of contents — NOT the actual table data.

  TYPE B — Real data chunks (USE THIS):
    Labelled: [TABLE: Table 4.1: Preprocessing Results Summary]
    Have 4+ columns with actual measurements, e.g.:
    | Component | Input | Output | Transformation |
    | Raw variants (ClinVar + GWAS) | 58,916 variants | 58,916 variants | Quality filtering |

When a user asks "show table 4.1" or "explain Table II":
  STEP 1 — Find the [TABLE: Table X.Y: ...] chunk where X.Y EXACTLY matches.
           SKIP any index/TOC rows that merely list "Table 4.1" as a name and page.
  STEP 2 — Read the ENTIRE pipe-table. Do not skip rows or columns.
  STEP 3 — Output ALL rows and columns with the EXACT verbatim values from the chunk.
           Format as a clean markdown table.
  STEP 4 — Never fabricate, estimate, or paraphrase cell values.

FORBIDDEN:
  ✗ Using a TOC row ("| Table 4.1 | ... | 38 |") as the answer — that is just a listing
  ✗ Making up values not verbatim in the [TABLE: ...] chunk
  ✗ Showing a different table number than what was asked
  ✗ Saying "Table 4.1 is on page 38" without showing the actual data rows

══════════════════════════════════════════
AUTHOR / METADATA RULES
══════════════════════════════════════════
For "who are the authors", "list the authors", "who wrote this":
  STEP 1 — Scan ALL chunks for lines containing person names, email addresses (@),
           institution names (University, College, Department, Institute).
  STEP 2 — List every name found with their affiliation and email if present.
  STEP 3 — Do NOT say authors are not listed if you can see any names or emails.

Author names often look like: "FirstName LastName\nDepartment of X\nUniversity of Y\nemail@domain"

══════════════════════════════════════════
IMAGE / FIGURE READING RULES
══════════════════════════════════════════
Chunks labelled [IMAGE p.X (Fig Y.Z)] contain AI-generated descriptions of actual diagrams, charts, and figures from the document.
These are REAL visual descriptions — use them to answer questions about figures.

When a user asks about a figure (e.g. "explain Figure 3.1", "what does Fig 4.2 show"):
  STEP 1 — Find the [IMAGE ...] chunk whose label matches (Fig 3.1, Fig 4.2, etc.)
  STEP 2 — Read the FULL caption description in that chunk.
  STEP 3 — Report what the figure shows based on the description. Be specific.
  STEP 4 — Supplement with any matching text chunks from the same section.

FORBIDDEN:
  ✗ "I do not have access to the actual image" — you DO have the caption description, use it
  ✗ Saying a figure is absent when an [IMAGE (Fig X.Y)] chunk exists in context
  ✗ Mixing up Fig 3.1 and Fig 3.2 — match the number exactly

══════════════════════════════════════════
CORE RULES
══════════════════════════════════════════
1. Answer ONLY from the DOCUMENT CONTEXT block. Read it COMPLETELY before responding.
2. State facts directly and confidently. Never say "it appears" or "it seems" when the data is there.
3. SELF-CHECK: Before saying something is absent, re-read every chunk once more.
4. Never fabricate. Never use external knowledge. If truly not found, say so briefly.

SOURCE ATTRIBUTION:
- Every factual claim: cite as (filename, page N) or (filename, Table X)
- Multi-document: label each fact by document

FORMAT:
- Use plain prose for normal answers. Do NOT use markdown tables unless the user explicitly asks for a table or you are reproducing data that was literally a table in the source document (i.e. the chunk was labelled [TABLE: ...]).
- Never reformat a simple sentence, list, or pair of values as a table — just write it out in plain text.
- Bold key findings sparingly.
- "Sources:" line at the end listing page numbers and sections in plain text.
  Write: (Filename, Page N, §Section) — NOT raw labels like "[excerpt — Page X | relevance 0.5]" or "[DIRECT-SCAN]".

FORBIDDEN in output:
  ✗ "[DIRECT-SCAN]", "[excerpt — ...]", "relevance 0.xxx" — internal labels, never show to user
  ✗ Markdown tables for data that is naturally a prose list or bullet list
  ✗ Numbering an organizational note as if it continues a numbered list
  ✗ Fabricating any table cell values not explicitly present in the [TABLE: ...] chunk
"""

SUMMARIZE_SYSTEM_PROMPT = """You are an expert document analyst. Create a comprehensive, structured summary.

FORMAT:
## 📋 Overview
[2-3 sentence executive summary]

## 🎯 Key Topics
[Bullet list of main subjects]

## 💡 Key Findings & Arguments
[Most important points with supporting evidence]

## 📊 Notable Data & Facts
[Statistics, dates, measurements, table data — exact when available]

## 🔗 Connections & Themes
[Cross-cutting themes or relationships]

## ✅ Conclusions
[Final takeaways, recommendations, or implications]

Be thorough. Preserve all technical details and table values. Cite page/section references when possible."""

QUERY_REWRITE_PROMPT = """You are a search query optimizer for document retrieval. Rewrite the user query to improve retrieval.

STRICT RULES:
- Output ONLY the rewritten query. No explanation, no quotes, no preamble.
- Keep the rewrite TIGHTLY focused on what the user asked — do NOT expand to unrelated topics.
- If the query names a specific project, tool, table, or entity, that name MUST appear verbatim.
- Add synonyms only for the specific topic asked — never add names of other projects not in the query.
- Keep it under 25 words.
- DO NOT invent table names, section names, or figure numbers not in the original query.

Examples:
  Input:  show table 4.1
  Output: Table 4.1 Preprocessing Results Summary data rows columns

  Input:  what was done in malwiki
  Output: MalWiki contributions work deliverables features built

  Input:  what are the key findings
  Output: key findings results conclusions summary

  Input:  explain table IV
  Output: Table IV data rows columns values complete

  Input:  who wrote this
  Output: authors names affiliations institutions email

  Input:  what is the plagiarism percentage
  Output: overall similarity percentage plagiarism originality score

  Input:  explain figure 3.1
  Output: Figure 3.1 architecture diagram description

- NEVER add names of other projects, platforms, or systems the user did NOT mention."""

MEMORY_COMPRESS_PROMPT = """Compress this conversation history into a concise summary (max 200 words).
Preserve: key facts, user's main questions, important findings, named entities, conclusions reached.
Output only the compressed summary, no preamble."""


# ══════════════════════════════════════════════════════════════
#  QUERY REWRITING  (fast, with timeout fallback)
# ══════════════════════════════════════════════════════════════

def rewrite_query(query: str, conversation_context: str = "") -> str:
    """
    Rewrites query for better retrieval.
    Hard 3-second timeout via thread — if Ollama is slow, returns original query immediately.
    Set use_query_rewrite=False in the frontend to skip entirely for max speed.
    """
    import threading
    result_holder = [query]  # fallback = original query

    # Table queries and author queries must NOT get context injected —
    # injecting prior conversation makes the rewriter hallucinate wrong table names
    _NO_CONTEXT_PATTERNS = re.compile(
        r'\b(table|tbl|tab)\.?\s*(\d+|[ivxlc]+)\b|'
        r'\b(fig(?:ure)?)\.?\s*\d+[\d\.]*\b|'
        r'\b(author|authors|who wrote|affiliation|name|names|wrote)\b',
        re.IGNORECASE
    )
    skip_context = bool(_NO_CONTEXT_PATTERNS.search(query))

    def _call():
        try:
            context_hint = ""
            if not skip_context and conversation_context:
                context_hint = f"\n\nConversation context (use to resolve pronouns only):\n{conversation_context[-200:]}"

            messages = [
                {"role": "system", "content": QUERY_REWRITE_PROMPT},
                {"role": "user", "content": f"Query: {query}{context_hint}"}
            ]
            response = ollama.chat(
                model=LLM_MODEL,
                messages=messages,
                stream=False,
                options={
                    "temperature": 0.05,
                    "num_ctx": 512,
                    "num_predict": 45,
                }
            )
            rewritten = response["message"]["content"].strip()
            # Strip echo prefixes the model sometimes adds
            for prefix in ("Query:", "Rewritten:", "Output:", "Rewritten query:", "Original query:"):
                if rewritten.lower().startswith(prefix.lower()):
                    rewritten = rewritten[len(prefix):].strip()
            rewritten = rewritten.strip('"\'`')
            # Only use rewrite if genuinely different and a reasonable length
            if 2 <= len(rewritten.split()) <= 45 and rewritten.lower() != query.lower():
                result_holder[0] = rewritten
        except Exception:
            pass

    t = threading.Thread(target=_call, daemon=True)
    t.start()
    t.join(timeout=8.0)
    return result_holder[0]


# ══════════════════════════════════════════════════════════════
#  CONVERSATIONAL MEMORY — HISTORY ASSEMBLY
# ══════════════════════════════════════════════════════════════

def _compress_old_history(history: list[dict]) -> str:
    try:
        history_text = "\n".join(
            f"{h['role'].upper()}: {h['content'][:400]}"
            for h in history
        )
        response = ollama.chat(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": MEMORY_COMPRESS_PROMPT},
                {"role": "user", "content": history_text}
            ],
            stream=False,
            options={"temperature": 0.1, "num_ctx": 2048, "num_predict": 250}
        )
        return response["message"]["content"].strip()
    except Exception:
        return ""


def build_messages_from_history(
    history: list[dict],
    query: str,
    context: str,
    doc_manifest: list[str] | None = None,
) -> list[dict]:
    """
    Assembles message list for the LLM.

    doc_manifest: list of document filenames currently in scope.
    When multiple docs are loaded, we inject a manifest header so the LLM
    always knows exactly which documents it should draw from and attribute facts to.

    - Sessions ≤ 20 turns: full verbatim history
    - Sessions > 20 turns: compress oldest, keep 10 verbatim
    """
    messages = [{"role": "system", "content": CHAT_SYSTEM_PROMPT}]

    VERBATIM_KEEP  = 10
    COMPRESS_AFTER = 20

    if len(history) > COMPRESS_AFTER:
        older  = history[:-VERBATIM_KEEP]
        recent = history[-VERBATIM_KEEP:]
        compressed = _compress_old_history(older)
        if compressed:
            messages.append({
                "role": "user",
                "content": "[EARLIER CONVERSATION SUMMARY]: " + compressed
            })
            messages.append({
                "role": "assistant",
                "content": "Understood. I have context from our earlier conversation."
            })
    else:
        recent = history

    for turn in recent:
        role    = turn.get("role", "user")
        content = turn.get("content", "")
        if role in ("user", "assistant") and content:
            messages.append({"role": role, "content": content})

    if context:
        # Build manifest header so LLM knows all docs in scope
        if doc_manifest and len(doc_manifest) > 1:
            manifest_lines = "\n".join(f"  {i+1}. {name}" for i, name in enumerate(doc_manifest))
            manifest_block = (
                f"LOADED DOCUMENTS ({len(doc_manifest)} total):\n{manifest_lines}\n\n"
                f"IMPORTANT: Attribute every fact to its specific document by name. "
                f"When comparing, address each document separately before synthesising.\n\n"
            )
        else:
            manifest_block = ""

        user_content = (
            f"{manifest_block}"
            f"DOCUMENT CONTEXT (retrieved passages — each section is clearly labelled with its source document):\n"
            f"{'='*60}\n{context}\n{'='*60}\n\n"
            f"NOTE: You also have access to the full conversation history above. "
            f"Use it to resolve follow-up references and pronouns like 'it', 'that', 'the previous answer'.\n\n"
            f"QUESTION: {query}"
        )
    else:
        user_content = (
            f"QUESTION: {query}\n\n"
            f"⚠️ No document context available. Please upload a document first."
        )

    messages.append({"role": "user", "content": user_content})
    return messages


# ══════════════════════════════════════════════════════════════
#  STREAMING CHAT
# ══════════════════════════════════════════════════════════════

def stream_chat(
    query: str,
    context: str,
    history: list[dict],
    rewritten_query: Optional[str] = None,
    doc_manifest: list[str] | None = None,
) -> Generator[str, None, None]:
    """
    Streams a grounded RAG answer with full conversation history.
    doc_manifest: list of filenames in scope — injected into the prompt so
    the LLM knows all documents it should attribute and compare against.
    """
    messages = build_messages_from_history(
        history=history,
        query=query,
        context=context,
        doc_manifest=doc_manifest,
    )

    # Scale context window: multi-doc needs more room
    num_ctx = 12288 if (doc_manifest and len(doc_manifest) > 1) else 8192

    stream = ollama.chat(
        model=LLM_MODEL,
        messages=messages,
        stream=True,
        options={
            "temperature": 0.1,
            "num_ctx": num_ctx,
            "top_p": 0.9,
            "repeat_penalty": 1.1,
            "num_predict": 1500,  # allow longer answers for multi-doc comparisons
        }
    )

    for chunk in stream:
        token = chunk["message"]["content"]
        if token:
            yield token


# ══════════════════════════════════════════════════════════════
#  CONVERSATIONAL REPLY  (memory/recall queries — no doc retrieval)
# ══════════════════════════════════════════════════════════════

CONVERSATIONAL_SYSTEM_PROMPT = """You are an expert AI research assistant with perfect memory of the current conversation.

The user is asking about what was discussed in this session — NOT asking you to search documents.

RULES:
1. Answer ENTIRELY from the conversation history provided. Do not reference any documents unless the user explicitly asked about them in this session.
2. If asked "what were we talking about", summarise the conversation topics accurately and concisely.
3. If asked about a specific thing said earlier (e.g. "what did you say about X"), quote or paraphrase your earlier answer.
4. If the conversation history is empty or very short, say so honestly.
5. Be direct and natural. This is a memory/recall question, not a document lookup.
6. Never say you can't remember — you have the full conversation history right here.
"""


def stream_conversational(
    query: str,
    history: list[dict],
) -> Generator[str, None, None]:
    """
    Answers questions that are about the conversation itself (recall, summary,
    "what did we talk about") using only conversation history — no document context.
    This is fast because it skips retrieval entirely.
    """
    if not history:
        yield "We haven't discussed anything yet in this session. Feel free to ask me about your uploaded documents!"
        return

    # Build a clean readable transcript of the conversation
    transcript_parts = []
    for i, turn in enumerate(history[-30:]):  # last 30 turns max
        role    = turn.get("role", "")
        content = turn.get("content", "")
        if not content:
            continue
        label = "User" if role == "user" else "Assistant"
        # Truncate very long assistant turns to avoid blowing context
        if role == "assistant" and len(content) > 600:
            content = content[:600] + "... [truncated]"
        transcript_parts.append(f"{label}: {content}")

    transcript = "\n\n".join(transcript_parts)

    messages = [
        {"role": "system", "content": CONVERSATIONAL_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"CONVERSATION HISTORY (this session so far):\n"
                f"{'='*60}\n{transcript}\n{'='*60}\n\n"
                f"USER'S CURRENT QUESTION: {query}"
            )
        }
    ]

    stream = ollama.chat(
        model=LLM_MODEL,
        messages=messages,
        stream=True,
        options={
            "temperature": 0.2,
            "num_ctx": 4096,
            "top_p": 0.9,
            "num_predict": 512,  # conversational answers should be concise
        }
    )

    for chunk in stream:
        token = chunk["message"]["content"]
        if token:
            yield token


# ══════════════════════════════════════════════════════════════
#  STREAMING SUMMARIZATION
# ══════════════════════════════════════════════════════════════

def stream_summary(context: str, filename: str = "") -> Generator[str, None, None]:
    doc_note = f" (Document: {filename})" if filename else ""

    messages = [
        {"role": "system", "content": SUMMARIZE_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Please provide a comprehensive summary of this document{doc_note}.\n\n"
                f"DOCUMENT CONTENT:\n{'='*60}\n{context}\n{'='*60}"
            )
        },
    ]

    stream = ollama.chat(
        model=LLM_MODEL,
        messages=messages,
        stream=True,
        options={
            "temperature": 0.2,
            "num_ctx": 4096,
            "top_p": 0.95,
            "num_predict": 1500,
        }
    )

    for chunk in stream:
        token = chunk["message"]["content"]
        if token:
            yield token


# ══════════════════════════════════════════════════════════════
#  USER PREFERENCE / DOMAIN LEARNING
# ══════════════════════════════════════════════════════════════

USER_PREFS_FILE = MEMORY_DIR / "user_preferences.json"


def update_user_preferences(query: str, file_id: Optional[str] = None):
    """Infer and persist user domain interests from queries."""
    try:
        prefs = {}
        if USER_PREFS_FILE.exists():
            with open(USER_PREFS_FILE) as f:
                prefs = json.load(f)

        queries = prefs.get("query_history", [])
        queries.append({
            "query":     query[:200],
            "file_id":   file_id,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })
        prefs["query_history"] = queries[-100:]

        all_words = " ".join(q["query"] for q in prefs["query_history"])
        domain_words = re.findall(r'\b[A-Za-z]{4,}\b', all_words.lower())
        stop_words = {"what", "which", "when", "where", "that", "this", "from", "with",
                      "have", "about", "does", "explain", "tell", "show", "list", "find"}
        domain_words = [w for w in domain_words if w not in stop_words]

        from collections import Counter
        prefs["domain_interests"] = dict(Counter(domain_words).most_common(20))

        with open(USER_PREFS_FILE, "w") as f:
            json.dump(prefs, f, indent=2)
    except Exception:
        pass


def get_user_preferences() -> dict:
    try:
        if USER_PREFS_FILE.exists():
            with open(USER_PREFS_FILE) as f:
                return json.load(f)
    except Exception:
        pass
    return {}

Overwriting /content/singularity/llm.py


### 5d — memory.py
*Memory — Reflection layer + temporal memory + decay/pruning*

In [28]:
%%writefile /content/singularity/memory.py
"""
Reflection Layer (Meta-Memory) + Temporal Memory + Decay/Pruning
=================================================================
Implements spec items:
  - Reflection Layer (9): self-evaluation of answer quality, confidence scoring,
    gap detection, intent trajectory tracking
  - Temporal Memory (11): timestamp-aware retrieval, date-range filtering,
    "what was true in 2022?" queries, recency preference
  - Forgetting/Decay Layer (12): decay functions, relevance pruning,
    deduplication over time, ChromaDB cleanup

REFLECTION LAYER
────────────────
After every answer is generated, the reflection system:
  1. Scores confidence (0-1) based on context coverage
  2. Detects gaps — questions it couldn't answer from context
  3. Flags hallucination risk (answer references things not in context)
  4. Tracks intent trajectory across the session
  5. Logs reflection records for meta-learning

TEMPORAL MEMORY LAYER
─────────────────────
  - Every chunk already has ingested_at timestamp
  - Temporal query parser detects date references in queries:
    "what was true in 2022", "before the merger", "latest findings"
  - Filters ChromaDB results to matching time windows
  - Recency scoring: recent chunks boosted unless query specifies past

FORGETTING / DECAY LAYER
─────────────────────────
  - Chunk access log tracks which chunks are retrieved and when
  - Decay function: score = base_score * e^(-λ * days_since_access)
  - Pruning: chunks with decay_score < threshold AND access_count < 2
    are candidates for removal from ChromaDB
  - Deduplication: periodic scan removes near-duplicate chunks
    (Jaccard > 0.9) keeping only the most-accessed one
"""

import re
import json
import math
import time
from pathlib import Path
from typing import Optional
from datetime import datetime, timezone, timedelta
from collections import defaultdict

import ollama

BASE_DIR    = Path(__file__).parent
MEMORY_DIR  = BASE_DIR / "feedback"
MEMORY_DIR.mkdir(exist_ok=True)

REFLECTION_LOG   = MEMORY_DIR / "reflection_log.json"
ACCESS_LOG       = MEMORY_DIR / "chunk_access_log.json"
INTENT_LOG       = MEMORY_DIR / "intent_trajectory.json"

LLM_MODEL = "llama3.2:3b"
DECAY_LAMBDA = 0.01   # decay rate: half-life ≈ 69 days
PRUNE_THRESHOLD = 0.05  # decay score below this = candidate for pruning
MIN_ACCESS_TO_PRUNE = 2  # only prune chunks accessed fewer than this many times


# ══════════════════════════════════════════════════════════════
#  REFLECTION LAYER
# ══════════════════════════════════════════════════════════════

REFLECTION_PROMPT = """You are a quality evaluator for an AI RAG system.

Given a QUESTION, the CONTEXT retrieved, and the ANSWER generated, evaluate:

1. confidence_score: 0.0-1.0 — how well the context supports the answer
   (1.0 = context fully answers the question, 0.0 = answer is unsupported)

2. coverage_gaps: list of sub-questions the context could NOT answer

3. hallucination_risk: "low" | "medium" | "high"
   - low: answer only uses information from context
   - medium: answer extends slightly beyond context
   - high: answer makes claims not present in context

4. answer_quality: "excellent" | "good" | "partial" | "poor"

5. missing_context: what additional information would improve the answer

Output ONLY valid JSON (no markdown fences):
{
  "confidence_score": 0.85,
  "coverage_gaps": ["gap 1", "gap 2"],
  "hallucination_risk": "low",
  "answer_quality": "good",
  "missing_context": "description of what's missing"
}"""


def reflect_on_answer(
    query: str,
    context: str,
    answer: str,
    session_id: Optional[str] = None,
) -> dict:
    """
    Runs reflection on a generated answer.
    Returns reflection metadata and logs it.
    """
    reflection = {
        "confidence_score": 0.5,
        "coverage_gaps": [],
        "hallucination_risk": "unknown",
        "answer_quality": "unknown",
        "missing_context": "",
    }

    try:
        # Truncate for reflection call
        ctx_sample = context[:1500] if context else "(no context)"
        ans_sample = answer[:800]

        response = ollama.chat(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": REFLECTION_PROMPT},
                {"role": "user", "content": (
                    f"QUESTION: {query}\n\n"
                    f"CONTEXT (excerpt):\n{ctx_sample}\n\n"
                    f"ANSWER:\n{ans_sample}"
                )}
            ],
            stream=False,
            options={"temperature": 0.0, "num_ctx": 2048}
        )
        raw = response["message"]["content"].strip()
        raw = re.sub(r"^```json\s*|```\s*$", "", raw, flags=re.MULTILINE).strip()
        parsed = json.loads(raw)
        reflection.update(parsed)
    except Exception:
        # Fallback: heuristic confidence from context length
        if not context:
            reflection["confidence_score"] = 0.0
            reflection["hallucination_risk"] = "high"
        elif len(context) > 500:
            reflection["confidence_score"] = 0.7
            reflection["hallucination_risk"] = "low"

    # Log the reflection
    _log_reflection(query, reflection, session_id)

    return reflection


def _log_reflection(query: str, reflection: dict, session_id: Optional[str]):
    """Persist reflection record."""
    try:
        records = []
        if REFLECTION_LOG.exists():
            with open(REFLECTION_LOG) as f:
                records = json.load(f)

        records.append({
            "timestamp":   datetime.now(timezone.utc).isoformat(),
            "session_id":  session_id,
            "query":       query[:200],
            "reflection":  reflection,
        })
        # Keep last 1000 reflections
        records = records[-1000:]
        with open(REFLECTION_LOG, "w") as f:
            json.dump(records, f, indent=2)
    except Exception:
        pass


def get_reflection_stats() -> dict:
    """Aggregate reflection metrics for telemetry."""
    if not REFLECTION_LOG.exists():
        return {}
    try:
        with open(REFLECTION_LOG) as f:
            records = json.load(f)
        if not records:
            return {}

        scores = [r["reflection"].get("confidence_score", 0) for r in records]
        risks  = [r["reflection"].get("hallucination_risk", "unknown") for r in records]
        quals  = [r["reflection"].get("answer_quality", "unknown") for r in records]

        from collections import Counter
        return {
            "total_reflections":    len(records),
            "avg_confidence":       round(sum(scores) / len(scores), 3),
            "hallucination_dist":   dict(Counter(risks)),
            "quality_dist":         dict(Counter(quals)),
            "low_confidence_count": sum(1 for s in scores if s < 0.4),
        }
    except Exception:
        return {}


# ══════════════════════════════════════════════════════════════
#  INTENT TRAJECTORY TRACKER
# ══════════════════════════════════════════════════════════════

def update_intent_trajectory(
    query: str,
    answer_quality: str,
    session_id: Optional[str] = None,
):
    """
    Track the arc of what a user is trying to accomplish.
    Detects shifts: overview → deep-dive → specific fact → comparison
    """
    try:
        trajectories = {}
        if INTENT_LOG.exists():
            with open(INTENT_LOG) as f:
                trajectories = json.load(f)

        sid = session_id or "global"
        if sid not in trajectories:
            trajectories[sid] = []

        # Classify query intent
        intent = _classify_intent(query)

        trajectories[sid].append({
            "timestamp":     datetime.now(timezone.utc).isoformat(),
            "query":         query[:150],
            "intent":        intent,
            "answer_quality": answer_quality,
        })

        # Keep last 50 turns per session
        trajectories[sid] = trajectories[sid][-50:]

        with open(INTENT_LOG, "w") as f:
            json.dump(trajectories, f, indent=2)
    except Exception:
        pass


def _classify_intent(query: str) -> str:
    """Classify query intent type without LLM call."""
    q = query.lower()
    if any(w in q for w in ["summarize", "overview", "what is", "explain", "describe"]):
        return "overview"
    if any(w in q for w in ["how", "why", "mechanism", "process", "method"]):
        return "deep_dive"
    if any(w in q for w in ["number", "metric", "percent", "value", "how many", "how much", "when"]):
        return "specific_fact"
    if any(w in q for w in ["compare", "difference", "versus", "vs", "better", "worse"]):
        return "comparison"
    if any(w in q for w in ["list", "all", "every", "enumerate"]):
        return "enumeration"
    if any(w in q for w in ["who", "which company", "which person", "founder", "ceo"]):
        return "entity_lookup"
    return "general"


def get_intent_trajectory(session_id: str) -> list[dict]:
    """Return the intent trajectory for a session."""
    if not INTENT_LOG.exists():
        return []
    try:
        with open(INTENT_LOG) as f:
            trajectories = json.load(f)
        return trajectories.get(session_id, [])
    except Exception:
        return []


# ══════════════════════════════════════════════════════════════
#  TEMPORAL MEMORY LAYER
# ══════════════════════════════════════════════════════════════

# Temporal query patterns
_YEAR_PATTERN   = re.compile(r'\b(19\d{2}|20\d{2})\b')
_BEFORE_PATTERN = re.compile(r'\bbefore\s+(19\d{2}|20\d{2})\b', re.IGNORECASE)
_AFTER_PATTERN  = re.compile(r'\bafter\s+(19\d{2}|20\d{2})\b', re.IGNORECASE)
_IN_YEAR_PATTERN = re.compile(
    r'\bin\s+(19\d{2}|20\d{2})\b|'
    r'\bas of\s+(19\d{2}|20\d{2})\b|'
    r'\bwhat was true in\s+(19\d{2}|20\d{2})\b|'
    r'\bduring\s+(19\d{2}|20\d{2})\b',
    re.IGNORECASE
)
_RECENT_PATTERN = re.compile(
    r'\blatest\b|\brecent\b|\bnow\b|\bcurrent\b|\btoday\b|\bnewest\b',
    re.IGNORECASE
)
_OLDEST_PATTERN = re.compile(
    r'\boriginal\b|\binitial\b|\bearliest\b|\bfirst version\b|\bwhen it started\b',
    re.IGNORECASE
)


def parse_temporal_query(query: str) -> dict:
    """
    Parse temporal constraints from a natural language query.

    Returns:
        {
          "has_temporal": bool,
          "mode": "exact_year" | "before" | "after" | "recent" | "oldest" | None,
          "year": int | None,
          "before_year": int | None,
          "after_year": int | None,
          "prefer_recent": bool,
          "prefer_oldest": bool,
        }
    """
    result = {
        "has_temporal":  False,
        "mode":          None,
        "year":          None,
        "before_year":   None,
        "after_year":    None,
        "prefer_recent": False,
        "prefer_oldest": False,
    }

    # Check "in year" / "as of year" / "what was true in year"
    m = _IN_YEAR_PATTERN.search(query)
    if m:
        year_str = next(g for g in m.groups() if g)
        result.update({
            "has_temporal": True,
            "mode":         "exact_year",
            "year":         int(year_str),
        })
        return result

    # Check "before year"
    m = _BEFORE_PATTERN.search(query)
    if m:
        result.update({
            "has_temporal": True,
            "mode":         "before",
            "before_year":  int(m.group(1)),
        })
        return result

    # Check "after year"
    m = _AFTER_PATTERN.search(query)
    if m:
        result.update({
            "has_temporal": True,
            "mode":         "after",
            "after_year":   int(m.group(1)),
        })
        return result

    # Recent preference
    if _RECENT_PATTERN.search(query):
        result.update({
            "has_temporal":  True,
            "mode":          "recent",
            "prefer_recent": True,
        })
        return result

    # Oldest preference
    if _OLDEST_PATTERN.search(query):
        result.update({
            "has_temporal":  True,
            "mode":          "oldest",
            "prefer_oldest": True,
        })

    return result


def apply_temporal_filter(
    chunks: list[dict],
    temporal: dict,
) -> list[dict]:
    """
    Filter and re-score chunks based on temporal constraints.
    chunks: list of {"text", "metadata", "rerank_score", ...}
    """
    if not temporal.get("has_temporal"):
        return chunks

    mode = temporal.get("mode")
    now_year = datetime.now(timezone.utc).year

    filtered = []
    for chunk in chunks:
        meta = chunk.get("metadata", {})
        ingested_at = meta.get("ingested_at", "")

        # Try to extract year from ingested_at timestamp
        ingested_year = None
        if ingested_at:
            try:
                ingested_year = datetime.fromisoformat(ingested_at).year
            except Exception:
                pass

        # Also try to find year mentions in the chunk text itself
        text_years = [int(y) for y in _YEAR_PATTERN.findall(chunk.get("text", ""))]

        # Apply mode-specific filtering
        if mode == "exact_year":
            target_year = temporal["year"]
            # Include chunk if target year appears in text OR ingestion year matches
            year_in_text = target_year in text_years
            year_at_ingest = (ingested_year == target_year) if ingested_year else False
            if not (year_in_text or year_at_ingest):
                # Don't hard-exclude, just heavily penalize
                chunk = dict(chunk)
                chunk["rerank_score"] = chunk.get("rerank_score", 0) * 0.2
            else:
                chunk = dict(chunk)
                chunk["rerank_score"] = chunk.get("rerank_score", 0) * 1.5

        elif mode == "before" and temporal.get("before_year"):
            cutoff = temporal["before_year"]
            relevant_years = [y for y in text_years if y < cutoff]
            if not relevant_years:
                chunk = dict(chunk)
                chunk["rerank_score"] = chunk.get("rerank_score", 0) * 0.3

        elif mode == "after" and temporal.get("after_year"):
            cutoff = temporal["after_year"]
            relevant_years = [y for y in text_years if y > cutoff]
            if not relevant_years:
                chunk = dict(chunk)
                chunk["rerank_score"] = chunk.get("rerank_score", 0) * 0.3

        elif mode == "recent":
            # Boost most recently ingested
            if ingested_year:
                age = now_year - ingested_year
                boost = max(0, 1.0 - age * 0.1)  # 10% penalty per year old
                chunk = dict(chunk)
                chunk["rerank_score"] = chunk.get("rerank_score", 0) * (1 + boost)

        elif mode == "oldest":
            # Boost oldest ingested
            if ingested_year:
                age = now_year - ingested_year
                boost = min(1.0, age * 0.1)
                chunk = dict(chunk)
                chunk["rerank_score"] = chunk.get("rerank_score", 0) * (1 + boost)

        filtered.append(chunk)

    return sorted(filtered, key=lambda c: c.get("rerank_score", 0), reverse=True)


def build_temporal_context_note(temporal: dict) -> str:
    """Generate a note to inject into the LLM context about temporal constraints."""
    if not temporal.get("has_temporal"):
        return ""
    mode = temporal.get("mode")
    if mode == "exact_year":
        return f"\n[TEMPORAL CONSTRAINT: Focus on information from or about {temporal['year']}]\n"
    elif mode == "before":
        return f"\n[TEMPORAL CONSTRAINT: Focus on information predating {temporal['before_year']}]\n"
    elif mode == "after":
        return f"\n[TEMPORAL CONSTRAINT: Focus on information after {temporal['after_year']}]\n"
    elif mode == "recent":
        return "\n[TEMPORAL CONSTRAINT: Prioritize the most recent information available]\n"
    elif mode == "oldest":
        return "\n[TEMPORAL CONSTRAINT: Focus on original/earliest information]\n"
    return ""


# ══════════════════════════════════════════════════════════════
#  CHUNK ACCESS LOG (for decay computation)
# ══════════════════════════════════════════════════════════════

def log_chunk_access(chunk_ids: list[str], query: str = ""):
    """Record which chunks were retrieved for a query."""
    try:
        log = {}
        if ACCESS_LOG.exists():
            with open(ACCESS_LOG) as f:
                log = json.load(f)

        now = time.time()
        for cid in chunk_ids:
            if cid not in log:
                log[cid] = {"access_count": 0, "first_access": now, "last_access": now}
            log[cid]["access_count"] += 1
            log[cid]["last_access"] = now

        with open(ACCESS_LOG, "w") as f:
            json.dump(log, f, indent=2)
    except Exception:
        pass


def compute_decay_score(chunk_id: str) -> float:
    """
    Compute decay score for a chunk using exponential decay.
    score = e^(-λ * days_since_last_access)
    Returns 1.0 for never-accessed chunks (protect new content).
    """
    if not ACCESS_LOG.exists():
        return 1.0
    try:
        with open(ACCESS_LOG) as f:
            log = json.load(f)
        if chunk_id not in log:
            return 1.0   # new chunk, not yet accessed → protect it
        last_access = log[chunk_id]["last_access"]
        days_since  = (time.time() - last_access) / 86400
        return math.exp(-DECAY_LAMBDA * days_since)
    except Exception:
        return 1.0


def get_pruning_candidates() -> list[dict]:
    """
    Return chunks that are candidates for removal:
    - decay_score < PRUNE_THRESHOLD
    - access_count < MIN_ACCESS_TO_PRUNE
    These can be safely removed from ChromaDB to keep the index lean.
    """
    if not ACCESS_LOG.exists():
        return []
    try:
        with open(ACCESS_LOG) as f:
            log = json.load(f)

        candidates = []
        for chunk_id, data in log.items():
            decay = compute_decay_score(chunk_id)
            if decay < PRUNE_THRESHOLD and data["access_count"] < MIN_ACCESS_TO_PRUNE:
                candidates.append({
                    "chunk_id":     chunk_id,
                    "decay_score":  round(decay, 4),
                    "access_count": data["access_count"],
                    "last_access":  datetime.fromtimestamp(data["last_access"]).isoformat(),
                })
        return sorted(candidates, key=lambda x: x["decay_score"])
    except Exception:
        return []


# ══════════════════════════════════════════════════════════════
#  PRUNING EXECUTOR
# ══════════════════════════════════════════════════════════════

def run_pruning(dry_run: bool = True) -> dict:
    """
    Execute the forgetting/pruning cycle.
    If dry_run=True, returns candidates without deleting.
    If dry_run=False, removes stale chunks from ChromaDB.
    """
    candidates = get_pruning_candidates()

    if dry_run or not candidates:
        return {
            "pruning_mode":  "dry_run" if dry_run else "live",
            "candidates":    len(candidates),
            "chunks":        candidates[:20],
            "deleted":       0,
        }

    # Live pruning — remove from FAISS vector store
    deleted = 0
    try:
        from retrieval import delete_document as _del
        # Group candidates by file_id prefix (chunk_id = fileId_vN_chunk_N)
        ids_to_delete = [c["chunk_id"] for c in candidates]

        # Use retrieval module to remove individual chunks by rebuilding index
        import faiss
        import numpy as np
        import json as _json
        from retrieval import FAISS_INDEX_FILE, METADATA_FILE, EMBED_DIM
        from retrieval import _faiss_index, _metadata_store, _store_mtime

        if FAISS_INDEX_FILE.exists() and METADATA_FILE.exists():
            index = faiss.read_index(str(FAISS_INDEX_FILE))
            with open(METADATA_FILE) as f:
                metadata = _json.load(f)

            delete_set = set(ids_to_delete)
            keep_indices = [i for i, m in enumerate(metadata) if m.get("id") not in delete_set]
            deleted = len(metadata) - len(keep_indices)

            new_index = faiss.IndexFlatIP(EMBED_DIM)
            if keep_indices:
                kept_vecs = np.zeros((len(keep_indices), EMBED_DIM), dtype="float32")
                for new_i, old_i in enumerate(keep_indices):
                    try:
                        index.reconstruct(old_i, kept_vecs[new_i])
                    except Exception:
                        pass
                new_index.add(kept_vecs)

            new_metadata = [metadata[i] for i in keep_indices]
            faiss.write_index(new_index, str(FAISS_INDEX_FILE))
            with open(METADATA_FILE, "w") as f:
                _json.dump(new_metadata, f)

        # Clean access log
        if ACCESS_LOG.exists():
            with open(ACCESS_LOG) as f:
                log = _json.load(f)
            for cid in ids_to_delete:
                log.pop(cid, None)
            with open(ACCESS_LOG, "w") as f:
                _json.dump(log, f, indent=2)

    except Exception as e:
        return {"error": str(e), "deleted": 0}

    return {
        "pruning_mode": "live",
        "candidates":   len(candidates),
        "deleted":      deleted,
    }


def get_memory_health() -> dict:
    """Overview of all memory layers for telemetry dashboard."""
    access_log_size = 0
    total_accesses  = 0
    if ACCESS_LOG.exists():
        try:
            with open(ACCESS_LOG) as f:
                log = json.load(f)
            access_log_size = len(log)
            total_accesses  = sum(v["access_count"] for v in log.values())
        except Exception:
            pass

    reflection_stats = get_reflection_stats()
    candidates       = get_pruning_candidates()

    intent_sessions = 0
    if INTENT_LOG.exists():
        try:
            with open(INTENT_LOG) as f:
                trajectories = json.load(f)
            intent_sessions = len(trajectories)
        except Exception:
            pass

    return {
        "chunk_access_tracked":  access_log_size,
        "total_chunk_accesses":  total_accesses,
        "pruning_candidates":    len(candidates),
        "intent_sessions":       intent_sessions,
        "reflection":            reflection_stats,
    }


Overwriting /content/singularity/memory.py


### 5e — knowledge_graph.py
*Knowledge graph — Entity extraction + multi-hop traversal*

In [29]:
%%writefile /content/singularity/knowledge_graph.py
"""
Knowledge Graph Layer (Relational Memory)
==========================================
Implements spec item 9 + Memory Layer 10.

Architecture:
  Document chunks → Entity Extraction (LLM) → Graph Nodes
  Entity pairs    → Relationship Detection  → Graph Edges
  Query           → Graph Traversal         → Multi-hop context enrichment

Graph Structure (JSON-persisted, no external DB needed):
  nodes: { entity_id: { name, type, aliases, file_ids, chunk_ids, created_at } }
  edges: { edge_id:   { source, target, relation, weight, file_id, chunk_id, created_at } }

Entity Types: PERSON, ORG, CONCEPT, METRIC, DATE, LOCATION, PRODUCT, EVENT
Relation Types: ACQUIRED, FOUNDED, OWNS, WORKS_AT, RELATED_TO, CAUSED, DEFINES,
                PART_OF, CITES, COMPETES_WITH, MEASURED_BY, LOCATED_IN

Multi-hop example:
  Q: "Which company acquired a startup founded by X?"
  Step 1: Find X → PERSON node
  Step 2: Traverse FOUNDED edges → find startup ORG node
  Step 3: Traverse ACQUIRED edges → find acquiring company
  Answer: company name with source attribution
"""

import re
import json
import hashlib
import time
from pathlib import Path
from typing import Optional
from collections import defaultdict

import ollama

BASE_DIR   = Path(__file__).parent
GRAPH_DIR  = BASE_DIR / "knowledge_graph"
GRAPH_DIR.mkdir(exist_ok=True)

GRAPH_FILE  = GRAPH_DIR / "graph.json"
INDEX_FILE  = GRAPH_DIR / "entity_index.json"   # name → entity_id fast lookup

LLM_MODEL  = "llama3.2:3b"

# ── Entity & relation type vocabularies ───────────────────────
ENTITY_TYPES  = {"PERSON", "ORG", "CONCEPT", "METRIC", "DATE", "LOCATION", "PRODUCT", "EVENT", "METHOD", "FINDING"}
RELATION_TYPES = {
    "ACQUIRED", "FOUNDED", "OWNS", "WORKS_AT", "RELATED_TO", "CAUSED",
    "DEFINES", "PART_OF", "CITES", "COMPETES_WITH", "MEASURED_BY",
    "LOCATED_IN", "DEVELOPED", "PUBLISHED", "SUPPORTS", "CONTRADICTS",
    "PRECEDES", "FOLLOWS", "USES", "PRODUCES",
}

# ══════════════════════════════════════════════════════════════
#  GRAPH STORAGE
# ══════════════════════════════════════════════════════════════

def _load_graph() -> dict:
    if GRAPH_FILE.exists():
        try:
            with open(GRAPH_FILE) as f:
                return json.load(f)
        except Exception:
            pass
    return {"nodes": {}, "edges": {}}


def _save_graph(graph: dict):
    with open(GRAPH_FILE, "w") as f:
        json.dump(graph, f, indent=2)


def _load_index() -> dict:
    if INDEX_FILE.exists():
        try:
            with open(INDEX_FILE) as f:
                return json.load(f)
        except Exception:
            pass
    return {}


def _save_index(index: dict):
    with open(INDEX_FILE, "w") as f:
        json.dump(index, f, indent=2)


def _entity_id(name: str, etype: str) -> str:
    return hashlib.md5(f"{etype}:{name.lower().strip()}".encode()).hexdigest()[:12]


def _edge_id(source_id: str, target_id: str, relation: str) -> str:
    return hashlib.md5(f"{source_id}:{relation}:{target_id}".encode()).hexdigest()[:12]


# ══════════════════════════════════════════════════════════════
#  ENTITY EXTRACTION (LLM-powered)
# ══════════════════════════════════════════════════════════════

EXTRACT_PROMPT = """You are an expert information extraction system.

Extract all named entities and relationships from the text below.

Output ONLY valid JSON in this exact format (no explanation, no markdown):
{
  "entities": [
    {"name": "entity name", "type": "ENTITY_TYPE", "aliases": ["alt name 1"]}
  ],
  "relations": [
    {"source": "entity name", "target": "entity name", "relation": "RELATION_TYPE"}
  ]
}

Entity types: PERSON, ORG, CONCEPT, METRIC, DATE, LOCATION, PRODUCT, EVENT, METHOD, FINDING
Relation types: ACQUIRED, FOUNDED, OWNS, WORKS_AT, RELATED_TO, CAUSED, DEFINES, PART_OF,
                CITES, COMPETES_WITH, MEASURED_BY, LOCATED_IN, DEVELOPED, PUBLISHED,
                SUPPORTS, CONTRADICTS, PRECEDES, FOLLOWS, USES, PRODUCES

Rules:
- Only extract entities clearly mentioned in the text
- Only extract relations that are explicitly stated
- If nothing to extract, return {"entities": [], "relations": []}
- Output raw JSON only, no ```json fences"""


def extract_entities_and_relations(text: str) -> dict:
    """Use LLM to extract entities and relationships from a chunk."""
    # Truncate very long chunks to avoid overwhelming the extraction
    text_sample = text[:1200]
    try:
        response = ollama.chat(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": EXTRACT_PROMPT},
                {"role": "user", "content": f"TEXT:\n{text_sample}"}
            ],
            stream=False,
            options={"temperature": 0.0, "num_ctx": 2048}
        )
        raw = response["message"]["content"].strip()
        # Strip any accidental markdown
        raw = re.sub(r"^```json\s*|```\s*$", "", raw, flags=re.MULTILINE).strip()
        data = json.loads(raw)
        # Validate types
        data["entities"] = [
            e for e in data.get("entities", [])
            if e.get("type", "").upper() in ENTITY_TYPES and e.get("name")
        ]
        data["relations"] = [
            r for r in data.get("relations", [])
            if r.get("relation", "").upper() in RELATION_TYPES
            and r.get("source") and r.get("target")
        ]
        return data
    except Exception:
        return {"entities": [], "relations": []}


# ══════════════════════════════════════════════════════════════
#  GRAPH BUILDING
# ══════════════════════════════════════════════════════════════

def add_chunk_to_graph(
    chunk_text: str,
    chunk_id: str,
    file_id: str,
    filename: str,
    page_num: int = 1,
):
    """
    Extract entities/relations from a chunk and add to the knowledge graph.
    Called during ingestion for representative chunks (every Nth chunk).
    """
    extracted = extract_entities_and_relations(chunk_text)

    if not extracted["entities"] and not extracted["relations"]:
        return {"nodes_added": 0, "edges_added": 0}

    graph = _load_graph()
    index = _load_index()
    now   = time.time()
    nodes_added = 0
    edges_added = 0

    # ── Add / merge entity nodes ───────────────────────────────
    name_to_id = {}   # local map: extracted name → graph node id

    for ent in extracted["entities"]:
        name  = ent["name"].strip()
        etype = ent["type"].upper()
        eid   = _entity_id(name, etype)
        name_to_id[name.lower()] = eid

        # Also map aliases
        for alias in ent.get("aliases", []):
            name_to_id[alias.lower()] = eid
            index[alias.lower()] = eid

        if eid not in graph["nodes"]:
            graph["nodes"][eid] = {
                "id":         eid,
                "name":       name,
                "type":       etype,
                "aliases":    ent.get("aliases", []),
                "file_ids":   [file_id],
                "chunk_ids":  [chunk_id],
                "mentions":   1,
                "created_at": now,
                "updated_at": now,
            }
            nodes_added += 1
        else:
            # Merge — update mention count and sources
            node = graph["nodes"][eid]
            node["mentions"] = node.get("mentions", 1) + 1
            node["updated_at"] = now
            if file_id not in node["file_ids"]:
                node["file_ids"].append(file_id)
            if chunk_id not in node["chunk_ids"]:
                node["chunk_ids"].append(chunk_id)

        index[name.lower()] = eid

    # ── Add relationship edges ─────────────────────────────────
    for rel in extracted["relations"]:
        src_name  = rel["source"].strip().lower()
        tgt_name  = rel["target"].strip().lower()
        relation  = rel["relation"].upper()

        src_id = name_to_id.get(src_name) or index.get(src_name)
        tgt_id = name_to_id.get(tgt_name) or index.get(tgt_name)

        if not src_id or not tgt_id:
            continue

        eid = _edge_id(src_id, tgt_id, relation)
        if eid not in graph["edges"]:
            graph["edges"][eid] = {
                "id":         eid,
                "source":     src_id,
                "target":     tgt_id,
                "relation":   relation,
                "weight":     1,
                "file_id":    file_id,
                "chunk_id":   chunk_id,
                "filename":   filename,
                "page_num":   page_num,
                "created_at": now,
            }
            edges_added += 1
        else:
            # Strengthen existing edge
            graph["edges"][eid]["weight"] = graph["edges"][eid].get("weight", 1) + 1

    _save_graph(graph)
    _save_index(index)
    return {"nodes_added": nodes_added, "edges_added": edges_added}


# ══════════════════════════════════════════════════════════════
#  GRAPH TRAVERSAL (Multi-hop)
# ══════════════════════════════════════════════════════════════

def _find_entities_by_name(name: str, index: dict, graph: dict) -> list[str]:
    """Fuzzy entity lookup — exact then substring match."""
    name_lower = name.lower().strip()

    # Exact match
    if name_lower in index:
        return [index[name_lower]]

    # Substring match
    matches = []
    for indexed_name, eid in index.items():
        if name_lower in indexed_name or indexed_name in name_lower:
            matches.append(eid)

    return list(set(matches))


def traverse_graph(
    query: str,
    max_hops: int = 3,
    max_nodes: int = 15,
) -> dict:
    """
    Multi-hop graph traversal from query entities.

    Returns:
        {
          "entities_found": [...],
          "relationships": [...],
          "chunk_ids": [...],   # chunks to retrieve as additional context
          "graph_context": str  # formatted text for LLM injection
        }
    """
    graph = _load_graph()
    index = _load_index()

    if not graph["nodes"]:
        return {"entities_found": [], "relationships": [], "chunk_ids": [], "graph_context": ""}

    # ── Extract query entities ─────────────────────────────────
    # Simple extraction: capitalized words / phrases
    candidate_names = re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', query)
    # Also try quoted strings
    candidate_names += re.findall(r'"([^"]+)"', query)
    candidate_names = list(set(candidate_names))

    # Find starting nodes
    seed_ids = set()
    for name in candidate_names:
        found = _find_entities_by_name(name, index, graph)
        seed_ids.update(found)

    if not seed_ids:
        return {"entities_found": [], "relationships": [], "chunk_ids": [], "graph_context": ""}

    # ── BFS traversal ─────────────────────────────────────────
    visited_nodes = set(seed_ids)
    visited_edges = set()
    frontier      = set(seed_ids)
    all_relations = []
    all_chunk_ids = []

    for hop in range(max_hops):
        if not frontier or len(visited_nodes) >= max_nodes:
            break

        next_frontier = set()
        for node_id in frontier:
            # Find all edges from/to this node
            for edge_id, edge in graph["edges"].items():
                if edge_id in visited_edges:
                    continue
                if edge["source"] == node_id or edge["target"] == node_id:
                    visited_edges.add(edge_id)
                    other_id = edge["target"] if edge["source"] == node_id else edge["source"]

                    src_node = graph["nodes"].get(edge["source"], {})
                    tgt_node = graph["nodes"].get(edge["target"], {})

                    rel_str = (
                        f"{src_node.get('name','?')} "
                        f"—[{edge['relation']}]→ "
                        f"{tgt_node.get('name','?')} "
                        f"[{edge.get('filename','?')} p.{edge.get('page_num','?')}]"
                    )
                    all_relations.append(rel_str)

                    # Collect source chunk IDs for retrieval enrichment
                    if edge.get("chunk_id") and edge["chunk_id"] not in all_chunk_ids:
                        all_chunk_ids.append(edge["chunk_id"])

                    if other_id not in visited_nodes:
                        visited_nodes.add(other_id)
                        next_frontier.add(other_id)

        frontier = next_frontier

    # ── Format graph context ───────────────────────────────────
    entity_summaries = []
    for nid in visited_nodes:
        node = graph["nodes"].get(nid)
        if node:
            sources = ", ".join(set(node.get("file_ids", [])))
            entity_summaries.append(
                f"  {node['type']}: {node['name']}"
                f" (mentioned {node.get('mentions',1)}x, sources: {sources})"
            )

    graph_context = ""
    if entity_summaries or all_relations:
        graph_context = (
            "KNOWLEDGE GRAPH CONTEXT:\n"
            + "Entities:\n" + "\n".join(entity_summaries[:max_nodes]) + "\n"
            + "Relationships:\n" + "\n".join(f"  {r}" for r in all_relations[:30])
        )

    return {
        "entities_found": list(visited_nodes),
        "relationships":  all_relations,
        "chunk_ids":      all_chunk_ids[:10],
        "graph_context":  graph_context,
    }


# ══════════════════════════════════════════════════════════════
#  GRAPH STATS & MANAGEMENT
# ══════════════════════════════════════════════════════════════

def get_graph_stats() -> dict:
    graph = _load_graph()
    node_types: dict = defaultdict(int)
    rel_types: dict  = defaultdict(int)

    for node in graph["nodes"].values():
        node_types[node.get("type", "UNKNOWN")] += 1
    for edge in graph["edges"].values():
        rel_types[edge.get("relation", "UNKNOWN")] += 1

    return {
        "total_nodes":    len(graph["nodes"]),
        "total_edges":    len(graph["edges"]),
        "node_types":     dict(node_types),
        "relation_types": dict(rel_types),
        "top_entities":   sorted(
            [{"name": n["name"], "type": n["type"], "mentions": n.get("mentions", 1)}
             for n in graph["nodes"].values()],
            key=lambda x: x["mentions"], reverse=True
        )[:15],
    }


def delete_graph_for_file(file_id: str):
    """Remove all graph nodes and edges associated with a file."""
    graph = _load_graph()
    index = _load_index()

    # Remove nodes
    nodes_to_remove = {
        nid for nid, node in graph["nodes"].items()
        if file_id in node.get("file_ids", [])
    }
    # If node has multiple sources, just remove this file from its sources
    for nid in list(nodes_to_remove):
        node = graph["nodes"][nid]
        node["file_ids"] = [f for f in node["file_ids"] if f != file_id]
        if not node["file_ids"]:
            del graph["nodes"][nid]
        else:
            nodes_to_remove.discard(nid)  # still referenced by other files

    # Remove edges from this file
    graph["edges"] = {
        eid: edge for eid, edge in graph["edges"].items()
        if edge.get("file_id") != file_id
    }

    # Rebuild index (remove stale entries)
    index = {name: eid for name, eid in index.items() if eid in graph["nodes"]}

    _save_graph(graph)
    _save_index(index)


Overwriting /content/singularity/knowledge_graph.py


### 5f — main.py
*FastAPI backend — All endpoints with session manifest cache*

In [30]:
%%writefile /content/singularity/main.py
"""
Enterprise NotebookLM-Style RAG Backend  v4.0
==============================================
Changes vs v3:
  - Supports PDF, DOCX, TXT, CSV, XLSX, PPTX, MD, HTML
  - Query rewriting disabled by default (can be toggled in frontend) — eliminates 3-10s pre-retrieval latency
  - History bug fixed: history is sanitised properly before passing to LLM
  - Knowledge-graph extraction still runs in background thread
"""

import uuid
import shutil
import json
import time
import threading
import platform
import subprocess
from pathlib import Path
from typing import Optional, List

from tracer import Tracer, get_recent_traces, get_trace_by_id, get_trace_stats

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

from ingestion import ingest_document, SUPPORTED_EXTENSIONS
from retrieval import (
    get_relevant_context,
    get_full_document_context,
    get_all_documents_context,
    list_all_documents,
    delete_document,
    record_feedback,
)
from llm import (
    stream_chat,
    stream_conversational,
    stream_summary,
    rewrite_query,
    update_user_preferences,
    get_user_preferences,
)

import re as _re

# Patterns that signal the user is asking ABOUT the conversation itself,
# not querying documents. When matched, skip retrieval entirely.
_CONVERSATIONAL_PATTERNS = [
    r"what (did|were|was|have) (we|you|i) (talk|discuss|say|mention|cover|ask)",
    r"what (just|we) (said|talked|covered|discussed)",
    r"(remind|tell) me (what|about) (we|you) (just|were|have|had)",
    r"(summarize|recap|summary of|what was) (our|this|the) (conversation|chat|discussion|session)",
    r"what (did i|i) (ask|say|tell) (you|just)",
    r"(earlier|before|just now|previously|last question|previous question)",
    r"what (was|were) (we|you) (just|talking|discussing)",
    r"(go back|refer back|look back) to (what|our)",
    r"(you said|you mentioned|you told me|you explained)",
    r"(what i (said|asked|mentioned))",
    r"^(and|so|also|but)? ?(can you )?(tell me )?(more|again|continue|expand)",
]
_CONV_RE = [_re.compile(p, _re.IGNORECASE) for p in _CONVERSATIONAL_PATTERNS]


def _is_conversational(query: str, history: list[dict]) -> bool:
    """
    Returns True if the query is about the conversation itself (memory/recall)
    rather than a document retrieval question. When True, skip retrieval and
    answer from history only.
    """
    if not history:
        return False
    q = query.strip().lower()
    # Short queries that reference 'we/you/I' are almost always conversational
    if len(q.split()) <= 12:
        for pattern in _CONV_RE:
            if pattern.search(q):
                return True
    return False


_reflection_available = False
try:
    from memory import (
        reflect_on_answer, get_memory_health, run_pruning,
        update_intent_trajectory, get_intent_trajectory,
    )
    from knowledge_graph import get_graph_stats, delete_graph_for_file
    _reflection_available = True
except ImportError:
    pass

BASE_DIR   = Path(__file__).parent
UPLOAD_DIR = BASE_DIR / "uploads"
UPLOAD_DIR.mkdir(exist_ok=True)

app = FastAPI(title="Enterprise RAG — NotebookLM", version="4.0")

def _print_system_card():
    """Print a full system card to stdout at startup."""
    # GPU info
    try:
        import torch
        if torch.cuda.is_available():
            gpu_name = torch.cuda.get_device_name(0)
            vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
            gpu_str  = f"{gpu_name}  ({vram_gb:.1f} GB VRAM)"
        else:
            gpu_str  = "CPU only"
    except Exception:
        gpu_str = "Unknown"

    # RAM
    try:
        import psutil
        ram_gb = psutil.virtual_memory().total / 1e9
        ram_str = f"{ram_gb:.1f} GB"
    except Exception:
        try:
            mem = subprocess.run(['free','-h'], capture_output=True, text=True).stdout
            ram_str = mem.split('\n')[1].split()[1] if mem else "Unknown"
        except Exception:
            ram_str = "Unknown"

    # Python / OS
    py_ver = platform.python_version()
    os_str = f"{platform.system()} {platform.release()}"

    W = 62
    def row(label, value, w=W):
        label = f"  {label}"
        dots  = "." * max(1, w - len(label) - len(str(value)) - 2)
        return f"│{label}{dots}{value}│"

    def divider(w=W):
        return f"├{'─'*(w)}┤"

    card = [
        f"╔{'═'*W}╗",
        f"║{'  🌑  SINGULARITY  —  Enterprise RAG  v4.0':^{W}}║",
        f"╠{'═'*W}╣",
        f"║{'  HARDWARE':^{W}}║",
        divider(),
        row("GPU",          gpu_str),
        row("RAM",          ram_str),
        row("OS",           os_str),
        row("Python",       py_ver),
        f"╠{'═'*W}╣",
        f"║{'  PIPELINE':^{W}}║",
        divider(),
        row("Document parsing",  "PyMuPDF + pdfplumber (tables) + pytesseract (OCR)"),
        row("Embedding model",   "BAAI/bge-m3  →  currently all-MiniLM-L6-v2"),
        row("Embed device",      gpu_str.split("(")[0].strip()),
        row("Vector store",      "FAISS (IndexFlatIP) + BM25  →  RRF fusion"),
        row("Cross-encoder",     "ms-marco-MiniLM-L-6-v2  (re-rank top-21)"),
        row("LLM",               "Ollama  llama3.2:3b"),
        row("Chunk size",        "200 words  |  overlap: 50 words"),
        f"╠{'═'*W}╣",
        f"║{'  PERFORMANCE  (Colab T4 benchmarks)':^{W}}║",
        divider(),
        row("Avg latency / query",  "35 – 45 s"),
        row("P95 latency",          "60 s"),
        row("P99 latency",          "80 s"),
        row("Throughput",           "0.03 QPS"),
        row("Avg input tokens",     "800"),
        row("Avg output tokens",    "600"),
        row("Max input tokens",     "5,500"),
        row("Max output tokens",    "2,048"),
        f"╠{'═'*W}╣",
        f"║{'  STAGE BREAKDOWN  (per query)':^{W}}║",
        divider(),
        row("Query rewriting",      "~12 s"),
        row("Hybrid retrieval",     "~3 s"),
        row("Re-ranking",           "~1.1 s"),
        row("Context assembly",     "<10 ms"),
        row("LLM generation",       "~20 – 60 s"),
        f"╠{'═'*W}╣",
        f"║{'  REQUIREMENTS':^{W}}║",
        divider(),
        row("Min RAM",              "16 GB"),
        row("Min GPU VRAM",         "8 GB  (16 GB recommended)"),
        row("Model weights",        "~8 GB  (llama3.2:3b + embed)"),
        row("Vector DB + docs",     "~200 MB typical"),
        f"╚{'═'*W}╝",
    ]

    print("\n".join(card))
    print()

_print_system_card()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


# ══════════════════════════════════════════════════════════════
#  UPLOAD
# ══════════════════════════════════════════════════════════════

def _build_graph_background(chunks, file_id, original_filename, version):
    if not _reflection_available:
        return
    try:
        from knowledge_graph import add_chunk_to_graph
        total_nodes = total_edges = 0
        for i, chunk in enumerate(chunks[::3]):
            chunk_id = f"{file_id}_v{version}_chunk_{i*3}"
            result = add_chunk_to_graph(
                chunk_text=chunk["text"],
                chunk_id=chunk_id,
                file_id=file_id,
                filename=original_filename,
                page_num=chunk.get("page_num", 1),
            )
            total_nodes += result.get("nodes_added", 0)
            total_edges += result.get("edges_added", 0)
        print(f"[KG-BG] ✓ Graph: +{total_nodes} nodes, +{total_edges} edges for {file_id}")
    except Exception as e:
        print(f"[KG-BG] Warning: {e}")


@app.post("/upload")
async def upload_file(file: UploadFile = File(...)):
    """
    Ingestion pipeline: Parse → Chunk → Dedup → Embed → FAISS store.
    Supports: PDF, DOCX, TXT, CSV, XLSX, PPTX, MD, HTML
    """
    ext = Path(file.filename).suffix.lower()

    if ext not in SUPPORTED_EXTENSIONS:
        raise HTTPException(
            status_code=400,
            detail=f"Unsupported type '{ext}'. Supported: {', '.join(sorted(SUPPORTED_EXTENSIONS))}"
        )

    file_id   = str(uuid.uuid4())
    save_path = UPLOAD_DIR / f"{file_id}{ext}"

    with save_path.open("wb") as f:
        shutil.copyfileobj(file.file, f)

    try:
        result = ingest_document(
            file_path=str(save_path),
            file_id=file_id,
            original_filename=file.filename,
        )

        if _reflection_available and result.get("_chunks_ref"):
            t = threading.Thread(
                target=_build_graph_background,
                args=(result["_chunks_ref"], file_id, file.filename, result["version"]),
                daemon=True,
            )
            t.start()

        return {
            "file_id":           file_id,
            "filename":          file.filename,
            "chunks_created":    result["chunks"],
            "images_extracted":  result["images"],
            "version":           result["version"],
            "sections_detected": result.get("sections", []),
            "status":            "ingested",
        }
    except Exception as e:
        save_path.unlink(missing_ok=True)
        raise HTTPException(status_code=500, detail=f"Ingestion failed: {str(e)}")


# ══════════════════════════════════════════════════════════════
#  CHAT
# ══════════════════════════════════════════════════════════════

class ChatRequest(BaseModel):
    message: str
    file_id: Optional[str] = None
    history: Optional[List[dict]] = []
    use_query_rewrite: bool = False   # default OFF for speed; user can toggle on
    use_multihop: bool = True


@app.post("/chat")
async def chat(req: ChatRequest):
    # Sanitise history — only keep role+content, filter invalid roles
    clean_history = [
        {"role": m["role"], "content": str(m["content"])}
        for m in (req.history or [])
        if m.get("role") in ("user", "assistant") and m.get("content")
    ]

    # ── Tracer: start ────────────────────────────────────────────
    tracer = Tracer(session_id=req.session_id if hasattr(req, "session_id") else None)
    tracer.start()
    tracer.log_query_intake(
        raw_query=req.message,
        history=clean_history,
        session_memory_used=bool(clean_history),
    )
    tracer.log_memory(
        session_entries_used=len(clean_history),
        session_tokens=sum(len(m["content"].split()) for m in clean_history),
    )

    # Fire-and-forget preference tracking
    threading.Thread(
        target=update_user_preferences,
        args=(req.message, req.file_id),
        daemon=True,
    ).start()

    all_docs = list_all_documents()

    # ── Conversational shortcut ──────────────────────────────────────────────
    if _is_conversational(req.message, clean_history):
        def _gen_conv():
            try:
                meta_payload = {"type": "meta", "rewritten_query": "", "conversational": True,
                                "trace_id": tracer.get_trace_id()}
                yield f"data: {json.dumps(meta_payload)}\n\n"
                for token in stream_conversational(
                    query=req.message,
                    history=clean_history,
                ):
                    safe = token.replace("\\", "\\\\").replace('"', '\\"').replace("\n", "\\n")
                    yield f"data: {safe}\n\n"
            except Exception as e:
                yield f"data: [ERROR] {str(e)}\n\n"
            finally:
                tracer.finalize()
                yield "data: [DONE]\n\n"

        return StreamingResponse(_gen_conv(), media_type="text/event-stream",
                                 headers={"Cache-Control": "no-cache",
                                          "X-Accel-Buffering": "no",
                                          "Connection": "keep-alive"})

    # ── Document retrieval path (normal queries) ─────────────────────────────

    _t_start = time.perf_counter()

    # Optional query rewriting
    rewritten = None
    _t_rw0 = time.perf_counter()
    if req.use_query_rewrite:
        history_text = " ".join(h.get("content", "") for h in clean_history[-4:])
        rw = rewrite_query(req.message, history_text)
        if rw and rw != req.message:
            rewritten = rw
    _t_after_rewrite = time.perf_counter()
    rewrite_ms = (_t_after_rewrite - _t_rw0) * 1000
    tracer.log_rewrite(
        triggered=bool(rewritten),
        rewritten_query=rewritten,
        latency_ms=rewrite_ms,
    )
    if req.use_query_rewrite:
        print(f"[TIMER] Query rewriting: {rewrite_ms:.0f} ms")

    search_query = rewritten or req.message

    # Retrieval
    _t_ret0 = time.perf_counter()
    context_result = get_relevant_context(
        query=search_query,
        file_id=req.file_id,
        n_results=7,
        use_multihop=req.use_multihop,
    )
    if isinstance(context_result, tuple):
        context, retrieval_meta = context_result
    else:
        context, retrieval_meta = context_result, {}
    _t_after_retrieval = time.perf_counter()
    retrieval_ms = (_t_after_retrieval - _t_ret0) * 1000
    tracer.log_retrieval(
        retrieval_meta=retrieval_meta,
        context=context or "",
        latency_ms=retrieval_ms,
        n_results=7,
    )
    print(f"[TIMER] Retrieval (BM25+FAISS+RRF+rerank): {retrieval_ms:.0f} ms")

    if not all_docs:
        tracer.finalize()
        async def no_docs():
            yield "data: No documents have been uploaded yet. Please upload a document using the sidebar.\n\n"
            yield "data: [DONE]\n\n"
        return StreamingResponse(no_docs(), media_type="text/event-stream",
                                 headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

    if not context:
        if req.file_id:
            context = get_full_document_context(req.file_id, max_chunks=10)
        else:
            context = get_all_documents_context(max_chunks_per_doc=5)

    # Build doc manifest
    if req.file_id:
        scoped_docs = [d for d in all_docs if d["file_id"] == req.file_id]
    else:
        scoped_docs = all_docs
    doc_manifest = [d.get("filename", d["file_id"]) for d in scoped_docs]

    temporal_info = retrieval_meta.get("temporal", {})
    has_graph     = bool(retrieval_meta.get("graph_context"))
    _answer_buffer = []
    _t_after_context_assembly = time.perf_counter()
    print(f"[TIMER] Context assembly: {(_t_after_context_assembly - _t_after_retrieval)*1000:.0f} ms")

    def generate():
        _t_llm_start = time.perf_counter()
        _t_first_token = None
        _token_count = 0
        tracer.log_generation_start()
        try:
            meta_payload = {
                "type": "meta",
                "rewritten_query": rewritten or "",
                "doc_count": len(doc_manifest),
                "trace_id": tracer.get_trace_id(),
            }
            if temporal_info.get("has_temporal"):
                meta_payload["temporal_mode"] = temporal_info.get("mode")
            if has_graph:
                meta_payload["graph_enriched"] = True
            if len(doc_manifest) > 1:
                meta_payload["multi_doc"] = True
                meta_payload["doc_names"] = doc_manifest
            yield f"data: {json.dumps(meta_payload)}\n\n"

            for token in stream_chat(
                query=req.message,
                context=context,
                history=clean_history,
                rewritten_query=rewritten,
                doc_manifest=doc_manifest,
            ):
                if _t_first_token is None:
                    _t_first_token = time.perf_counter()
                    print(f"[TIMER] Time to first token: {(_t_first_token - _t_llm_start)*1000:.0f} ms")
                _token_count += 1
                _answer_buffer.append(token)
                safe = token.replace('\\', '\\\\').replace('"', '\\"').replace('\n', '\\n')
                yield f"data: {safe}\n\n"

        except Exception as e:
            yield f"data: [ERROR] {str(e)}\n\n"
        finally:
            _t_end = time.perf_counter()
            _t_ft  = _t_first_token or _t_llm_start
            total_ms     = (_t_end - _t_start)                       * 1000
            rewrite_ms_f = (_t_after_rewrite - _t_start)             * 1000
            retrieval_ms_f = (_t_after_retrieval - _t_after_rewrite) * 1000
            context_ms   = (_t_after_context_assembly - _t_after_retrieval) * 1000
            llm_ms       = (_t_end - _t_ft)                          * 1000

            print(f"\n[QUERY SUMMARY] ─────────────────────────────────────")
            print(f"[QUERY SUMMARY]  Original  : \"{req.message[:70]}\"")
            print(f"[QUERY SUMMARY]  Rewritten : \"{(rewritten or req.message)[:70]}\"")
            print(f"[QUERY SUMMARY]  ┌─────────────────────────┬──────────┐")
            print(f"[QUERY SUMMARY]  │ Stage                   │ Time     │")
            print(f"[QUERY SUMMARY]  ├─────────────────────────┼──────────┤")
            print(f"[QUERY SUMMARY]  │ Query rewriting         │ {rewrite_ms_f:>6.0f}ms │")
            print(f"[QUERY SUMMARY]  │ Hybrid retrieval        │ {retrieval_ms_f:>6.0f}ms │")
            print(f"[QUERY SUMMARY]  │ Context assembly        │ {context_ms:>6.0f}ms │")
            print(f"[QUERY SUMMARY]  │ LLM generation          │ {llm_ms:>6.0f}ms │")
            print(f"[QUERY SUMMARY]  ├─────────────────────────┼──────────┤")
            print(f"[QUERY SUMMARY]  │ TOTAL                   │ {total_ms:>6.0f}ms │")
            print(f"[QUERY SUMMARY]  └─────────────────────────┴──────────┘")
            print(f"[QUERY SUMMARY]  Output tokens (approx): {_token_count}")
            print(f"[QUERY SUMMARY]  Trace ID: {tracer.get_trace_id()}")
            print(f"[QUERY SUMMARY] ─────────────────────────────────────\n")

            yield "data: [DONE]\n\n"

            # ── Post-generation: reflection + tracer finalize ────
            full_answer = "".join(_answer_buffer)
            tracer.log_generation_end(
                answer=full_answer,
                token_count=_token_count,
                latency_ms=llm_ms,
            )

            if _reflection_available and full_answer:
                def _reflect():
                    try:
                        reflection = reflect_on_answer(
                            query=req.message, context=context,
                            answer=full_answer, session_id=None,
                        )
                        tracer.log_generation_end(
                            answer=full_answer,
                            token_count=_token_count,
                            latency_ms=llm_ms,
                            reflection=reflection,
                        )
                        update_intent_trajectory(
                            query=req.message,
                            answer_quality=reflection.get("answer_quality", "unknown"),
                        )
                    except Exception:
                        pass
                    finally:
                        tracer.finalize()
                threading.Thread(target=_reflect, daemon=True).start()
            else:
                tracer.finalize()

    return StreamingResponse(generate(), media_type="text/event-stream",
                             headers={
                                 "Cache-Control": "no-cache",
                                 "X-Accel-Buffering": "no",
                                 "Connection": "keep-alive",
                             })
    # Sanitise history — only keep role+content, filter invalid roles
    clean_history = [
        {"role": m["role"], "content": str(m["content"])}
        for m in (req.history or [])
        if m.get("role") in ("user", "assistant") and m.get("content")
    ]

    # Fire-and-forget preference tracking
    threading.Thread(
        target=update_user_preferences,
        args=(req.message, req.file_id),
        daemon=True,
    ).start()

    all_docs = list_all_documents()

    # ── Conversational shortcut ──────────────────────────────────────────────
    # If the user is asking ABOUT the conversation (recall, summary, "what did
    # we say"), skip document retrieval entirely and answer purely from history.
    if _is_conversational(req.message, clean_history):
        def _gen_conv():
            try:
                meta_payload = {"type": "meta", "rewritten_query": "", "conversational": True}
                yield f"data: {json.dumps(meta_payload)}\n\n"
                for token in stream_conversational(
                    query=req.message,
                    history=clean_history,
                ):
                    safe = token.replace("\\", "\\\\").replace('"', '\\"').replace("\n", "\\n")
                    yield f"data: {safe}\n\n"
            except Exception as e:
                yield f"data: [ERROR] {str(e)}\n\n"
            finally:
                yield "data: [DONE]\n\n"

        return StreamingResponse(_gen_conv(), media_type="text/event-stream",
                                 headers={"Cache-Control": "no-cache",
                                          "X-Accel-Buffering": "no",
                                          "Connection": "keep-alive"})

    # ── Document retrieval path (normal queries) ─────────────────────────────

    _t_start = time.perf_counter()

    # Optional query rewriting — only for non-conversational queries
    rewritten = None
    if req.use_query_rewrite:
        history_text = " ".join(h.get("content", "") for h in clean_history[-4:])
        rw = rewrite_query(req.message, history_text)
        if rw and rw != req.message:
            rewritten = rw

    search_query = rewritten or req.message
    _t_after_rewrite = time.perf_counter()
    if req.use_query_rewrite:
        print(f"[TIMER] Query rewriting: {(_t_after_rewrite - _t_start)*1000:.0f} ms")

    # Retrieval
    context_result = get_relevant_context(
        query=search_query,
        file_id=req.file_id,
        n_results=7,
        use_multihop=req.use_multihop,
    )
    if isinstance(context_result, tuple):
        context, retrieval_meta = context_result
    else:
        context, retrieval_meta = context_result, {}
    _t_after_retrieval = time.perf_counter()
    print(f"[TIMER] Retrieval (BM25+FAISS+RRF+rerank): {(_t_after_retrieval - _t_after_rewrite)*1000:.0f} ms")

    if not all_docs:
        async def no_docs():
            yield "data: No documents have been uploaded yet. Please upload a document using the sidebar.\n\n"
            yield "data: [DONE]\n\n"
        return StreamingResponse(no_docs(), media_type="text/event-stream",
                                 headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

    if not context:
        if req.file_id:
            context = get_full_document_context(req.file_id, max_chunks=10)
        else:
            context = get_all_documents_context(max_chunks_per_doc=5)

    # Build doc manifest — tells the LLM exactly which documents are in scope
    if req.file_id:
        scoped_docs = [d for d in all_docs if d["file_id"] == req.file_id]
    else:
        scoped_docs = all_docs
    doc_manifest = [d.get("filename", d["file_id"]) for d in scoped_docs]

    temporal_info = retrieval_meta.get("temporal", {})
    has_graph     = bool(retrieval_meta.get("graph_context"))
    _answer_buffer = []
    _t_after_context_assembly = time.perf_counter()
    print(f"[TIMER] Context assembly: {(_t_after_context_assembly - _t_after_retrieval)*1000:.0f} ms")

    def generate():
        _t_llm_start = time.perf_counter()
        _t_first_token = None
        _token_count = 0
        try:
            meta_payload = {"type": "meta", "rewritten_query": rewritten or "", "doc_count": len(doc_manifest)}
            if temporal_info.get("has_temporal"):
                meta_payload["temporal_mode"] = temporal_info.get("mode")
            if has_graph:
                meta_payload["graph_enriched"] = True
            if len(doc_manifest) > 1:
                meta_payload["multi_doc"] = True
                meta_payload["doc_names"] = doc_manifest
            yield f"data: {json.dumps(meta_payload)}\n\n"

            for token in stream_chat(
                query=req.message,
                context=context,
                history=clean_history,
                rewritten_query=rewritten,
                doc_manifest=doc_manifest,
            ):
                if _t_first_token is None:
                    _t_first_token = time.perf_counter()
                    print(f"[TIMER] Time to first token: {(_t_first_token - _t_llm_start)*1000:.0f} ms")
                _token_count += 1
                _answer_buffer.append(token)
                # Escape for SSE: newlines → \\n, quotes → \\"
                safe = token.replace('\\', '\\\\').replace('"', '\\"').replace('\n', '\\n')
                yield f"data: {safe}\n\n"

        except Exception as e:
            yield f"data: [ERROR] {str(e)}\n\n"
        finally:
            _t_end = time.perf_counter()
            _t_ft  = _t_first_token or _t_llm_start
            total_ms   = (_t_end - _t_start)   * 1000
            rewrite_ms = (_t_after_rewrite - _t_start) * 1000
            retrieval_ms = (_t_after_retrieval - _t_after_rewrite) * 1000
            context_ms = (_t_after_context_assembly - _t_after_retrieval) * 1000
            llm_ms     = (_t_end - _t_ft) * 1000

            print(f"\n[QUERY SUMMARY] ─────────────────────────────────────")
            print(f"[QUERY SUMMARY]  Original  : \"{req.message[:70]}\"")
            print(f"[QUERY SUMMARY]  Rewritten : \"{(rewritten or req.message)[:70]}\"")
            print(f"[QUERY SUMMARY]  ┌─────────────────────────┬──────────┐")
            print(f"[QUERY SUMMARY]  │ Stage                   │ Time     │")
            print(f"[QUERY SUMMARY]  ├─────────────────────────┼──────────┤")
            print(f"[QUERY SUMMARY]  │ Query rewriting         │ {rewrite_ms:>6.0f}ms │")
            print(f"[QUERY SUMMARY]  │ Hybrid retrieval        │ {retrieval_ms:>6.0f}ms │")
            print(f"[QUERY SUMMARY]  │ Context assembly        │ {context_ms:>6.0f}ms │")
            print(f"[QUERY SUMMARY]  │ LLM generation          │ {llm_ms:>6.0f}ms │")
            print(f"[QUERY SUMMARY]  ├─────────────────────────┼──────────┤")
            print(f"[QUERY SUMMARY]  │ TOTAL                   │ {total_ms:>6.0f}ms │")
            print(f"[QUERY SUMMARY]  └─────────────────────────┴──────────┘")
            print(f"[QUERY SUMMARY]  Output tokens (approx): {_token_count}")
            print(f"[QUERY SUMMARY] ─────────────────────────────────────\n")
            yield "data: [DONE]\n\n"
            if _reflection_available and _answer_buffer:
                full_answer = "".join(_answer_buffer)
                def _reflect():
                    try:
                        reflection = reflect_on_answer(
                            query=req.message, context=context,
                            answer=full_answer, session_id=None,
                        )
                        update_intent_trajectory(
                            query=req.message,
                            answer_quality=reflection.get("answer_quality", "unknown"),
                        )
                    except Exception:
                        pass
                threading.Thread(target=_reflect, daemon=True).start()

    return StreamingResponse(generate(), media_type="text/event-stream",
                             headers={
                                 "Cache-Control": "no-cache",
                                 "X-Accel-Buffering": "no",
                                 "Connection": "keep-alive",
                             })


# ══════════════════════════════════════════════════════════════
#  SUMMARIZE
# ══════════════════════════════════════════════════════════════

@app.post("/summarize/{file_id}")
async def summarize(file_id: str):
    docs     = list_all_documents()
    doc_info = next((d for d in docs if d["file_id"] == file_id), {})
    filename = doc_info.get("filename", "")
    context  = get_full_document_context(file_id=file_id, max_chunks=30)
    if not context:
        raise HTTPException(status_code=404, detail="No content found for document.")

    def generate():
        try:
            for token in stream_summary(context, filename=filename):
                safe = token.replace('\\', '\\\\').replace('"', '\\"').replace('\n', '\\n')
                yield f"data: {safe}\n\n"
        except Exception as e:
            yield f"data: [ERROR] {str(e)}\n\n"
        finally:
            yield "data: [DONE]\n\n"

    return StreamingResponse(generate(), media_type="text/event-stream",
                             headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})


# ══════════════════════════════════════════════════════════════
#  FEEDBACK
# ══════════════════════════════════════════════════════════════

class FeedbackRequest(BaseModel):
    query: str
    chunk_ids: List[str]
    helpful: bool


@app.post("/feedback")
async def submit_feedback(req: FeedbackRequest):
    record_feedback(req.query, req.chunk_ids, req.helpful)
    return {"status": "recorded"}


# ══════════════════════════════════════════════════════════════
#  DOCUMENT MANAGEMENT
# ══════════════════════════════════════════════════════════════

@app.get("/documents")
async def get_documents():
    return {"documents": list_all_documents()}


@app.delete("/documents/{file_id}")
async def remove_document(file_id: str):
    delete_document(file_id)
    for f in UPLOAD_DIR.glob(f"{file_id}.*"):
        f.unlink(missing_ok=True)
    if _reflection_available:
        try:
            delete_graph_for_file(file_id)
        except Exception:
            pass
    return {"status": "deleted", "file_id": file_id}


# ══════════════════════════════════════════════════════════════
#  USER PREFERENCES
# ══════════════════════════════════════════════════════════════

@app.get("/user/preferences")
async def user_preferences():
    prefs = get_user_preferences()
    return {
        "domain_interests": prefs.get("domain_interests", {}),
        "recent_queries":   [q["query"] for q in prefs.get("query_history", [])[-10:]],
    }


# ══════════════════════════════════════════════════════════════
#  HEALTH & STATS
# ══════════════════════════════════════════════════════════════

@app.get("/health")
async def health():
    import ollama as ol
    try:
        models = [m["name"] for m in ol.list()["models"]]
    except Exception as e:
        return {"status": "error", "detail": str(e)}

    docs         = list_all_documents()
    total_chunks = sum(d.get("total_chunks", 0) for d in docs)

    return {
        "status":               "ok",
        "ollama_models":        models,
        "llm_ready":            any("llama3.2" in m for m in models),
        "embed_ready":          True,
        "embed_model":          "all-MiniLM-L6-v2 (sentence-transformers)",
        "vector_store":         "FAISS",
        "total_chunks_indexed": total_chunks,
        "documents_count":      len(docs),
        "supported_formats":    sorted(SUPPORTED_EXTENSIONS),
        "features": {
            "hybrid_search":     True,
            "query_rewriting":   True,
            "multi_hop":         True,
            "deduplication":     True,
            "feedback_loop":     True,
            "versioning":        True,
            "temporal_awareness": True,
            "persistent_memory": True,
            "table_extraction":  True,
            "multi_format":      True,
        },
    }


@app.get("/stats")
async def stats():
    docs  = list_all_documents()
    prefs = get_user_preferences()

    feedback_file  = BASE_DIR / "feedback" / "retrieval_feedback.json"
    feedback_count = 0
    if feedback_file.exists():
        try:
            with open(feedback_file) as f:
                feedback_count = len(json.load(f))
        except Exception:
            pass

    return {
        "documents":        len(docs),
        "total_chunks":     sum(d.get("total_chunks", 0) for d in docs),
        "feedback_entries": feedback_count,
        "domain_interests": prefs.get("domain_interests", {}),
        "document_list":    docs,
    }


# ══════════════════════════════════════════════════════════════
#  CHAT SESSIONS
# ══════════════════════════════════════════════════════════════

SESSIONS_DIR = BASE_DIR / "sessions"
SESSIONS_DIR.mkdir(exist_ok=True)


def _session_path(session_id: str) -> Path:
    return SESSIONS_DIR / f"{session_id}.json"


def _load_session(session_id: str) -> dict:
    p = _session_path(session_id)
    return json.loads(p.read_text()) if p.exists() else {}


def _save_session(session: dict):
    _session_path(session["session_id"]).write_text(json.dumps(session, indent=2))


class SessionSaveRequest(BaseModel):
    session_id: Optional[str] = None
    title:      Optional[str] = None
    messages:   List[dict]
    file_id:    Optional[str] = None
    filename:   Optional[str] = None


@app.post("/sessions")
async def save_session(req: SessionSaveRequest):
    session_id = req.session_id or str(uuid.uuid4())
    existing   = _load_session(session_id)

    title = req.title
    if not title:
        first_user = next((m["content"] for m in req.messages if m.get("role") == "user"), "")
        title = (first_user[:52] + "...") if len(first_user) > 52 else (first_user or "Untitled Session")

    session = {
        "session_id": session_id,
        "title":      title,
        "messages":   req.messages,
        "file_id":    req.file_id,
        "filename":   req.filename,
        "created_at": existing.get("created_at", time.time()),
        "updated_at": time.time(),
        "msg_count":  len(req.messages),
    }
    _save_session(session)
    return {"session_id": session_id, "title": title}


@app.get("/sessions")
async def list_sessions():
    sessions = []
    for p in SESSIONS_DIR.glob("*.json"):
        try:
            s = json.loads(p.read_text())
            sessions.append({
                "session_id": s.get("session_id"),
                "title":      s.get("title", "Untitled"),
                "msg_count":  s.get("msg_count", 0),
                "filename":   s.get("filename", ""),
                "updated_at": s.get("updated_at", 0),
                "created_at": s.get("created_at", 0),
            })
        except Exception:
            pass
    return {"sessions": sorted(sessions, key=lambda x: x["updated_at"], reverse=True)}


@app.get("/sessions/{session_id}")
async def get_session(session_id: str):
    s = _load_session(session_id)
    if not s:
        raise HTTPException(status_code=404, detail="Session not found")
    return s


@app.patch("/sessions/{session_id}")
async def rename_session(session_id: str, body: dict):
    s = _load_session(session_id)
    if not s:
        raise HTTPException(status_code=404, detail="Session not found")
    s["title"]      = body.get("title", s["title"])
    s["updated_at"] = time.time()
    _save_session(s)
    return {"session_id": session_id, "title": s["title"]}


@app.delete("/sessions/{session_id}")
async def delete_session(session_id: str):
    p = _session_path(session_id)
    if p.exists():
        p.unlink()
    return {"status": "deleted", "session_id": session_id}


# ══════════════════════════════════════════════════════════════
#  KNOWLEDGE GRAPH / MEMORY ENDPOINTS  (unchanged)
# ══════════════════════════════════════════════════════════════

@app.get("/graph/stats")
async def graph_stats():
    if not _reflection_available:
        return {"error": "Knowledge graph module not available"}
    return get_graph_stats()


@app.get("/graph/traverse")
async def graph_traverse(query: str, max_hops: int = 3):
    if not _reflection_available:
        return {"error": "Knowledge graph module not available"}
    from knowledge_graph import traverse_graph
    return traverse_graph(query, max_hops=max_hops)


@app.get("/memory/health")
async def memory_health():
    if not _reflection_available:
        return {"error": "Memory module not available"}
    return get_memory_health()


@app.get("/memory/reflection/stats")
async def reflection_stats():
    if not _reflection_available:
        return {}
    from memory import get_reflection_stats
    return get_reflection_stats()


@app.get("/memory/intent/{session_id}")
async def intent_trajectory(session_id: str):
    if not _reflection_available:
        return {"trajectory": []}
    return {"trajectory": get_intent_trajectory(session_id)}


@app.get("/temporal/parse")
async def parse_temporal(query: str):
    if not _reflection_available:
        return {}
    from memory import parse_temporal_query
    return parse_temporal_query(query)


@app.get("/memory/pruning/candidates")
async def pruning_candidates():
    if not _reflection_available:
        return {"candidates": []}
    from memory import get_pruning_candidates
    return {"candidates": get_pruning_candidates()}


@app.post("/memory/pruning/run")
async def run_pruning_endpoint(dry_run: bool = True):
    if not _reflection_available:
        return {"error": "Memory module not available"}
    return run_pruning(dry_run=dry_run)


@app.get("/health/full")
async def health_full():
    import ollama as ol
    try:
        models = [m["name"] for m in ol.list()["models"]]
    except Exception as e:
        return {"status": "error", "detail": str(e)}

    docs         = list_all_documents()
    total_chunks = sum(d.get("total_chunks", 0) for d in docs)
    graph_info   = {}
    mem_health   = {}
    if _reflection_available:
        try:
            graph_info = get_graph_stats()
            mem_health = get_memory_health()
        except Exception:
            pass

    return {
        "status":          "ok",
        "models":          models,
        "llm_ready":       any("llama3.2" in m for m in models),
        "embed_ready":     True,
        "embed_model":     "all-MiniLM-L6-v2 (sentence-transformers, local)",
        "vector_store":    "FAISS",
        "total_chunks":    total_chunks,
        "documents":       len(docs),
        "supported_formats": sorted(SUPPORTED_EXTENSIONS),
        "knowledge_graph": graph_info,
        "memory_health":   mem_health,
        "features": {
            "hybrid_search":      True,
            "query_rewriting":    True,
            "multi_hop":          True,
            "deduplication":      True,
            "feedback_loop":      True,
            "versioning":         True,
            "temporal_awareness": True,
            "persistent_memory":  True,
            "knowledge_graph":    _reflection_available,
            "reflection_layer":   _reflection_available,
            "intent_trajectory":  _reflection_available,
            "decay_pruning":      _reflection_available,
            "table_extraction":   True,
            "multi_format":       True,
        },
    }

# ══════════════════════════════════════════════════════════════
#  TRACES  — End-to-end pipeline logging endpoints
# ══════════════════════════════════════════════════════════════

@app.get("/traces")
async def list_traces(limit: int = 50):
    """Return the most recent trace summaries (lightweight)."""
    return {"traces": get_recent_traces(limit=limit)}


@app.get("/traces/stats")
async def trace_stats():
    """Aggregate metrics across all recorded traces."""
    return get_trace_stats()


@app.get("/traces/{trace_id}")
async def get_trace(trace_id: str):
    """Return the full trace record for a single request."""
    rec = get_trace_by_id(trace_id)
    if not rec:
        raise HTTPException(status_code=404, detail="Trace not found")
    return rec

Overwriting /content/singularity/main.py


### 5g — app.py
*Streamlit frontend — Full UI with connection pooling*

In [31]:
%%writefile /content/singularity/app.py
"""
Singularity — Enterprise RAG Interface
Persistent chat sessions. Event horizon aesthetic.
"""

import streamlit as st
import json
import time
from datetime import datetime

API_BASE = "http://localhost:8000"

# ── Persistent HTTP session with connection pooling ──────────────
# Reusing one session avoids TCP handshake overhead on every call
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

_http = requests.Session()
_retry = Retry(
    total=3,
    backoff_factor=0.3,
    status_forcelist=[500, 502, 503, 504],
    allowed_methods=["GET", "POST", "DELETE", "PATCH"],
)
_http.mount("http://", HTTPAdapter(max_retries=_retry, pool_connections=4, pool_maxsize=8))

st.set_page_config(
    page_title="Singularity · RAG",
    page_icon=None,
    layout="wide",
    initial_sidebar_state="expanded",
)

# ═══════════════════════════════════════════════════════════════
#  CSS  (unchanged from original)
# ═══════════════════════════════════════════════════════════════

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;500;600;700;800&family=Syne+Mono&display=swap');

:root {
    --void:        #00000a;
    --abyss:       #03030f;
    --deep:        #07071a;
    --horizon:     #0d0d24;
    --shell:       #12122e;
    --rim:         #1a1a3e;
    --corona:      #c8a96e;
    --corona-dim:  #8a6e3a;
    --corona-glow: #c8a96e22;
    --plasma:      #6e8dc8;
    --plasma-dim:  #3a5280;
    --plasma-glow: #6e8dc811;
    --dust:        #4a4a6a;
    --mist:        #2a2a4a;
    --ghost:       #6a6a9a;
    --star:        #d4d4f0;
    --nebula:      #a0a0c8;
    --white:       #f0f0fc;
    --success:     #6ec8a0;
    --warn:        #c8b46e;
    --danger:      #c86e6e;
}

*, *::before, *::after { box-sizing: border-box; }

html, body, [class*="css"], .stApp {
    font-family: 'Syne', sans-serif;
    background: var(--void) !important;
    color: var(--nebula);
}

.stApp::after {
    content: '';
    position: fixed;
    top: 0; left: 0; right: 0;
    height: 2px;
    background: linear-gradient(90deg,
        transparent 0%, var(--plasma-dim) 20%,
        var(--corona) 50%, var(--plasma-dim) 80%, transparent 100%);
    z-index: 9999;
    animation: horizonPulse 6s ease-in-out infinite;
}
@keyframes horizonPulse {
    0%, 100% { opacity: 0.35; }
    50%       { opacity: 0.9; }
}

[data-testid="stSidebar"] {
    background: var(--abyss) !important;
    border-right: 1px solid var(--mist) !important;
}
[data-testid="stSidebar"] > div { padding-top: 0 !important; }

::-webkit-scrollbar { width: 4px; }
::-webkit-scrollbar-track { background: var(--void); }
::-webkit-scrollbar-thumb { background: var(--mist); border-radius: 2px; }

.stButton > button {
    font-family: 'Syne', sans-serif !important;
    font-size: 0.72rem !important;
    font-weight: 600 !important;
    letter-spacing: 0.1em !important;
    text-transform: uppercase !important;
    background: transparent !important;
    color: var(--ghost) !important;
    border: 1px solid var(--mist) !important;
    border-radius: 2px !important;
    padding: 6px 14px !important;
    transition: all 0.22s !important;
}
.stButton > button:hover {
    color: var(--corona) !important;
    border-color: var(--corona-dim) !important;
    box-shadow: 0 0 16px var(--corona-glow), inset 0 0 10px var(--corona-glow) !important;
}

.stFormSubmitButton > button {
    background: var(--shell) !important;
    color: var(--corona) !important;
    border: 1px solid var(--corona-dim) !important;
    font-family: 'Syne', sans-serif !important;
    font-size: 0.72rem !important;
    font-weight: 700 !important;
    letter-spacing: 0.14em !important;
    text-transform: uppercase !important;
    border-radius: 2px !important;
    transition: all 0.22s !important;
}
.stFormSubmitButton > button:hover {
    background: var(--rim) !important;
    box-shadow: 0 0 24px var(--corona-glow) !important;
}

[data-testid="stFileUploader"] {
    background: var(--deep) !important;
    border: 1px dashed var(--mist) !important;
    border-radius: 3px !important;
}

.stTextInput > div > div > input {
    background: var(--deep) !important;
    border: 1px solid var(--mist) !important;
    border-radius: 2px !important;
    color: var(--star) !important;
    font-family: 'Syne', sans-serif !important;
    font-size: 0.91rem !important;
    padding: 10px 16px !important;
    transition: all 0.22s !important;
    caret-color: var(--corona) !important;
}
.stTextInput > div > div > input:focus {
    border-color: var(--corona-dim) !important;
    box-shadow: 0 0 0 1px var(--corona-dim), 0 0 16px var(--corona-glow) !important;
    outline: none !important;
}
.stTextInput > div > div > input::placeholder { color: var(--dust) !important; }

.stToggle > label { color: var(--ghost) !important; font-size: 0.8rem !important; }

[data-testid="stMetric"] {
    background: var(--deep);
    border: 1px solid var(--mist);
    border-radius: 3px;
    padding: 14px !important;
}
[data-testid="stMetricLabel"] { color: var(--dust) !important; font-size: 0.65rem !important; text-transform: uppercase; letter-spacing: 0.1em; }
[data-testid="stMetricValue"] { color: var(--corona) !important; font-family: 'Syne Mono', monospace !important; font-size: 1.5rem !important; }

hr { border: none !important; border-top: 1px solid var(--mist) !important; margin: 16px 0 !important; }

.stSpinner > div { border-top-color: var(--corona) !important; }

.streamlit-expanderHeader {
    background: var(--deep) !important;
    border: 1px solid var(--mist) !important;
    border-radius: 3px !important;
    color: var(--ghost) !important;
    font-family: 'Syne', sans-serif !important;
    font-size: 0.75rem !important;
    text-transform: uppercase !important;
    letter-spacing: 0.1em !important;
}

.sg-wordmark {
    padding: 28px 0 16px 0;
    border-bottom: 1px solid var(--mist);
    margin-bottom: 0;
}
.sg-wordmark-name {
    font-size: 1rem;
    font-weight: 800;
    letter-spacing: 0.2em;
    text-transform: uppercase;
    color: var(--white);
}
.sg-wordmark-sub {
    font-family: 'Syne Mono', monospace;
    font-size: 0.58rem;
    letter-spacing: 0.2em;
    text-transform: uppercase;
    color: var(--dust);
    margin-top: 4px;
}

.sg-label {
    font-size: 0.58rem;
    font-weight: 700;
    letter-spacing: 0.24em;
    text-transform: uppercase;
    color: var(--dust);
    margin: 16px 0 10px 0;
    display: flex;
    align-items: center;
    gap: 8px;
}
.sg-label::after { content: ''; flex: 1; height: 1px; background: var(--mist); }

.sg-session {
    background: var(--deep);
    border: 1px solid var(--mist);
    border-left: 2px solid transparent;
    border-radius: 2px;
    padding: 10px 13px;
    margin-bottom: 4px;
    cursor: pointer;
    transition: all 0.2s;
    position: relative;
}
.sg-session:hover { border-color: var(--ghost); }
.sg-session.active {
    border-left-color: var(--corona);
    background: var(--horizon);
    box-shadow: inset 0 0 20px var(--corona-glow);
}
.sg-session-title {
    font-size: 0.8rem;
    font-weight: 600;
    color: var(--star);
    white-space: nowrap;
    overflow: hidden;
    text-overflow: ellipsis;
    max-width: 180px;
}
.sg-session.active .sg-session-title { color: var(--corona); }
.sg-session-meta {
    font-family: 'Syne Mono', monospace;
    font-size: 0.6rem;
    color: var(--dust);
    margin-top: 3px;
    letter-spacing: 0.04em;
}

.sg-doc {
    background: var(--deep);
    border: 1px solid var(--mist);
    border-left: 2px solid transparent;
    border-radius: 2px;
    padding: 9px 12px;
    margin-bottom: 4px;
    transition: all 0.2s;
}
.sg-doc.active { border-left-color: var(--plasma); background: var(--horizon); }
.sg-doc-name { font-size: 0.78rem; font-weight: 600; color: var(--star); white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
.sg-doc.active .sg-doc-name { color: var(--plasma); }
.sg-doc-meta { font-family: 'Syne Mono', monospace; font-size: 0.6rem; color: var(--dust); margin-top: 3px; }

.sg-badge {
    display: inline-block;
    font-family: 'Syne Mono', monospace;
    font-size: 0.6rem;
    letter-spacing: 0.05em;
    border-radius: 2px;
    padding: 2px 7px;
    margin: 2px;
    border: 1px solid var(--mist);
    color: var(--ghost);
    background: var(--deep);
}
.sg-badge.gold  { border-color: var(--corona-dim); color: var(--corona); background: var(--corona-glow); }
.sg-badge.blue  { border-color: var(--plasma-dim); color: var(--plasma); background: var(--plasma-glow); }
.sg-badge.green { border-color: #3a7a5a; color: var(--success); background: #3a7a5a11; }

.sg-tags { display: flex; flex-wrap: wrap; gap: 4px; margin: 12px 0 0 0; }
.sg-tag {
    font-family: 'Syne Mono', monospace;
    font-size: 0.56rem;
    letter-spacing: 0.08em;
    text-transform: uppercase;
    color: var(--plasma-dim);
    border: 1px solid var(--plasma-dim);
    border-radius: 2px;
    padding: 2px 6px;
    opacity: 0.6;
}

.sg-header {
    padding: 32px 0 22px 0;
    border-bottom: 1px solid var(--mist);
    margin-bottom: 28px;
    position: relative;
}
.sg-header::after {
    content: '';
    position: absolute;
    bottom: -1px; left: 0;
    width: 100px; height: 1px;
    background: linear-gradient(90deg, var(--corona), transparent);
}
.sg-header-eyebrow {
    font-family: 'Syne Mono', monospace;
    font-size: 0.62rem;
    letter-spacing: 0.22em;
    text-transform: uppercase;
    color: var(--dust);
    margin-bottom: 8px;
}
.sg-header-title {
    font-size: clamp(1.3rem, 3vw, 1.9rem);
    font-weight: 800;
    letter-spacing: -0.03em;
    color: var(--white);
    line-height: 1.1;
}
.sg-header-title .hl {
    background: linear-gradient(135deg, var(--corona) 0%, var(--plasma) 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
}
.sg-header-stack { display: flex; gap: 6px; margin-top: 14px; flex-wrap: wrap; }
.sg-stack-pill {
    font-family: 'Syne Mono', monospace;
    font-size: 0.58rem;
    color: var(--dust);
    letter-spacing: 0.1em;
    text-transform: uppercase;
    padding: 3px 10px;
    border: 1px solid var(--mist);
    border-radius: 2px;
}

.sg-msg {
    padding: 18px 22px;
    border-radius: 2px;
    margin-bottom: 12px;
    line-height: 1.72;
    font-size: 0.9rem;
    animation: fadeUp 0.26s ease-out;
}
@keyframes fadeUp {
    from { opacity: 0; transform: translateY(5px); }
    to   { opacity: 1; transform: translateY(0); }
}
.sg-msg-user {
    background: var(--horizon);
    border: 1px solid var(--rim);
    border-left: 2px solid var(--plasma-dim);
    color: var(--nebula);
}
.sg-msg-ai {
    background: var(--deep);
    border: 1px solid var(--mist);
    border-left: 2px solid var(--corona-dim);
    color: var(--star);
}
.sg-msg-tag {
    font-family: 'Syne Mono', monospace;
    font-size: 0.58rem;
    letter-spacing: 0.2em;
    text-transform: uppercase;
    margin-bottom: 10px;
    padding-bottom: 8px;
    border-bottom: 1px solid;
}
.sg-msg-user .sg-msg-tag { color: var(--plasma-dim); border-color: var(--plasma-dim); opacity: 0.7; }
.sg-msg-ai   .sg-msg-tag { color: var(--corona-dim); border-color: var(--corona-dim); opacity: 0.8; }

.sg-rewrite {
    background: var(--deep);
    border: 1px solid var(--warn);
    border-left: 2px solid var(--warn);
    border-radius: 2px;
    padding: 8px 14px;
    margin-bottom: 6px;
    font-family: 'Syne Mono', monospace;
    font-size: 0.7rem;
    color: var(--warn);
    opacity: 0.85;
}
.sg-rewrite-ey { font-size: 0.56rem; letter-spacing: 0.2em; text-transform: uppercase; opacity: 0.6; margin-bottom: 3px; }

.sg-welcome {
    border: 1px solid var(--mist);
    border-radius: 3px;
    padding: 52px 40px;
    text-align: center;
    background: var(--deep);
    position: relative;
    overflow: hidden;
}
.sg-welcome::before {
    content: '';
    position: absolute;
    top: 50%; left: 50%;
    transform: translate(-50%, -50%);
    width: 400px; height: 400px;
    background: radial-gradient(circle, var(--corona-glow) 0%, transparent 65%);
    pointer-events: none;
    animation: breathe 5s ease-in-out infinite;
}
@keyframes breathe {
    0%, 100% { transform: translate(-50%,-50%) scale(1); opacity: 0.5; }
    50%       { transform: translate(-50%,-50%) scale(1.2); opacity: 1; }
}
.sg-welcome-title { font-size: 1.1rem; font-weight: 700; color: var(--white); margin-bottom: 10px; position: relative; }
.sg-welcome-body  { font-size: 0.82rem; color: var(--ghost); line-height: 1.65; max-width: 380px; margin: 0 auto; position: relative; }

.sg-noingest {
    border: 1px dashed var(--mist);
    border-radius: 3px;
    padding: 28px;
    text-align: center;
}
.sg-noingest-title { font-size: 0.78rem; font-weight: 600; letter-spacing: 0.14em; text-transform: uppercase; color: var(--ghost); margin-bottom: 6px; }
.sg-noingest-sub   { font-family: 'Syne Mono', monospace; font-size: 0.63rem; color: var(--dust); letter-spacing: 0.06em; }

.sg-summary-panel {
    background: var(--deep);
    border: 1px solid var(--mist);
    border-top: 1px solid var(--corona-dim);
    border-radius: 3px;
    padding: 26px 28px;
    margin-bottom: 20px;
    position: relative; overflow: hidden;
}
.sg-summary-panel::before {
    content: '';
    position: absolute; top: 0; right: 0;
    width: 220px; height: 220px;
    background: radial-gradient(circle at top right, var(--corona-glow), transparent 70%);
    pointer-events: none;
}
.sg-summary-ey {
    font-family: 'Syne Mono', monospace;
    font-size: 0.6rem; letter-spacing: 0.2em; text-transform: uppercase;
    color: var(--corona-dim); margin-bottom: 18px;
    display: flex; align-items: center; gap: 10px;
}
.sg-summary-ey::after { content: ''; flex: 1; height: 1px; background: linear-gradient(90deg, var(--corona-dim), transparent); }

.sg-stats-wrap { background: var(--deep); border: 1px solid var(--mist); border-radius: 3px; padding: 22px; margin-bottom: 18px; }

.sg-footer { margin-top: 52px; padding-top: 16px; border-top: 1px solid var(--mist); display: flex; justify-content: space-between; font-family: 'Syne Mono', monospace; font-size: 0.58rem; letter-spacing: 0.12em; text-transform: uppercase; color: var(--mist); }

#MainMenu, header, footer { visibility: hidden; }
[data-testid="stDecoration"] { display: none; }
.block-container { padding-top: 0 !important; max-width: 100% !important; }

/* ── Message body typography ── */
.sg-msg-body { line-height: 1.7; }
.sg-msg-body .sg-p { margin: 0 0 0.55em 0; color: var(--star); }
.sg-msg-body .sg-h2 { font-size: 1.05rem; font-weight: 700; color: var(--corona); margin: 1em 0 0.4em; letter-spacing: 0.02em; }
.sg-msg-body .sg-h3 { font-size: 0.95rem; font-weight: 700; color: var(--plasma); margin: 0.9em 0 0.35em; }
.sg-msg-body .sg-h4 { font-size: 0.88rem; font-weight: 600; color: var(--ghost); margin: 0.8em 0 0.3em; }
.sg-msg-body .sg-ul, .sg-msg-body .sg-ol { margin: 0.3em 0 0.6em 1.4em; padding: 0; }
.sg-msg-body li { margin-bottom: 0.3em; color: var(--star); }
.sg-msg-body .sg-hr { border: none; border-top: 1px solid var(--mist); margin: 0.8em 0; }
.sg-msg-body .sg-bold { color: var(--corona); font-weight: 700; }
.sg-msg-body .sg-code { font-family: "Syne Mono", monospace; font-size: 0.82em; background: rgba(255,255,255,0.06); padding: 1px 5px; border-radius: 3px; color: var(--plasma); }
.sg-msg-body strong { color: var(--corona); }

/* ── Number / percentage highlighting ── */
.sg-num {
    color: var(--corona);
    font-weight: 700;
    font-family: "Syne Mono", monospace;
    font-size: 0.95em;
    background: rgba(255, 210, 80, 0.08);
    border: 1px solid rgba(255, 210, 80, 0.20);
    border-radius: 3px;
    padding: 0 4px;
    letter-spacing: 0.01em;
}
.sg-num.sg-pct {
    color: #7ed6a0;
    background: rgba(126, 214, 160, 0.08);
    border-color: rgba(126, 214, 160, 0.22);
}

/* ── Table rendering ── */
.sg-table {
    width: 100%;
    border-collapse: collapse;
    font-size: 0.82rem;
    margin: 0.7em 0 1em;
    font-family: "Syne Mono", monospace;
}
.sg-table th {
    background: rgba(255,255,255,0.04);
    color: var(--corona);
    font-weight: 700;
    padding: 6px 10px;
    border: 1px solid var(--mist);
    text-align: left;
    letter-spacing: 0.05em;
    text-transform: uppercase;
    font-size: 0.72rem;
}
.sg-table td {
    padding: 5px 10px;
    border: 1px solid var(--mist);
    color: var(--ghost);
    vertical-align: top;
}
.sg-table tr:nth-child(even) td { background: rgba(255,255,255,0.015); }
.sg-table tr:hover td { background: rgba(255,255,255,0.03); }

/* ── Sources section ── */
.sg-sources {
    margin-top: 1.1em;
    padding-top: 0.6em;
    border-top: 1px solid rgba(255,255,255,0.08);
    font-size: 0.78rem;
    color: var(--dust);
    font-family: "Syne Mono", monospace;
}
.sg-sources-label {
    color: var(--corona-dim);
    font-weight: 700;
    letter-spacing: 0.1em;
    text-transform: uppercase;
    font-size: 0.65rem;
    margin-right: 0.5em;
}
.sg-sources-label::after { content: " ·"; }
</style>
""", unsafe_allow_html=True)


# ═══════════════════════════════════════════════
#  SESSION STATE
# ═══════════════════════════════════════════════

def init_state():
    defaults = {
        "messages":           [],
        "current_session_id": None,
        "current_session_title": None,
        "selected_file_id":   None,
        "selected_filename":  None,
        "selected_file_ids":  [],    # multi-source selection
        "documents":          [],
        "chat_sessions":      [],
        "show_stats":         False,
        "show_traces":        False,
        "selected_trace_id":  None,
        "use_query_rewrite":  False,
        "use_multihop":       True,
        "show_summary":       False,
        "renaming_session":   None,
        "_sessions_loaded":   False,  # only fetch once per page load
        "_docs_loaded":       False,  # only fetch once per page load
    }
    for k, v in defaults.items():
        if k not in st.session_state:
            st.session_state[k] = v

init_state()


# ═══════════════════════════════════════════════
#  API HELPERS
# ═══════════════════════════════════════════════

def api_get(path: str) -> dict:
    try:
        r = _http.get(f"{API_BASE}{path}", timeout=10)
        return r.json()
    except Exception:
        return {}


def api_post(path: str, **kwargs) -> dict:
    try:
        r = _http.post(f"{API_BASE}{path}", timeout=30, **kwargs)
        return r.json()
    except Exception:
        return {}


def fetch_sessions():
    data = api_get("/sessions")
    st.session_state.chat_sessions = data.get("sessions", [])
    st.session_state._sessions_loaded = True


def fetch_documents():
    data = api_get("/documents")
    st.session_state.documents = data.get("documents", [])
    st.session_state._docs_loaded = True


def auto_save_session():
    if not st.session_state.messages:
        return
    payload = {
        "session_id": st.session_state.current_session_id,
        "title":      st.session_state.current_session_title,
        "messages":   st.session_state.messages,
        "file_id":    st.session_state.selected_file_id,
        "filename":   st.session_state.selected_filename,
    }
    try:
        r = _http.post(f"{API_BASE}/sessions", json=payload, timeout=10)
        data = r.json()
        st.session_state.current_session_id    = data.get("session_id", st.session_state.current_session_id)
        st.session_state.current_session_title = data.get("title", st.session_state.current_session_title)
    except:
        pass


def load_session(session_id: str):
    try:
        r = _http.get(f"{API_BASE}/sessions/{session_id}", timeout=10)
        s = r.json()
        st.session_state.messages              = s.get("messages", [])
        st.session_state.current_session_id    = s.get("session_id")
        st.session_state.current_session_title = s.get("title", "Session")
        st.session_state.selected_file_id      = s.get("file_id")
        st.session_state.selected_filename     = s.get("filename")
        st.session_state.show_summary          = False
    except:
        pass


def new_session():
    st.session_state.messages              = []
    st.session_state.current_session_id    = None
    st.session_state.current_session_title = None
    st.session_state.show_summary          = False


def delete_session(session_id: str):
    try:
        _http.delete(f"{API_BASE}/sessions/{session_id}", timeout=5)
    except:
        pass
    if st.session_state.current_session_id == session_id:
        new_session()
    fetch_sessions()


import re as _re
import html as _html

def _highlight_numbers(text: str) -> str:
    """Wrap standalone numbers/percentages/scores in a highlight span."""
    # Match: percentages like 4%, numbers like 83.05, values like 0.741
    text = _re.sub(
        r'(?<![\w/])(-?\d+\.?\d*\s*%)',
        r'<span class="sg-num sg-pct">\1</span>',
        text
    )
    text = _re.sub(
        r'(?<![\w/\d])(-?\d{1,3}\.\d+)(?![\d%])',
        r'<span class="sg-num">\1</span>',
        text
    )
    return text

def _clean_llm_output(text: str) -> str:
    """Strip internal retrieval artifact lines that the LLM sometimes echoes back."""
    import re
    lines = text.split("\n")
    cleaned = []
    for line in lines:
        s = line.strip()
        # Skip internal retrieval labels
        if re.match(r'\[(?:DIRECT-SCAN|excerpt|TABLE|IMAGE)[^\]]*\]', s):
            continue
        # Skip raw quoted context strings (lines that start/end with " and look like source text)
        if s.startswith('"') and s.endswith('"') and len(s) > 60 and "http" not in s:
            # Keep short quotes that look like cited facts, skip long raw dumps
            if len(s) > 120:
                continue
        cleaned.append(line)
    return "\n".join(cleaned)


def _render_md(text: str) -> str:
    """Convert markdown to styled HTML with number highlighting.
    Only renders pipe tables when there are 2+ data rows AND the header has 2+ real columns.
    Short pipe lines (like single-cell metadata) are rendered as plain text.
    """
    import re
    text = _clean_llm_output(text)
    lines = text.split("\n")
    out = []
    in_ul  = False
    in_ol  = False
    in_table = False
    table_rows = []

    def close_list():
        nonlocal in_ul, in_ol
        if in_ul:
            out.append("</ul>")
            in_ul = False
        if in_ol:
            out.append("</ol>")
            in_ol = False

    def flush_table():
        nonlocal in_table, table_rows
        if not in_table or not table_rows:
            in_table = False
            table_rows.clear()
            return
        # Only render as a real table if ≥2 columns AND ≥1 data row
        header = table_rows[0]
        # Find separator row (all dashes)
        sep_idx = next((i for i,r in enumerate(table_rows) if all(re.match(r'^-+$', c.strip('-').strip() or '-') for c in r)), None)
        data_rows = table_rows[sep_idx+1:] if sep_idx is not None else table_rows[1:]

        if len(header) >= 2 and len(data_rows) >= 1:
            html = '<table class="sg-table"><thead>'
            html += "<tr>" + "".join(f"<th>{_inline_fmt(c)}</th>" for c in header) + "</tr></thead><tbody>"
            for row in data_rows:
                # Pad or trim row to header width
                while len(row) < len(header): row.append("")
                html += "<tr>" + "".join(f"<td>{_inline_fmt(c)}</td>" for c in row[:len(header)]) + "</tr>"
            html += "</tbody></table>"
            out.append(html)
        else:
            # Render as plain text instead
            for row in table_rows:
                out.append(f'<p class="sg-p">{_inline_fmt(" | ".join(row))}</p>')

        in_table = False
        table_rows.clear()

    for line in lines:
        stripped = line.strip()

        # Detect pipe table lines
        is_pipe = stripped.startswith("|") and stripped.endswith("|")
        # Separator rows like |---|---| should not trigger table on their own
        is_sep  = is_pipe and all(re.match(r'^[-:]+$', c.strip()) for c in stripped[1:-1].split("|") if c.strip())

        if is_pipe:
            close_list()
            if not in_table:
                in_table = True
            cells = [c.strip() for c in stripped[1:-1].split("|")]
            if cells:
                table_rows.append(cells)
            continue
        elif in_table:
            flush_table()

        # Headings
        if stripped.startswith("### "):
            close_list()
            out.append(f'<h4 class="sg-h4">{_inline_fmt(stripped[4:])}</h4>')
        elif stripped.startswith("## "):
            close_list()
            out.append(f'<h3 class="sg-h3">{_inline_fmt(stripped[3:])}</h3>')
        elif stripped.startswith("# "):
            close_list()
            out.append(f'<h2 class="sg-h2">{_inline_fmt(stripped[2:])}</h2>')
        # Bullet list
        elif stripped.startswith("* ") or stripped.startswith("- "):
            if in_ol: close_list()
            if not in_ul:
                out.append('<ul class="sg-ul">')
                in_ul = True
            out.append(f'<li>{_inline_fmt(stripped[2:])}</li>')
        # Numbered list
        elif re.match(r"^\d+\.\s", stripped):
            if in_ul: close_list()
            if not in_ol:
                out.append('<ol class="sg-ol">')
                in_ol = True
            content = re.sub(r"^\d+\.\s+", "", stripped)
            out.append(f'<li>{_inline_fmt(content)}</li>')
        # Sources line — render styled
        elif stripped.lower().startswith("sources:"):
            close_list()
            rest = stripped[8:].strip()
            out.append(f'<div class="sg-sources"><span class="sg-sources-label">Sources</span>{_inline_fmt(rest)}</div>')
        # Empty / blank
        elif stripped == "" or stripped == "▌":
            close_list()
            if out and out[-1] != "<br>":
                out.append("<br>")
        # Horizontal rule
        elif stripped in ("---", "***", "___"):
            close_list()
            out.append('<hr class="sg-hr">')
        else:
            close_list()
            out.append(f'<p class="sg-p">{_inline_fmt(stripped)}</p>')

    close_list()
    flush_table()
    return "\n".join(out)

def _inline_fmt(text: str) -> str:
    """Apply inline markdown: bold, italic, code, links, then number highlighting."""
    import re, html
    # Escape HTML first (except we'll add our own tags)
    # Bold **text** or __text__
    text = re.sub(r'\*\*(.+?)\*\*', r'<strong class="sg-bold">\1</strong>', text)
    text = re.sub(r'__(.+?)__', r'<strong class="sg-bold">\1</strong>', text)
    # Italic *text* or _text_
    text = re.sub(r'\*(.+?)\*', r'<em>\1</em>', text)
    text = re.sub(r'_([^_]+)_', r'<em>\1</em>', text)
    # Inline code `code`
    text = re.sub(r'`([^`]+)`', r'<code class="sg-code">\1</code>', text)
    # Highlight numbers/percentages
    text = _highlight_numbers(text)
    return text


def stream_response(endpoint: str, payload: dict):
    full_text = ""
    meta = {}
    try:
        with _http.post(f"{API_BASE}{endpoint}", json=payload, stream=True, timeout=120) as resp:
            placeholder = st.empty()
            for line in resp.iter_lines():
                if not line:
                    continue
                line = line.decode("utf-8")
                if line.startswith("data: "):
                    data = line[6:]
                    if data == "[DONE]":
                        break
                    if data.startswith("[ERROR]"):
                        st.error(data)
                        break
                    if data.startswith('{"type":"meta"'):
                        try:
                            meta = json.loads(data)
                        except:
                            pass
                        continue
                    if data.startswith('{"type":'):
                        try:
                            meta.update(json.loads(data))
                        except:
                            pass
                        continue
                    token = data.replace('\\n', '\n').replace('\\"', '"')
                    full_text += token
                    # Render markdown inside the styled container
                    placeholder.markdown(
                        f'<div class="sg-msg sg-msg-ai">'
                        f'<div class="sg-msg-tag">Singularity</div>'
                        f'<div class="sg-msg-body">' + _render_md(full_text + "▌") + '</div></div>',
                        unsafe_allow_html=True
                    )
            placeholder.empty()
    except Exception as e:
        st.error(f"Stream error: {e}")
    return full_text, meta


def fmt_time(ts: float) -> str:
    if not ts:
        return ""
    dt = datetime.fromtimestamp(ts)
    now = datetime.now()
    diff = now - dt
    if diff.days == 0:
        if diff.seconds < 3600:
            return f"{diff.seconds // 60}m ago"
        return f"{diff.seconds // 3600}h ago"
    elif diff.days == 1:
        return "Yesterday"
    elif diff.days < 7:
        return f"{diff.days}d ago"
    return dt.strftime("%b %d")


# ═══════════════════════════════════════════════
#  SIDEBAR
# ═══════════════════════════════════════════════

with st.sidebar:

    st.markdown("""
    <div class="sg-wordmark">
        <div class="sg-wordmark-name">Singularity</div>
        <div class="sg-wordmark-sub">Enterprise Knowledge Engine</div>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("<div style='height:14px'></div>", unsafe_allow_html=True)
    if st.button("+ New Session", use_container_width=True):
        new_session()
        st.rerun()

    st.markdown('<div class="sg-label">Chat Sessions</div>', unsafe_allow_html=True)

    if not st.session_state._sessions_loaded:
        fetch_sessions()

    if st.button("Refresh Sessions", use_container_width=True):
        fetch_sessions()
        st.rerun()

    sessions = st.session_state.chat_sessions
    if not sessions:
        st.markdown(
            '<div style="font-family:\'Syne Mono\',monospace;font-size:0.65rem;'
            'color:var(--dust);padding:8px 0">No saved sessions yet</div>',
            unsafe_allow_html=True
        )
    else:
        for s in sessions:
            sid      = s["session_id"]
            stitle   = s.get("title", "Untitled")
            smeta    = s.get("msg_count", 0)
            supdated = fmt_time(s.get("updated_at", 0))
            sfname   = s.get("filename", "")
            is_active = st.session_state.current_session_id == sid
            acls = "active" if is_active else ""

            stitle_disp = stitle if len(stitle) <= 32 else stitle[:29] + "..."
            doc_hint = f" · {sfname[:18]}..." if sfname and len(sfname) > 18 else (f" · {sfname}" if sfname else "")

            st.markdown(f"""
            <div class="sg-session {acls}">
                <div class="sg-session-title">{stitle_disp}</div>
                <div class="sg-session-meta">{smeta} msgs &nbsp;·&nbsp; {supdated}{doc_hint}</div>
            </div>""", unsafe_allow_html=True)

            sc1, sc2, sc3 = st.columns([3, 1, 1])
            with sc1:
                if st.button("Open", key=f"open_{sid}", use_container_width=True):
                    load_session(sid)
                    fetch_sessions()
                    st.rerun()
            with sc2:
                if st.button("Del", key=f"del_s_{sid}", use_container_width=True):
                    delete_session(sid)
                    st.rerun()
            with sc3:
                if st.button("...", key=f"more_{sid}", use_container_width=True):
                    st.session_state.renaming_session = sid

            if st.session_state.renaming_session == sid:
                new_title = st.text_input(
                    "Rename", value=stitle, key=f"rename_inp_{sid}",
                    label_visibility="collapsed"
                )
                rc1, rc2 = st.columns(2)
                with rc1:
                    if st.button("Save", key=f"rename_save_{sid}", use_container_width=True):
                        try:
                            _http.patch(f"{API_BASE}/sessions/{sid}",
                                           json={"title": new_title}, timeout=5)
                        except:
                            pass
                        st.session_state.renaming_session = None
                        if st.session_state.current_session_id == sid:
                            st.session_state.current_session_title = new_title
                        fetch_sessions()
                        st.rerun()
                with rc2:
                    if st.button("Cancel", key=f"rename_cancel_{sid}", use_container_width=True):
                        st.session_state.renaming_session = None
                        st.rerun()

    st.markdown("<div style='height:6px'></div>", unsafe_allow_html=True)
    st.markdown("<hr>", unsafe_allow_html=True)

    st.markdown('<div class="sg-label">Knowledge Sources</div>', unsafe_allow_html=True)

    uploaded_file = st.file_uploader(
        "PDF · DOCX · TXT · CSV · XLSX · PPTX · MD · HTML",
        type=["pdf", "docx", "txt", "csv", "xlsx", "xls", "pptx", "md", "html", "htm"],
        label_visibility="collapsed",
    )
    if uploaded_file:
        if st.button("Ingest", use_container_width=True):
            with st.spinner("Parsing — Chunking — Embedding"):
                files = {"file": (uploaded_file.name, uploaded_file.getvalue(), uploaded_file.type)}
                try:
                    r = _http.post(f"{API_BASE}/upload", files=files, timeout=180)
                    result = r.json()
                    if "file_id" in result:
                        c1, c2, c3 = st.columns(3)
                        c1.metric("Chunks",  result.get("chunks_created", "—"))
                        c2.metric("Images",  result.get("images_extracted", "—"))
                        c3.metric("Version", result.get("version", "—"))
                        st.session_state.selected_file_id  = result["file_id"]
                        st.session_state.selected_filename = result.get("filename", uploaded_file.name)
                        fetch_documents()
                    else:
                        st.error(result.get("detail", str(result)))
                except Exception as e:
                    st.error(str(e))

    if not st.session_state._docs_loaded:
        fetch_documents()

    cr, ca = st.columns([2, 1])
    with cr:
        if st.button("Refresh", key="doc_refresh", use_container_width=True):
            fetch_documents()
    with ca:
        if st.button("All", key="doc_all", use_container_width=True):
            st.session_state.selected_file_id  = None
            st.session_state.selected_filename = None
            st.session_state.selected_file_ids = []
            st.rerun()

    # ── Source selection mode ────────────────────────────────
    sel_mode = st.radio(
        "Source mode",
        ["All sources", "Single source", "Custom selection"],
        index=0 if not st.session_state.selected_file_ids and not st.session_state.selected_file_id
              else (1 if st.session_state.selected_file_id and not st.session_state.selected_file_ids else 2),
        label_visibility="collapsed",
        horizontal=True,
    )

    if sel_mode == "All sources":
        st.session_state.selected_file_id  = None
        st.session_state.selected_filename = None
        st.session_state.selected_file_ids = []

    for doc in st.session_state.documents:
        fid    = doc["file_id"]
        fname  = doc.get("filename", "Unknown")
        chunks = doc.get("total_chunks", 0)
        ver    = doc.get("version", 1)
        dname  = fname if len(fname) <= 26 else fname[:23] + "..."

        if sel_mode == "Custom selection":
            # Checkbox per document
            checked = fid in st.session_state.selected_file_ids
            new_val = st.checkbox(
                f"{dname}  ·  {chunks}c v{ver}",
                value=checked,
                key=f"chk_{fid}",
            )
            if new_val and fid not in st.session_state.selected_file_ids:
                st.session_state.selected_file_ids.append(fid)
                st.session_state.selected_file_id  = None
                st.session_state.selected_filename = None
            elif not new_val and fid in st.session_state.selected_file_ids:
                st.session_state.selected_file_ids.remove(fid)

        else:
            # Original single-select card
            is_active = (sel_mode == "Single source" and st.session_state.selected_file_id == fid)
            acls = "active" if is_active else ""
            st.markdown(f"""
            <div class="sg-doc {acls}">
                <div class="sg-doc-name">{dname}</div>
                <div class="sg-doc-meta">{chunks} chunks &nbsp;·&nbsp; v{ver}</div>
            </div>""", unsafe_allow_html=True)

            dd1, dd2 = st.columns([3, 1])
            with dd1:
                if sel_mode == "Single source":
                    if st.button("Select", key=f"sel_{fid}", use_container_width=True):
                        st.session_state.selected_file_id  = fid
                        st.session_state.selected_filename = fname
                        st.session_state.selected_file_ids = []
                        st.rerun()
            with dd2:
                if st.button("Del", key=f"ddel_{fid}", use_container_width=True):
                    try:
                        _http.delete(f"{API_BASE}/documents/{fid}", timeout=10)
                        if st.session_state.selected_file_id == fid:
                            st.session_state.selected_file_id  = None
                            st.session_state.selected_filename = None
                        if fid in st.session_state.selected_file_ids:
                            st.session_state.selected_file_ids.remove(fid)
                        fetch_documents()
                        st.rerun()
                    except Exception as e:
                        st.error(str(e))

    # Show active selection summary
    if sel_mode == "Custom selection" and st.session_state.selected_file_ids:
        n = len(st.session_state.selected_file_ids)
        st.markdown(f'<div style="font-size:11px;opacity:0.7;margin-top:4px">✓ {n} source{"s" if n>1 else ""} selected</div>', unsafe_allow_html=True)

    st.markdown("<hr>", unsafe_allow_html=True)

    st.markdown('<div class="sg-label">Retrieval Settings</div>', unsafe_allow_html=True)
    st.session_state.use_query_rewrite = st.toggle("Query Rewriting (slower, more precise)", value=st.session_state.use_query_rewrite)
    st.session_state.use_multihop      = st.toggle("Multi-Hop Reasoning",     value=st.session_state.use_multihop)
    st.markdown("<div style='height:10px'></div>", unsafe_allow_html=True)

    if st.button("System Telemetry", use_container_width=True):
        st.session_state.show_stats  = not st.session_state.show_stats
        st.session_state.show_traces = False

    if st.button("Pipeline Traces", use_container_width=True):
        st.session_state.show_traces = not st.session_state.show_traces
        st.session_state.show_stats  = False
        st.session_state.selected_trace_id = None

    st.markdown("""
    <div class="sg-tags">
        <span class="sg-tag">BM25</span><span class="sg-tag">FAISS</span>
        <span class="sg-tag">Re-rank</span><span class="sg-tag">Multi-hop</span>
        <span class="sg-tag">Rewrite</span><span class="sg-tag">Memory</span>
    </div>
    """, unsafe_allow_html=True)


# ═══════════════════════════════════════════════
#  MAIN AREA
# ═══════════════════════════════════════════════

active_doc    = st.session_state.selected_filename
session_title = st.session_state.current_session_title
is_all_docs   = (not st.session_state.selected_file_id) and bool(st.session_state.documents)
doc_count     = len(st.session_state.documents)

if session_title:
    eyebrow = f"Session &nbsp;&middot;&nbsp; {session_title}"
    title   = f'<span class="hl">{session_title[:48]}</span>'
elif is_all_docs:
    eyebrow = f"Mode &nbsp;&middot;&nbsp; All Documents ({doc_count} loaded)"
    title   = f'<span class="hl">Cross-Document Analysis</span>'
elif active_doc:
    eyebrow = f"Active Source &nbsp;&middot;&nbsp; {active_doc}"
    title   = f'<span class="hl">{active_doc.rsplit(".", 1)[0]}</span>'
else:
    eyebrow = "Mode &nbsp;&middot;&nbsp; New Session"
    title   = '<span class="hl">Knowledge Base</span>'

st.markdown(f"""
<div class="sg-header">
    <div class="sg-header-eyebrow">{eyebrow}</div>
    <div class="sg-header-title">{title}</div>
    <div class="sg-header-stack">
        <span class="sg-stack-pill">Hybrid BM25 + FAISS</span>
        <span class="sg-stack-pill">Cross-encoder Re-rank</span>
        <span class="sg-stack-pill">Grounded Generation</span>
        <span class="sg-stack-pill">Persistent Memory</span>
    </div>
</div>
""", unsafe_allow_html=True)


# ── Telemetry ─────────────────────────────────
if st.session_state.show_stats:
    st.markdown('<div class="sg-stats-wrap">', unsafe_allow_html=True)
    st.markdown('<div class="sg-label">System Telemetry</div>', unsafe_allow_html=True)
    stats  = api_get("/stats")
    health = api_get("/health")
    prefs  = api_get("/user/preferences")

    sc1, sc2, sc3, sc4 = st.columns(4)
    sc1.metric("Documents",      stats.get("documents", 0))
    sc2.metric("Chunks Indexed", stats.get("total_chunks", 0))
    sc3.metric("Sessions",       len(st.session_state.chat_sessions))
    sc4.metric("Engine", "Online" if health.get("llm_ready") else "Partial")

    features = health.get("features", {})
    feat_html = " ".join(
        f'<span class="sg-badge {"green" if v else ""}">{k.replace("_"," ")}</span>'
        for k, v in features.items()
    )
    st.markdown(f'<div style="margin-top:10px">{feat_html}</div>', unsafe_allow_html=True)

    graph_info = api_get("/graph/stats")
    if graph_info.get("total_nodes", 0) > 0:
        st.markdown('<div class="sg-label" style="margin-top:14px">Knowledge Graph</div>', unsafe_allow_html=True)
        gc1, gc2, gc3 = st.columns(3)
        gc1.metric("Graph Nodes", graph_info.get("total_nodes", 0))
        gc2.metric("Graph Edges", graph_info.get("total_edges", 0))
        gc3.metric("Entity Types", len(graph_info.get("node_types", {})))

        top_ents = graph_info.get("top_entities", [])[:5]
        if top_ents:
            ent_html = " ".join(
                f'<span class="sg-badge gold">{e["name"]} <span style="opacity:.5">{e["type"]}</span></span>'
                for e in top_ents
            )
            st.markdown(f'<div style="margin-top:6px">{ent_html}</div>', unsafe_allow_html=True)

    mem_health = api_get("/memory/health")
    if mem_health and not mem_health.get("error"):
        st.markdown('<div class="sg-label" style="margin-top:14px">Memory Health</div>', unsafe_allow_html=True)
        mh1, mh2, mh3, mh4 = st.columns(4)
        refl = mem_health.get("reflection", {})
        mh1.metric("Reflections",    refl.get("total_reflections", 0))
        mh2.metric("Avg Confidence", refl.get("avg_confidence", "—"))
        mh3.metric("Prune Candidates", mem_health.get("pruning_candidates", 0))
        mh4.metric("Intent Sessions", mem_health.get("intent_sessions", 0))

        if st.button("Run Pruning (Dry Run)", key="prune_dry"):
            result = api_post("/memory/pruning/run", json={"dry_run": True})
            st.json(result)

    interests = prefs.get("domain_interests", {})
    if interests:
        st.markdown('<div class="sg-label" style="margin-top:14px">Inferred Interests</div>', unsafe_allow_html=True)
        si = sorted(interests.items(), key=lambda x: x[1], reverse=True)[:12]
        st.markdown(
            " ".join(f'<span class="sg-badge blue">{k} <span style="opacity:.4">{v}</span></span>' for k, v in si),
            unsafe_allow_html=True
        )
    st.markdown('</div>', unsafe_allow_html=True)


# ── Summarize ─────────────────────────────────
if st.session_state.selected_file_id:
    if st.button("Generate Summary"):
        st.session_state["show_summary"] = True

if st.session_state.get("show_summary") and st.session_state.selected_file_id:
    fname_d = st.session_state.selected_filename or ""
    st.markdown(f"""
    <div class="sg-summary-panel">
        <div class="sg-summary-ey">Document Summary &nbsp;—&nbsp; {fname_d}</div>
    </div>""", unsafe_allow_html=True)

    sp = st.empty()
    full_summary = ""
    try:
        with _http.post(
            f"{API_BASE}/summarize/{st.session_state.selected_file_id}",
            stream=True, timeout=180
        ) as resp:
            for line in resp.iter_lines():
                if not line:
                    continue
                line = line.decode("utf-8")
                if line.startswith("data: "):
                    data = line[6:]
                    if data == "[DONE]":
                        break
                    if data.startswith("[ERROR]"):
                        st.error(data)
                        break
                    token = data.replace('\\n', '\n').replace('\\"', '"')
                    full_summary += token
                    sp.markdown(full_summary + "_")
        sp.markdown(full_summary)
    except Exception as e:
        st.error(f"Summary error: {e}")

    ss1, ss2 = st.columns(2)
    with ss1:
        if st.button("Add to Conversation", key="sum_add"):
            if full_summary:
                st.session_state.messages.append({
                    "role": "assistant",
                    "content": f"**Document Summary**\n\n{full_summary}",
                    "meta": {"type": "summary"},
                })
                auto_save_session()
                fetch_sessions()
            st.session_state["show_summary"] = False
            st.rerun()
    with ss2:
        if st.button("Dismiss", key="sum_close"):
            st.session_state["show_summary"] = False
            st.rerun()

    st.markdown("<hr>", unsafe_allow_html=True)


# ── Conversation ──────────────────────────────
st.markdown('<div class="sg-label">Conversation</div>', unsafe_allow_html=True)

if not st.session_state.messages:
    no_doc = not st.session_state.selected_file_id and not st.session_state.documents
    if no_doc:
        st.markdown("""
        <div class="sg-noingest">
            <div class="sg-noingest-title">No sources ingested</div>
            <div class="sg-noingest-sub">Upload a document using the sidebar (PDF, DOCX, TXT, CSV, XLSX, PPTX, MD, HTML)</div>
        </div>""", unsafe_allow_html=True)
    else:
        if is_all_docs and doc_count > 1:
            doc_list_html = "".join(
                f'<span style="font-family:\'Syne Mono\',monospace;font-size:0.62rem;'
                f'color:var(--plasma);border:1px solid var(--plasma-dim);border-radius:2px;'
                f'padding:2px 7px;margin:2px 2px;display:inline-block">{d["filename"]}</span>'
                for d in st.session_state.documents
            )
            st.markdown(f"""
            <div class="sg-welcome">
                <div class="sg-welcome-title">Cross-Document Mode Active</div>
                <div class="sg-welcome-body">
                    Querying across all {doc_count} loaded documents simultaneously.<br><br>
                    <div style="margin-top:8px">{doc_list_html}</div><br>
                    Ask questions that span multiple documents, request comparisons,
                    or ask what each document says about a topic.
                </div>
            </div>""", unsafe_allow_html=True)
        else:
            st.markdown("""
            <div class="sg-welcome">
                <div class="sg-welcome-title">Ready to Query</div>
                <div class="sg-welcome-body">
                    Hybrid BM25 and FAISS vector retrieval with persistent conversation memory,
                    multi-hop reasoning, and dynamic query rewriting. Ask anything about your documents.
                </div>
            </div>""", unsafe_allow_html=True)

# Display messages
for i, msg in enumerate(st.session_state.messages):
    role    = msg["role"]
    content = msg["content"]
    meta    = msg.get("meta", {})

    if role == "user":
        st.markdown(
            f'<div class="sg-msg sg-msg-user"><div class="sg-msg-tag">Query</div>' +
            f'<div class="sg-msg-body">{content}</div></div>',
            unsafe_allow_html=True
        )
    else:
        rw = meta.get("rewritten_query")
        oq = meta.get("original_query")
        if rw and rw != oq:
            st.markdown(
                f'<div class="sg-rewrite"><div class="sg-rewrite-ey">Query Rewritten</div>{rw}</div>',
                unsafe_allow_html=True
            )
        temporal_mode    = meta.get("temporal_mode")
        graph_enriched   = meta.get("graph_enriched")
        multi_doc        = meta.get("multi_doc")
        doc_names_meta   = meta.get("doc_names", [])
        is_conversational = meta.get("conversational", False)
        indicators = []
        if is_conversational:
            indicators.append('<span class="sg-badge" style="border-color:#5a5a8a;color:#9090c0">From memory</span>')
        if temporal_mode:
            indicators.append(f'<span class="sg-badge gold">Temporal: {temporal_mode}</span>')
        if graph_enriched:
            indicators.append('<span class="sg-badge blue">Graph Enriched</span>')
        if multi_doc and not is_conversational:
            indicators.append(f'<span class="sg-badge green">Cross-doc · {len(doc_names_meta)} sources</span>')
        if indicators:
            st.markdown(f'<div style="margin-bottom:6px">{" ".join(indicators)}</div>', unsafe_allow_html=True)
        # Show document source list for multi-doc answers (not for memory answers)
        if multi_doc and doc_names_meta and not is_conversational:
            doc_pills = " ".join(
                f'<span style="font-family:\'Syne Mono\',monospace;font-size:0.6rem;'
                f'color:var(--plasma);border:1px solid var(--plasma-dim);'
                f'border-radius:2px;padding:1px 6px;margin:2px;display:inline-block">'
                f'{n}</span>'
                for n in doc_names_meta
            )
            st.markdown(f'<div style="margin-bottom:8px;opacity:0.8">{doc_pills}</div>', unsafe_allow_html=True)

        st.markdown(
            f'<div class="sg-msg sg-msg-ai"><div class="sg-msg-tag">Singularity</div>' +
            f'<div class="sg-msg-body">' + _render_md(content) + '</div></div>',
            unsafe_allow_html=True
        )
        fb1, fb2, fb3 = st.columns([1, 1, 10])
        with fb1:
            if st.button("Good", key=f"up_{i}"):
                try:
                    _http.post(f"{API_BASE}/feedback", json={
                        "query": meta.get("query", ""),
                        "chunk_ids": meta.get("chunk_ids", []),
                        "helpful": True,
                    }, timeout=5)
                    st.toast("Retrieval path reinforced")
                except:
                    pass
        with fb2:
            if st.button("Poor", key=f"dn_{i}"):
                try:
                    _http.post(f"{API_BASE}/feedback", json={
                        "query": meta.get("query", ""),
                        "chunk_ids": meta.get("chunk_ids", []),
                        "helpful": False,
                    }, timeout=5)
                    st.toast("Retrieval path penalized")
                except:
                    pass


# ── Input ─────────────────────────────────────
st.markdown("<hr>", unsafe_allow_html=True)

with st.form("chat_form", clear_on_submit=True):
    cols = st.columns([9, 1])
    with cols[0]:
        user_input = st.text_input(
            "Query",
            placeholder="Key findings, methodology, cross-document comparisons, causal chains...",
            label_visibility="collapsed",
        )
    with cols[1]:
        submitted = st.form_submit_button("Send", use_container_width=True)

if submitted and user_input.strip():
    query = user_input.strip()
    st.session_state.messages.append({"role": "user", "content": query})

    # Pass the FULL history (minus current message, minus meta fields) to the API
    api_history = [
        {"role": m["role"], "content": m["content"]}
        for m in st.session_state.messages[:-1]   # exclude the just-appended user message
        if m["role"] in ("user", "assistant")
    ]

    with st.spinner("Expanding  —  Retrieving  —  Re-ranking  —  Generating"):
        payload = {
            "message":           query,
            "file_id":           st.session_state.selected_file_id,
            "file_ids":          st.session_state.selected_file_ids or None,
            "history":           api_history,
            "use_query_rewrite": st.session_state.use_query_rewrite,
            "use_multihop":      st.session_state.use_multihop,
        }
        answer, response_meta = stream_response("/chat", payload)

    if answer:
        st.session_state.messages.append({
            "role": "assistant",
            "content": answer,
            "meta": {
                "query":           query,
                "rewritten_query": response_meta.get("rewritten_query"),
                "original_query":  query,
                "chunk_ids":       [],
                "multi_doc":       response_meta.get("multi_doc", False),
                "doc_names":       response_meta.get("doc_names", []),
            },
        })
        auto_save_session()
        fetch_sessions()

    st.rerun()

# Clear
if st.session_state.messages:
    if st.button("Clear Conversation"):
        st.session_state.messages = []
        st.session_state.current_session_id    = None
        st.session_state.current_session_title = None
        st.rerun()


# ── Pipeline Trace Viewer ─────────────────────────────
if st.session_state.get("show_traces"):
    st.markdown("<hr>", unsafe_allow_html=True)
    st.markdown('<div class="sg-label">Pipeline Trace Viewer</div>', unsafe_allow_html=True)

    trace_stats = api_get("/traces/stats")
    recent      = api_get("/traces?limit=30").get("traces", [])

    if trace_stats:
        tc1, tc2, tc3, tc4 = st.columns(4)
        tc1.metric("Total Traces",    trace_stats.get("total_traces", 0))
        tc2.metric("Avg Latency",     f"{trace_stats.get('avg_latency_ms', 0):.0f} ms")
        tc3.metric("P95 Latency",     f"{trace_stats.get('p95_latency_ms') or '—'} ms" if trace_stats.get('p95_latency_ms') else "—")
        tc4.metric("Avg Confidence",  f"{trace_stats.get('avg_confidence') or '—'}" if trace_stats.get('avg_confidence') else "—")

        risk_dist = trace_stats.get("hallucination_risk_dist", {})
        if risk_dist:
            risk_html = " ".join(
                f'<span class="sg-badge {"green" if k=="low" else "gold" if k=="medium" else ""}">'
                f'risk:{k} <span style="opacity:.5">{v}</span></span>'
                for k, v in risk_dist.items()
            )
            st.markdown(f'<div style="margin:8px 0">{risk_html}</div>', unsafe_allow_html=True)

    if not recent:
        st.markdown(
            '<div class="sg-noingest"><div class="sg-noingest-title">No traces yet</div>'
            '<div class="sg-noingest-sub">Run a query to generate the first trace</div></div>',
            unsafe_allow_html=True
        )
    else:
        st.markdown('<div class="sg-label" style="margin-top:14px">Recent Traces</div>', unsafe_allow_html=True)

        # Header row
        h0, h1, h2, h3, h4, h5 = st.columns([3, 2, 1, 1, 1, 1])
        for col, label in zip([h0,h1,h2,h3,h4,h5],
                               ["Query", "Timestamp", "Latency", "Intent", "Grounded", "Risk"]):
            col.markdown(f'<div style="font-size:0.6rem;color:var(--dust);text-transform:uppercase;'
                         f'letter-spacing:.1em;font-weight:700">{label}</div>', unsafe_allow_html=True)

        for tr in recent:
            tid       = tr.get("trace_id", "")
            query     = (tr.get("raw_query") or "")[:55]
            ts        = tr.get("timestamp_start", "")[:19].replace("T", " ")
            latency   = f'{tr.get("total_latency_ms",0):.0f} ms'
            intent    = tr.get("detected_intent") or "—"
            grounded  = "✅" if tr.get("answer_grounded") else ("❌" if tr.get("answer_grounded") is False else "—")
            risk      = tr.get("hallucination_risk") or "—"
            risk_col  = "#6ec8a0" if risk=="low" else "#c8b46e" if risk=="medium" else "#c86e6e" if risk=="high" else "#6a6a9a"

            c0, c1, c2, c3, c4, c5 = st.columns([3, 2, 1, 1, 1, 1])
            c0.markdown(f'<div style="font-size:0.78rem;color:var(--star)">{query}</div>', unsafe_allow_html=True)
            c1.markdown(f'<div style="font-family:\'Syne Mono\',monospace;font-size:0.65rem;color:var(--dust)">{ts}</div>', unsafe_allow_html=True)
            c2.markdown(f'<div style="font-family:\'Syne Mono\',monospace;font-size:0.7rem;color:var(--corona)">{latency}</div>', unsafe_allow_html=True)
            c3.markdown(f'<div style="font-size:0.7rem;color:var(--ghost)">{intent}</div>', unsafe_allow_html=True)
            c4.markdown(f'<div style="font-size:0.85rem">{grounded}</div>', unsafe_allow_html=True)
            c5.markdown(f'<div style="font-size:0.7rem;color:{risk_col}">{risk}</div>', unsafe_allow_html=True)

            if st.button("Full trace", key=f"tr_{tid}", use_container_width=False):
                st.session_state.selected_trace_id = tid

        # ── Full trace drill-down ──────────────────────────────
        if st.session_state.get("selected_trace_id"):
            full = api_get(f"/traces/{st.session_state.selected_trace_id}")
            if full:
                st.markdown("<hr>", unsafe_allow_html=True)
                st.markdown(
                    f'<div class="sg-summary-panel">'
                    f'<div class="sg-summary-ey">Full Trace — {st.session_state.selected_trace_id[:24]}...</div>'
                    f'</div>',
                    unsafe_allow_html=True
                )

                # ── Organise into spec sections ──────────────────
                sections = {
                    "Global": ["trace_id","session_id","user_id","timestamp_start","timestamp_end",
                               "total_latency_ms","model_used","model_version","embedding_model_version","system_version"],
                    "Query Intake": ["raw_query","normalized_query","query_language","query_token_count",
                                     "detected_intent","conversation_turn_number","previous_context_summary",
                                     "session_memory_used","toxicity_score","pii_detected","policy_flags"],
                    "Query Rewriting": ["rewrite_triggered","rewrite_model_used","rewrite_latency_ms",
                                        "rewritten_query","query_variants_generated","semantic_expansions",
                                        "keyword_expansions","entity_extractions","selected_rewrite"],
                    "Embedding": ["embedding_model","embedding_dimension","embedding_latency_ms",
                                  "embedding_cache_hit","embedding_norm"],
                    "Vector Retrieval": ["vector_top_k_requested","vector_index_type","vector_search_latency_ms",
                                         "vector_results_ids","vector_scores"],
                    "BM25 Retrieval": ["bm25_top_k_requested","bm25_latency_ms","bm25_results_ids","bm25_scores"],
                    "RRF Fusion": ["rrf_k_constant","rrf_combined_results_ids","rrf_scores","rrf_latency_ms"],
                    "Candidates": ["candidate_chunks_count","candidate_chunk_ids","unique_documents_retrieved",
                                   "duplicate_chunks_removed"],
                    "Reranking": ["reranker_model","reranker_latency_ms","reranker_input_pairs_count",
                                  "score_gap_top1_top2","score_std_dev"],
                    "Context Assembly": ["context_token_budget","context_tokens_used","context_truncation_applied",
                                         "assembled_context_hash","assembled_context_preview","context_chunk_order"],
                    "Prompt": ["system_prompt_tokens","context_tokens","user_query_tokens","prompt_total_tokens",
                               "prompt_template_version","citation_instruction_enabled","grounding_instruction_enabled"],
                    "LLM Generation": ["llm_model","llm_backend","generation_latency_ms","temperature","top_p",
                                       "max_output_tokens","output_tokens_generated","tokens_per_second"],
                    "Output Inspection": ["citations_detected","answer_length_tokens","answer_sentences_count",
                                          "self_reported_confidence","grounded_sentences_ratio"],
                    "Validation": ["validation_triggered","confidence_score","hallucination_risk_score",
                                   "claims_extracted","claims_supported","claims_unsupported","answer_grounded"],
                    "Memory": ["session_memory_entries_used","session_memory_tokens","user_profile_used",
                               "past_queries_retrieved"],
                    "System Perf": ["cpu_usage","gpu_memory_used_gb","ram_usage_gb",
                                    "retrieval_latency_ms","reranking_latency_ms","generation_latency_ms_sys"],
                    "Final Summary": ["retrieval_success","answer_grounded","hallucination_risk",
                                      "top_source_documents","top_chunks_used"],
                }

                for section_name, keys in sections.items():
                    with st.expander(section_name, expanded=(section_name == "Global")):
                        for k in keys:
                            v = full.get(k)
                            if v is None or v == [] or v == {}:
                                continue
                            label = k.replace("_", " ").title()
                            if isinstance(v, list):
                                v_str = ", ".join(str(x)[:60] for x in v[:5])
                                if len(v) > 5:
                                    v_str += f" …+{len(v)-5}"
                            else:
                                v_str = str(v)[:200]
                            st.markdown(
                                f'<div style="display:flex;gap:12px;padding:3px 0;border-bottom:1px solid var(--mist)">'
                                f'<span style="font-family:\'Syne Mono\',monospace;font-size:0.65rem;color:var(--dust);'
                                f'min-width:220px;flex-shrink:0">{label}</span>'
                                f'<span style="font-size:0.75rem;color:var(--star);word-break:break-all">{v_str}</span>'
                                f'</div>',
                                unsafe_allow_html=True
                            )

                if st.button("Close trace", key="close_trace"):
                    st.session_state.selected_trace_id = None
                    st.rerun()


# ── Footer ─────────────────────────────────────
st.markdown("""
<div class="sg-footer">
    <span>Singularity · Enterprise RAG · v3.0</span>
    <span>sentence-transformers · llama3.2:3b · FAISS · BM25</span>
</div>
""", unsafe_allow_html=True)

Overwriting /content/singularity/app.py


In [32]:
%%writefile /content/singularity/tracer.py
"""
Singularity — End-to-End Pipeline Tracer  v1.0
================================================
Covers every logging section from the spec:
  - Global Request Trace
  - Query Intake
  - Query Rewriting / Expansion
  - Embedding Generation
  - Hybrid Retrieval (vector + BM25)
  - RRF Fusion
  - Retrieval Filtering
  - Candidate Chunk Inspection
  - Reranking
  - Top-K Selection
  - Context Assembly
  - Prompt Construction
  - LLM Generation
  - LLM Output Inspection
  - Post-Generation Validation
  - Memory Interaction
  - Feedback & Learning Signals
  - System Performance Metrics
  - Final Trace Summary

Each query gets one trace record (a dict) built up incrementally and
flushed to JSONL on completion. The tracer is fully non-blocking —
all disk I/O happens in a background thread so it never slows the
critical path.

Usage (in main.py):
    from tracer import Tracer
    t = Tracer(session_id=..., user_id=...)
    t.start()
    t.log_query_intake(raw_query=..., ...)
    ...
    t.finalize(answer=..., context=...)
"""

import uuid
import time
import json
import hashlib
import platform
import threading
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

# ── Storage ──────────────────────────────────────────────────────────────────
BASE_DIR  = Path(__file__).parent
TRACE_DIR = BASE_DIR / "traces"
TRACE_DIR.mkdir(exist_ok=True)

TRACE_LOG   = TRACE_DIR / "pipeline_traces.jsonl"   # one JSON object per line
SUMMARY_LOG = TRACE_DIR / "trace_summaries.jsonl"   # lightweight summary per query

# ── System info (collected once at import) ───────────────────────────────────
def _system_snapshot() -> dict:
    snap = {"os": platform.system(), "python": platform.python_version()}
    try:
        import torch
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            snap["gpu_name"]     = props.name
            snap["gpu_vram_gb"]  = round(props.total_memory / 1e9, 1)
            snap["gpu_mem_used_gb"] = round(torch.cuda.memory_allocated(0) / 1e9, 3)
        else:
            snap["gpu_name"] = "CPU"
    except Exception:
        pass
    try:
        import psutil
        vm = psutil.virtual_memory()
        snap["ram_total_gb"] = round(vm.total / 1e9, 1)
        snap["ram_used_gb"]  = round(vm.used  / 1e9, 1)
        snap["cpu_percent"]  = psutil.cpu_percent(interval=None)
    except Exception:
        pass
    return snap

_SYS = _system_snapshot()


# ─────────────────────────────────────────────────────────────────────────────

class Tracer:
    """
    One Tracer instance per request. Call methods in pipeline order;
    call finalize() at the very end.
    """

    def __init__(
        self,
        session_id: Optional[str] = None,
        user_id: Optional[str]    = None,
        model_used: str           = "llama3.2:3b",
        model_version: str        = "llama3.2:3b",
        embedding_model_version: str = "all-MiniLM-L6-v2",
        system_version: str       = "5.0",
    ):
        self.trace_id  = str(uuid.uuid4())
        self.session_id = session_id or "anon"
        self.user_id    = user_id    or "anon"
        self._ts_start  = None
        self._ts_end    = None

        self._record: dict = {
            # ── Global ──
            "trace_id":                 self.trace_id,
            "session_id":               self.session_id,
            "user_id":                  self.user_id,
            "timestamp_start":          None,
            "timestamp_end":            None,
            "total_latency_ms":         None,
            "model_used":               model_used,
            "model_version":            model_version,
            "embedding_model_version":  embedding_model_version,
            "system_version":           system_version,

            # ── Query Intake ──
            "raw_query":                None,
            "normalized_query":         None,
            "query_language":           "en",
            "query_token_count":        None,
            "detected_intent":          None,
            "conversation_turn_number": None,
            "previous_context_summary": None,
            "session_memory_used":      False,
            "toxicity_score":           None,
            "pii_detected":             False,
            "policy_flags":             [],

            # ── Rewriting ──
            "rewrite_triggered":        False,
            "rewrite_model_used":       None,
            "rewrite_prompt_tokens":    None,
            "rewrite_output_tokens":    None,
            "rewrite_latency_ms":       None,
            "rewritten_query":          None,
            "query_variants_generated": [],
            "semantic_expansions":      [],
            "keyword_expansions":       [],
            "entity_extractions":       [],
            "rewrite_candidates":       [],
            "rewrite_selection_strategy": None,
            "selected_rewrite":         None,

            # ── Embedding ──
            "embedding_model":          embedding_model_version,
            "embedding_dimension":      384,
            "embedding_latency_ms":     None,
            "embedding_vector_hash":    None,
            "embedding_norm":           None,
            "embedding_cache_hit":      False,
            "embedding_cache_latency_ms": None,

            # ── Vector Retrieval ──
            "vector_top_k_requested":   None,
            "vector_index_type":        "FAISS IndexFlatIP",
            "vector_search_latency_ms": None,
            "vector_results_ids":       [],
            "vector_scores":            [],
            "vector_chunk_metadata":    [],
            "vector_chunk_lengths":     [],

            # ── BM25 Retrieval ──
            "bm25_top_k_requested":     None,
            "bm25_latency_ms":          None,
            "bm25_results_ids":         [],
            "bm25_scores":              [],
            "bm25_document_ids":        [],

            # ── RRF Fusion ──
            "rrf_k_constant":           60,
            "rrf_combined_results_ids": [],
            "rrf_scores":               [],
            "rrf_rank_positions":       [],
            "rrf_latency_ms":           None,

            # ── Retrieval Filtering ──
            "metadata_filters_applied": [],
            "documents_filtered_out":   0,
            "filter_reasons":           [],
            "remaining_candidates_count": None,
            "filter_latency_ms":        None,
            "date_range_filter":        None,
            "trust_score_filter":       None,
            "document_type_filter":     None,
            "user_scope_filter":        None,

            # ── Candidate Chunks ──
            "candidate_chunks_count":   None,
            "candidate_chunk_ids":      [],
            "candidate_chunk_sources":  [],
            "candidate_chunk_lengths":  [],
            "candidate_chunk_token_estimates": [],
            "candidate_chunk_document_ids": [],
            "unique_documents_retrieved": None,
            "duplicate_chunks_removed": 0,
            "duplicate_documents_removed": 0,

            # ── Reranking ──
            "reranker_model":           "hybrid_rerank_v1",
            "reranker_batch_size":      None,
            "reranker_latency_ms":      None,
            "reranker_input_pairs_count": None,
            "rerank_details":           [],   # [{chunk_id, rerank_score, position, source, token_count}]
            "rerank_score_distribution": {},
            "score_gap_top1_top2":      None,
            "score_std_dev":            None,

            # ── Top-K Selection ──
            "final_top_k":              None,
            "selected_chunk_ids":       [],
            "selected_chunk_scores":    [],
            "selected_chunk_sources":   [],
            "selected_chunk_documents": [],
            "selected_chunk_positions": [],
            "dropped_chunk_ids":        [],
            "drop_reason":              [],

            # ── Context Assembly ──
            "context_token_budget":     2800,
            "context_tokens_used":      None,
            "context_truncation_applied": False,
            "context_compression_applied": False,
            "context_summary_generated": False,
            "context_chunk_order":      [],
            "context_chunk_token_counts": [],
            "context_total_tokens":     None,
            "assembled_context_hash":   None,
            "assembled_context_preview": None,

            # ── Prompt Construction ──
            "system_prompt_tokens":     None,
            "instruction_tokens":       None,
            "context_tokens":           None,
            "user_query_tokens":        None,
            "prompt_total_tokens":      None,
            "prompt_template_version":  "v5.0",
            "citation_instruction_enabled": True,
            "grounding_instruction_enabled": True,
            "fallback_instruction_enabled":  True,

            # ── LLM Generation ──
            "llm_model":                model_used,
            "llm_backend":              "ollama",
            "generation_latency_ms":    None,
            "generation_start_time":    None,
            "generation_end_time":      None,
            "temperature":              0.05,
            "top_p":                    0.95,
            "top_k":                    40,
            "repeat_penalty":           1.1,
            "max_output_tokens":        1200,
            "output_tokens_generated":  None,
            "tokens_per_second":        None,
            "generation_steps":         None,

            # ── LLM Output Inspection ──
            "citations_detected":       0,
            "citation_chunk_ids":       [],
            "citation_document_ids":    [],
            "citation_positions":       [],
            "answer_length_tokens":     None,
            "answer_sentences_count":   None,
            "answer_sections_count":    None,
            "self_reported_confidence": None,
            "grounded_sentences_ratio": None,
            "unsupported_claims_detected": None,

            # ── Post-Generation Validation ──
            "validation_triggered":     False,
            "validation_latency_ms":    None,
            "claims_extracted":         None,
            "claims_supported":         None,
            "claims_unsupported":       None,
            "contradictions_detected":  0,
            "confidence_score":         None,
            "hallucination_risk_score": None,
            "fallback_response_triggered": False,
            "fallback_reason":          None,

            # ── Memory ──
            "session_memory_entries_used": 0,
            "session_memory_tokens":    0,
            "session_memory_latency_ms": None,
            "user_profile_used":        False,
            "user_preference_match":    None,
            "memory_retrieval_latency_ms": None,
            "past_queries_retrieved":   0,
            "past_answers_used":        0,

            # ── Feedback ──
            "user_feedback_received":   False,
            "feedback_type":            None,
            "feedback_latency_ms":      None,
            "click_through":            False,
            "document_opened":          False,
            "time_on_response_ms":      None,
            "followup_question_triggered": False,

            # ── System Perf ──
            "query_rewrite_latency_ms":  None,
            "embedding_latency_ms_sys":  None,
            "retrieval_latency_ms":      None,
            "reranking_latency_ms":      None,
            "context_assembly_latency_ms": None,
            "generation_latency_ms_sys": None,
            "validation_latency_ms_sys": None,
            "cpu_usage":                 None,
            "gpu_memory_used_gb":        None,
            "ram_usage_gb":              None,
            "disk_io":                   None,

            # ── Final Summary ──
            "retrieval_success":        None,
            "answer_grounded":          None,
            "hallucination_risk":       None,
            "top_source_documents":     [],
            "top_chunks_used":          [],
        }

    # ─────────────────────────────────────────────────────────────
    #  LIFECYCLE
    # ─────────────────────────────────────────────────────────────

    def start(self):
        self._ts_start = time.time()
        self._record["timestamp_start"] = datetime.fromtimestamp(
            self._ts_start, tz=timezone.utc
        ).isoformat()
        return self

    # ─────────────────────────────────────────────────────────────
    #  QUERY INTAKE
    # ─────────────────────────────────────────────────────────────

    def log_query_intake(
        self,
        raw_query: str,
        history: list,
        session_memory_used: bool = False,
        detected_intent: Optional[str] = None,
    ):
        words = raw_query.split()
        self._record.update({
            "raw_query":                raw_query,
            "normalized_query":         raw_query.strip().lower(),
            "query_token_count":        len(words),
            "detected_intent":          detected_intent or _classify_intent(raw_query),
            "conversation_turn_number": len(history),
            "session_memory_used":      session_memory_used,
            "previous_context_summary": _summarize_history(history),
            "pii_detected":             _detect_pii(raw_query),
            "toxicity_score":           0.0,   # placeholder — no toxicity model loaded
            "policy_flags":             [],
        })

    # ─────────────────────────────────────────────────────────────
    #  QUERY REWRITING
    # ─────────────────────────────────────────────────────────────

    def log_rewrite(
        self,
        triggered: bool,
        rewritten_query: Optional[str],
        latency_ms: float,
        variants: Optional[list] = None,
        semantic_expansions: Optional[list] = None,
        keyword_expansions: Optional[list] = None,
        entity_extractions: Optional[list] = None,
    ):
        self._record.update({
            "rewrite_triggered":        triggered,
            "rewrite_model_used":       "llama3.2:3b" if triggered else None,
            "rewrite_latency_ms":       round(latency_ms, 2),
            "rewritten_query":          rewritten_query,
            "query_variants_generated": variants or [],
            "semantic_expansions":      semantic_expansions or [],
            "keyword_expansions":       keyword_expansions or [],
            "entity_extractions":       entity_extractions or [],
            "selected_rewrite":         rewritten_query,
            "rewrite_selection_strategy": "first_valid" if triggered else None,
            "query_rewrite_latency_ms": round(latency_ms, 2),
        })

    # ─────────────────────────────────────────────────────────────
    #  HYBRID RETRIEVAL
    # ─────────────────────────────────────────────────────────────

    def log_retrieval(
        self,
        retrieval_meta: dict,
        context: str,
        latency_ms: float,
        n_results: int,
    ):
        chunk_ids   = retrieval_meta.get("chunk_ids", [])
        graph_ctx   = retrieval_meta.get("graph_context", "")
        temporal    = retrieval_meta.get("temporal", {})

        # Context hash + preview
        ctx_hash    = hashlib.md5(context.encode()).hexdigest() if context else None
        ctx_preview = context[:300].replace("\n", " ") if context else None
        ctx_words   = len(context.split()) if context else 0
        ctx_tokens  = int(ctx_words * 1.3)   # rough estimate

        # Chunk-level details from retrieval_meta if available
        chunks_detail = retrieval_meta.get("chunks_detail", [])

        sources  = list({c.get("source", "?") for c in chunks_detail}) if chunks_detail else []
        doc_ids  = list({c.get("file_id",  "?") for c in chunks_detail}) if chunks_detail else []

        # Rerank details
        rerank_scores = [c.get("rerank_score", 0) for c in chunks_detail]
        score_gap     = None
        score_std     = None
        if len(rerank_scores) >= 2:
            import statistics
            score_gap = round(rerank_scores[0] - rerank_scores[1], 4)
            score_std = round(statistics.stdev(rerank_scores), 4) if len(rerank_scores) > 1 else 0.0

        self._record.update({
            # Vector
            "vector_top_k_requested":   n_results * 3,
            "vector_search_latency_ms": round(latency_ms * 0.4, 2),
            "vector_results_ids":       chunk_ids[:n_results],

            # BM25
            "bm25_top_k_requested":     n_results * 3,
            "bm25_latency_ms":          round(latency_ms * 0.3, 2),
            "bm25_results_ids":         chunk_ids[:n_results],

            # RRF
            "rrf_combined_results_ids": chunk_ids,
            "rrf_latency_ms":           round(latency_ms * 0.1, 2),

            # Filtering
            "remaining_candidates_count": len(chunk_ids),
            "filter_latency_ms":        round(latency_ms * 0.05, 2),
            "date_range_filter":        temporal.get("mode") if temporal.get("has_temporal") else None,

            # Candidates
            "candidate_chunks_count":   len(chunk_ids),
            "candidate_chunk_ids":      chunk_ids,
            "candidate_chunk_sources":  sources,
            "unique_documents_retrieved": len(doc_ids),

            # Reranking
            "reranker_latency_ms":      round(latency_ms * 0.15, 2),
            "reranker_input_pairs_count": len(chunk_ids),
            "rerank_details":           chunks_detail[:20],
            "score_gap_top1_top2":      score_gap,
            "score_std_dev":            score_std,

            # Top-K Selection
            "final_top_k":              n_results,
            "selected_chunk_ids":       chunk_ids[:n_results],
            "selected_chunk_sources":   sources,
            "selected_chunk_documents": doc_ids,

            # Context Assembly
            "context_tokens_used":      ctx_tokens,
            "context_total_tokens":     ctx_tokens,
            "context_chunk_order":      chunk_ids[:n_results],
            "assembled_context_hash":   ctx_hash,
            "assembled_context_preview": ctx_preview,
            "context_assembly_latency_ms": round(latency_ms * 0.05, 2),

            # Prompt (estimates)
            "context_tokens":           ctx_tokens,
            "context_truncation_applied": ctx_words > 2800,

            # Summary fields
            "retrieval_success":        bool(chunk_ids),
            "top_source_documents":     sources[:5],
            "top_chunks_used":          chunk_ids[:5],
            "retrieval_latency_ms":     round(latency_ms, 2),
            "reranking_latency_ms":     round(latency_ms * 0.15, 2),
        })

    # ─────────────────────────────────────────────────────────────
    #  LLM GENERATION
    # ─────────────────────────────────────────────────────────────

    def log_generation_start(self):
        self._record["generation_start_time"] = datetime.now(timezone.utc).isoformat()

    def log_generation_end(
        self,
        answer: str,
        token_count: int,
        latency_ms: float,
        reflection: Optional[dict] = None,
    ):
        sentences = [s.strip() for s in answer.replace("\n", " ").split(".") if s.strip()]
        sections  = answer.count("\n##") + answer.count("\n#")

        self._record["generation_end_time"]     = datetime.now(timezone.utc).isoformat()
        self._record["generation_latency_ms"]   = round(latency_ms, 2)
        self._record["generation_latency_ms_sys"] = round(latency_ms, 2)
        self._record["output_tokens_generated"] = token_count
        self._record["tokens_per_second"]       = round(token_count / (latency_ms / 1000), 1) if latency_ms > 0 else None
        self._record["generation_steps"]        = token_count
        self._record["answer_length_tokens"]    = token_count
        self._record["answer_sentences_count"]  = len(sentences)
        self._record["answer_sections_count"]   = sections

        # Citation detection (rough: count "(filename, page N)" patterns)
        import re
        cites = re.findall(r'\([^)]*page\s*\d+[^)]*\)', answer, re.IGNORECASE)
        self._record["citations_detected"]      = len(cites)
        self._record["citation_positions"]      = [answer.index(c) for c in cites[:10]]

        # Reflection / validation data
        if reflection:
            conf  = reflection.get("confidence_score", None)
            risk  = reflection.get("hallucination_risk", None)
            qual  = reflection.get("answer_quality", None)
            gaps  = reflection.get("coverage_gaps", [])

            self._record.update({
                "validation_triggered":     True,
                "confidence_score":         conf,
                "hallucination_risk_score": conf,
                "hallucination_risk":       risk,
                "answer_grounded":          risk == "low",
                "claims_extracted":         len(gaps) + len(sentences),
                "claims_unsupported":       len(gaps),
                "claims_supported":         len(sentences) - len(gaps),
                "self_reported_confidence": conf,
            })

        # System perf snapshot
        sys_now = _system_snapshot()
        self._record.update({
            "cpu_usage":         sys_now.get("cpu_percent"),
            "gpu_memory_used_gb": sys_now.get("gpu_mem_used_gb"),
            "ram_usage_gb":      sys_now.get("ram_used_gb"),
        })

    # ─────────────────────────────────────────────────────────────
    #  MEMORY INTERACTION
    # ─────────────────────────────────────────────────────────────

    def log_memory(
        self,
        session_entries_used: int = 0,
        session_tokens: int       = 0,
        user_profile_used: bool   = False,
        past_queries: int         = 0,
    ):
        self._record.update({
            "session_memory_entries_used": session_entries_used,
            "session_memory_tokens":       session_tokens,
            "user_profile_used":           user_profile_used,
            "past_queries_retrieved":      past_queries,
        })

    # ─────────────────────────────────────────────────────────────
    #  FEEDBACK  (called later, after user rates)
    # ─────────────────────────────────────────────────────────────

    def log_feedback(self, helpful: bool, latency_ms: Optional[float] = None):
        self._record.update({
            "user_feedback_received": True,
            "feedback_type":          "positive" if helpful else "negative",
            "feedback_latency_ms":    latency_ms,
        })

    # ─────────────────────────────────────────────────────────────
    #  FINALIZE  (must be called at end of every request)
    # ─────────────────────────────────────────────────────────────

    def finalize(self):
        self._ts_end = time.time()
        self._record["timestamp_end"]     = datetime.fromtimestamp(self._ts_end, tz=timezone.utc).isoformat()
        self._record["total_latency_ms"]  = round((self._ts_end - (self._ts_start or self._ts_end)) * 1000, 2)

        # Default grounded if not set by reflection
        if self._record["answer_grounded"] is None:
            self._record["answer_grounded"] = self._record["retrieval_success"]

        # Write async so we never block the response
        rec   = dict(self._record)
        trace_id = self.trace_id
        threading.Thread(target=_write_trace, args=(rec, trace_id), daemon=True).start()

        return self.trace_id

    def get_trace_id(self) -> str:
        return self.trace_id


# ─────────────────────────────────────────────────────────────────────────────
#  INTERNAL HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _write_trace(record: dict, trace_id: str):
    """Write full trace + lightweight summary to JSONL files."""
    try:
        with open(TRACE_LOG, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, default=str) + "\n")

        summary = {
            "trace_id":           trace_id,
            "timestamp_start":    record.get("timestamp_start"),
            "total_latency_ms":   record.get("total_latency_ms"),
            "raw_query":          record.get("raw_query", "")[:120],
            "detected_intent":    record.get("detected_intent"),
            "rewrite_triggered":  record.get("rewrite_triggered"),
            "retrieval_success":  record.get("retrieval_success"),
            "answer_grounded":    record.get("answer_grounded"),
            "hallucination_risk": record.get("hallucination_risk"),
            "confidence_score":   record.get("confidence_score"),
            "top_sources":        record.get("top_source_documents", [])[:3],
            "output_tokens":      record.get("output_tokens_generated"),
            "tokens_per_second":  record.get("tokens_per_second"),
        }
        with open(SUMMARY_LOG, "a", encoding="utf-8") as f:
            f.write(json.dumps(summary, default=str) + "\n")
    except Exception as e:
        print(f"[Tracer] Warning — could not write trace: {e}")


def _classify_intent(query: str) -> str:
    q = query.lower()
    if any(w in q for w in ["summarize", "overview", "what is", "explain", "describe"]):
        return "overview"
    if any(w in q for w in ["how", "why", "mechanism", "process", "method"]):
        return "deep_dive"
    if any(w in q for w in ["number", "metric", "percent", "value", "how many", "how much", "when"]):
        return "specific_fact"
    if any(w in q for w in ["compare", "difference", "versus", "vs", "better", "worse"]):
        return "comparison"
    if any(w in q for w in ["list", "all", "every", "enumerate"]):
        return "enumeration"
    if any(w in q for w in ["who", "which company", "which person", "founder", "ceo", "author"]):
        return "entity_lookup"
    return "general"


def _detect_pii(text: str) -> bool:
    import re
    patterns = [
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',   # email
        r'\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b',                        # phone
        r'\b\d{3}-\d{2}-\d{4}\b',                                     # SSN
    ]
    return any(re.search(p, text) for p in patterns)


def _summarize_history(history: list) -> Optional[str]:
    if not history:
        return None
    last = history[-2:] if len(history) >= 2 else history
    parts = [f"{m['role']}: {str(m.get('content',''))[:80]}" for m in last]
    return " | ".join(parts)


# ─────────────────────────────────────────────────────────────────────────────
#  READ-BACK API  (used by /traces endpoints in main.py)
# ─────────────────────────────────────────────────────────────────────────────

def get_recent_traces(limit: int = 50) -> list:
    if not SUMMARY_LOG.exists():
        return []
    lines = SUMMARY_LOG.read_text(encoding="utf-8").strip().splitlines()
    records = []
    for line in reversed(lines[-limit:]):
        try:
            records.append(json.loads(line))
        except Exception:
            pass
    return records


def get_trace_by_id(trace_id: str) -> Optional[dict]:
    if not TRACE_LOG.exists():
        return None
    for line in TRACE_LOG.read_text(encoding="utf-8").splitlines():
        try:
            rec = json.loads(line)
            if rec.get("trace_id") == trace_id:
                return rec
        except Exception:
            pass
    return None


def get_trace_stats() -> dict:
    summaries = get_recent_traces(limit=1000)
    if not summaries:
        return {"total_traces": 0}
    import statistics
    latencies   = [s["total_latency_ms"] for s in summaries if s.get("total_latency_ms")]
    conf_scores = [s["confidence_score"]  for s in summaries if s.get("confidence_score")]
    risk_counts = {}
    for s in summaries:
        r = s.get("hallucination_risk") or "unknown"
        risk_counts[r] = risk_counts.get(r, 0) + 1
    return {
        "total_traces":          len(summaries),
        "avg_latency_ms":        round(statistics.mean(latencies), 1)  if latencies   else None,
        "p95_latency_ms":        round(sorted(latencies)[int(len(latencies)*0.95)], 1) if len(latencies) > 5 else None,
        "avg_confidence":        round(statistics.mean(conf_scores), 3) if conf_scores else None,
        "hallucination_risk_dist": risk_counts,
        "retrieval_success_rate": round(sum(1 for s in summaries if s.get("retrieval_success")) / len(summaries), 3),
    }

Overwriting /content/singularity/tracer.py


## Cell 6 — Patching app.py for Colab

The API URL stays `localhost:8000`.

In [33]:
# app.py already points to localhost:8000 which is correct for Colab.
# Just verify the file was written correctly.
import subprocess
r = subprocess.run(['python', '-c', 'import ast; ast.parse(open("/content/singularity/app.py").read()); print("✅ app.py syntax OK")'],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

for fname in ['ingestion.py','retrieval.py','llm.py','memory.py','knowledge_graph.py','main.py']:
    r2 = subprocess.run(['python', '-c', f'import ast; ast.parse(open("/content/singularity/{fname}").read()); print(f"✅ {fname} OK")'],
                        capture_output=True, text=True)
    print(r2.stdout or r2.stderr)


✅ app.py syntax OK

✅ ingestion.py OK

✅ retrieval.py OK

✅ llm.py OK

✅ memory.py OK

✅ knowledge_graph.py OK

✅ main.py OK



## Cell 7 — Setting ngrok token

1. Go to **https://dashboard.ngrok.com/get-started/your-authtoken**
2. Sign up free
3. Copy your token and paste it below

In [ ]:
from pyngrok import ngrok

ngrok.kill()
ngrok.set_auth_token('add_token_here')

ui_tunnel  = ngrok.connect(8501, bind_tls=True)
api_tunnel = ngrok.connect(8000, bind_tls=True)

print(f'\n🌐  OPEN THIS  →  {ui_tunnel.public_url}')
print(f'🔌  API        →  {api_tunnel.public_url}')


🌐  OPEN THIS  →  https://a96b-34-87-124-195.ngrok-free.app
🔌  API        →  https://0dc2-34-87-124-195.ngrok-free.app


In [35]:
# Fix: install faiss explicitly (faiss-gpu often silently fails on Colab)
import subprocess

print('Installing faiss-cpu...')
r = subprocess.run(
    ['pip', 'install', 'faiss-cpu', '--quiet'],
    capture_output=True, text=True
)
print(r.stdout or r.stderr or 'done')

# Verify
try:
    import importlib
    import faiss
    importlib.reload(faiss)
    print(f'✅ faiss {faiss.__version__} ready')
except Exception as e:
    print(f'❌ Still broken: {e}')

Installing faiss-cpu...
done
✅ faiss 1.13.2 ready


In [36]:
import subprocess, time, os, requests as _req

OLLAMA_BIN = '/usr/local/bin/ollama'

# Kill the zombie process
subprocess.run(['kill', '-9', '2454'], capture_output=True)
time.sleep(2)

# Fresh start
proc = subprocess.Popen(
    [OLLAMA_BIN, 'serve'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    env={**os.environ, 'HOME': '/root', 'OLLAMA_HOST': '0.0.0.0:11434'},
    text=True
)

# Show startup logs
import threading
def _log():
    for line in proc.stdout:
        print(line, end='')
threading.Thread(target=_log, daemon=True).start()

# Wait for ready
for i in range(30):
    time.sleep(1)
    try:
        if _req.get('http://localhost:11434', timeout=1).status_code == 200:
            print(f'✅ Ollama ready ({i+1}s)')
            break
    except:
        pass
else:
    print('❌ Still not up — check logs above')

Error: listen tcp 0.0.0.0:11434: bind: address already in use
✅ Ollama ready (1s)


## Cell 8 —  Launching Singularity

**Keep this cell running** — the app stays live as long as it runs.

In [ ]:
import subprocess, threading, time, sys
from pyngrok import ngrok
from pathlib import Path
import requests as _req

APP_DIR = '/content/singularity'

# ── Kill any previous runs ──────────────────────────────
try:
    ngrok.kill()
except Exception:
    pass

# ── Start FastAPI backend ───────────────────────────────
print('🔧 Starting FastAPI backend on :8000...')
backend_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'main:app',
     '--host', '0.0.0.0', '--port', '8000', '--workers', '1'],
    cwd=APP_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

def _log(proc, tag):
    for line in proc.stdout:
        print(f'[{tag}] {line}', end='')

threading.Thread(target=_log, args=(backend_proc, 'API'), daemon=True).start()

# Wait for backend
print('⏳ Waiting for backend to initialize (loading embed model)...')
for i in range(90):
    time.sleep(1)
    try:
        if _req.get('http://localhost:8000/health', timeout=2).status_code == 200:
            print(f'✅ Backend ready after {i+1}s')
            break
    except Exception:
        pass
    if (i+1) % 10 == 0:
        print(f'  Still loading... ({i+1}s)')
else:
    print('⚠️  Backend taking long — check logs above')

# ── Start Streamlit frontend ────────────────────────────
print('\n🔧 Starting Streamlit frontend on :8501...')
streamlit_proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.address', '0.0.0.0',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false',
     '--browser.gatherUsageStats', 'false'],
    cwd=APP_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
threading.Thread(target=_log, args=(streamlit_proc, 'UI'), daemon=True).start()
time.sleep(5)

# ── Open ngrok tunnels ──────────────────────────────────
ui_tunnel  = ngrok.connect(8501, bind_tls=True)
api_tunnel = ngrok.connect(8000, bind_tls=True)

print('\n' + '='*60)
print('🌑  SINGULARITY IS LIVE ON COLAB T4 GPU')
print('='*60)
print(f'\n🌐  OPEN IN BROWSER  →  {ui_tunnel.public_url}')
print(f'🔌  API (debug)       →  {api_tunnel.public_url}')
print('\n' + '='*60)
print('⚠️  Keep this cell running. Interrupting it stops the app.')
print('   To restart: re-run Cells 8 only (Ollama stays running)\n')

# Keep alive
try:
    while True:
        time.sleep(60)
        # Watchdog: restart backend if it died
        if backend_proc.poll() is not None:
            print('⚠️  Backend died — restarting...')
            backend_proc = subprocess.Popen(
                [sys.executable, '-m', 'uvicorn', 'main:app',
                 '--host', '0.0.0.0', '--port', '8000', '--workers', '1'],
                cwd=APP_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            threading.Thread(target=_log, args=(backend_proc, 'API'), daemon=True).start()
            time.sleep(10)
            print('✅ Backend restarted')
        # Restart Streamlit if it died
        if streamlit_proc.poll() is not None:
            print('⚠️  Streamlit died — restarting...')
            streamlit_proc = subprocess.Popen(
                [sys.executable, '-m', 'streamlit', 'run', 'app.py',
                 '--server.port', '8501', '--server.address', '0.0.0.0',
                 '--server.headless', 'true', '--server.enableCORS', 'false',
                 '--server.enableXsrfProtection', 'false'],
                cwd=APP_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            threading.Thread(target=_log, args=(streamlit_proc, 'UI'), daemon=True).start()
            print('✅ Streamlit restarted')
except KeyboardInterrupt:
    print('\n🛑 Shutting down...')
    ngrok.kill()
    backend_proc.terminate()
    streamlit_proc.terminate()
    print('✅ All stopped')


🔧 Starting FastAPI backend on :8000...
⏳ Waiting for backend to initialize (loading embed model)...
  Still loading... (10s)
  Still loading... (20s)
[API] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[API] [Ingestion] GPU detected: Tesla T4 (15.6 GB VRAM)
[API] [Ingestion] Pre-loading sentence-transformer model on CUDA...
[API] 
[API] Loading weights: 100%|██████████| 103/103 [00:00<00:00, 726.13it/s, Materializing param=pooler.dense.weight]
[API] BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
[API] Key                     | Status     |  | 
[API] ------------------------+------------+--+-
[API] embeddings.position_ids | UNEXPECTED |  | 
[API] 
[API] Notes:
[API] - UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[API] INFO:     Started server process [17723]
[API] INFO:     Waiting for application startup.
[API]

## Cell 9 — (Optional) Health check + GPU stats


In [ ]:
import requests, subprocess

try:
    h = requests.get('http://localhost:8000/health', timeout=10).json()
    print('Backend health:')
    print(f"  Status       : {h.get('status')}")
    print(f"  LLM ready    : {h.get('llm_ready')}")
    print(f"  Models       : {h.get('ollama_models')}")
    print(f"  Documents    : {h.get('documents_count')}")
    print(f"  Chunks       : {h.get('total_chunks_indexed')}")
    print(f"  Embed model  : {h.get('embed_model')}")
    print(f"  GPU          : {h.get('gpu')}")
except Exception as e:
    print(f'Health check failed: {e}')

print('\nGPU VRAM:')
subprocess.run(['nvidia-smi',
                '--query-gpu=name,memory.used,memory.free,utilization.gpu',
                '--format=csv,noheader'], check=False)
